# GymAI drawings

Draws the exercises that have no picture yet, in the same black-ink style as the library (Everkinetic,
CC BY-SA 4.0). The pose of every exercise is already written in `exercise-poses.json` below, so no videos are
needed. (Optional: add clips as an input dataset and those exercises use the clip's pose instead.)

**Before running:** Settings → Accelerator → **GPU T4 x2** (not P100: current PyTorch no longer supports it),
and Settings → **Internet on**. Add the `everkinetic-style` dataset with **+ Add Input**. Then **Run All**.

The first run learns the style (~1 hour) and draws the first `LIMIT` exercises so you can check the quality.
Set `LIMIT = 0` to draw all of them (~3-4 hours). Download **gymai-drawings.zip** and **gymai-style-lora.zip**
from the Output panel; adding the LoRA zip as an input next time skips the training.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q diffusers==0.42.1 transformers==5.22.1 peft==0.21.2 accelerate controlnet_aux==0.0.10 opencv-python-headless

In [ ]:
%%writefile pipeline.py
"""GymAI drawing pipeline: learn the Everkinetic line-art style, then draw new exercises in it.

Steps (each is a sub-command):
  train     Fine-tune a small LoRA on the Everkinetic drawings (everkinetic-style.zip / metadata.jsonl).
  poses     Read exercise clips or photos and save a start and an end pose skeleton for each.
  specposes Make the start/end skeletons from written pose descriptions (exercise-poses.json), no video.
  generate  Draw each exercise's start and end frame from its poses, in the learned style.

Clip naming (in the clips folder): "<Exercise Name>.mp4" (any common video type), or two photos
"<Exercise Name>__start.jpg" and "<Exercise Name>__end.jpg". Use the exercise name exactly as the app shows it
in English, so the importer can match it.

Runs on a GPU (Kaggle T4). --tiny swaps in very small test models so the code can be checked on a CPU.
"""
import argparse
import json
import math
import random
import re
from pathlib import Path

import numpy as np
import torch
from PIL import Image, ImageOps

BASE = "stable-diffusion-v1-5/stable-diffusion-v1-5"
CONTROLNET = "lllyasviel/control_v11p_sd15_openpose"
TINY_BASE = "hf-internal-testing/tiny-stable-diffusion-pipe"
TRIGGER = "evkdraw"
# Kept under CLIP's 77-token limit so nothing is cut off. The first run drew walls, floors and stray weights,
# so those lead the list.
NEGATIVE = ("background, wall, bricks, floor, tiles, room, gym, frame, border, extra barbell, extra dumbbell, "
            "color, shading, gray, photo, text, blurry, extra limbs, deformed")
VIDEO_TYPES = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm"}
PHOTO_TYPES = {".jpg", ".jpeg", ".png", ".webp"}


def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "-", name.lower()).strip("-")


def square(image: Image.Image, size: int, fill) -> Image.Image:
    """Pad to a square without distorting, then resize."""
    image = ImageOps.contain(image, (size, size), Image.LANCZOS)
    canvas = Image.new(image.mode, (size, size), fill)
    canvas.paste(image, ((size - image.width) // 2, (size - image.height) // 2))
    return canvas


def device_and_dtype(tiny: bool):
    if torch.cuda.is_available() and not tiny:
        return "cuda", torch.float16
    return "cpu", torch.float32


# --------------------------------------------------------------------------- train

def load_captions(data: Path) -> list[tuple[Path, str]]:
    meta = next(data.rglob("metadata.jsonl"))
    rows = [json.loads(line) for line in meta.read_text(encoding="utf-8").splitlines() if line.strip()]
    return [(meta.parent / row["file_name"], row["text"]) for row in rows]


def train(args) -> None:
    from diffusers import AutoencoderKL, DDPMScheduler, StableDiffusionPipeline, UNet2DConditionModel
    from diffusers.utils import convert_state_dict_to_diffusers
    from peft import LoraConfig
    from peft.utils import get_peft_model_state_dict
    from transformers import CLIPTextModel, CLIPTokenizer

    device, dtype = device_and_dtype(args.tiny)
    base = TINY_BASE if args.tiny else args.base
    size = 64 if args.tiny else args.resolution
    torch.manual_seed(args.seed)
    random.seed(args.seed)

    tokenizer = CLIPTokenizer.from_pretrained(base, subfolder="tokenizer")
    text_encoder = CLIPTextModel.from_pretrained(base, subfolder="text_encoder").to(device, dtype)
    vae = AutoencoderKL.from_pretrained(base, subfolder="vae").to(device, dtype)
    unet = UNet2DConditionModel.from_pretrained(base, subfolder="unet").to(device, dtype)
    scheduler = DDPMScheduler.from_pretrained(base, subfolder="scheduler")
    for model in (text_encoder, vae, unet):
        model.requires_grad_(False)
    unet.add_adapter(LoraConfig(r=args.rank, lora_alpha=args.rank, init_lora_weights="gaussian",
                                target_modules=["to_k", "to_q", "to_v", "to_out.0"]))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:  # trainable LoRA weights stay in fp32 even when the base model is fp16
        p.data = p.data.float()
    if device == "cuda":
        unet.enable_gradient_checkpointing()
    optimizer = torch.optim.AdamW(params, lr=args.lr, weight_decay=1e-2)
    scaler = torch.amp.GradScaler("cuda", enabled=device == "cuda")

    samples = load_captions(Path(args.data))
    print(f"training on {len(samples)} drawings for {args.steps} steps ({device})")

    def batch():
        picks = random.sample(samples, min(args.batch, len(samples)))
        pixels = []
        for path, _ in picks:
            image = square(Image.open(path).convert("RGB"), size, "white")
            if random.random() < 0.5:
                image = ImageOps.mirror(image)
            array = np.asarray(image, dtype=np.float32) / 127.5 - 1.0
            pixels.append(torch.from_numpy(array).permute(2, 0, 1))
        ids = tokenizer([caption for _, caption in picks], padding="max_length", truncation=True,
                        max_length=tokenizer.model_max_length, return_tensors="pt").input_ids
        return torch.stack(pixels).to(device, dtype), ids.to(device)

    unet.train()
    for step in range(1, args.steps + 1):
        pixels, ids = batch()
        with torch.no_grad():
            latents = vae.encode(pixels).latent_dist.sample() * vae.config.scaling_factor
            hidden = text_encoder(ids)[0]
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, scheduler.config.num_train_timesteps, (latents.shape[0],), device=device).long()
        noisy = scheduler.add_noise(latents, noise, timesteps)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
            pred = unet(noisy, timesteps, hidden).sample
        loss = torch.nn.functional.mse_loss(pred.float(), noise.float())
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        scaler.step(optimizer)
        scaler.update()
        # cosine decay to 10% of the starting rate
        for group in optimizer.param_groups:
            group["lr"] = args.lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * step / args.steps)))
        if step % max(1, args.steps // 20) == 0 or step == 1:
            print(f"step {step}/{args.steps} loss {loss.item():.4f}")

    out = Path(args.lora)
    out.mkdir(parents=True, exist_ok=True)
    layers = convert_state_dict_to_diffusers(get_peft_model_state_dict(unet))
    StableDiffusionPipeline.save_lora_weights(save_directory=str(out), unet_lora_layers=layers, safe_serialization=True)
    print("saved LoRA to", out)


# --------------------------------------------------------------------------- poses

class PoseReader:
    def __init__(self, tiny: bool):
        self.tiny = tiny
        if not tiny:
            import sys

            # controlnet_aux imports its mediapipe face tool on load, which breaks on new mediapipe releases.
            # OpenPose does not need mediapipe, so hide it while the package loads.
            hidden = sys.modules.get("mediapipe")
            sys.modules["mediapipe"] = None
            try:
                from controlnet_aux import OpenposeDetector
            finally:
                if hidden is None:
                    sys.modules.pop("mediapipe", None)
                else:
                    sys.modules["mediapipe"] = hidden
            self.detector = OpenposeDetector.from_pretrained("lllyasviel/Annotators")

    def __call__(self, image: Image.Image, size: int) -> Image.Image:
        # Pad the frame to a square first so the skeleton keeps the body's real proportions.
        frame = square(image.convert("RGB"), size, (127, 127, 127))
        if self.tiny:  # test stand-in: edges instead of a skeleton
            import cv2
            edges = cv2.Canny(np.asarray(frame.convert("L")), 80, 160)
            return Image.fromarray(edges).convert("RGB")
        pose = self.detector(frame, detect_resolution=size, image_resolution=size,
                             include_body=True, include_hand=False, include_face=False)
        return pose.resize((size, size))


def video_frames(path: Path, count: int) -> list[Image.Image]:
    import cv2
    capture = cv2.VideoCapture(str(path))
    total = int(capture.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
    frames = []
    for index in np.linspace(0, total - 1, num=min(count, total)).astype(int):
        capture.set(cv2.CAP_PROP_POS_FRAMES, int(index))
        ok, frame = capture.read()
        if ok:
            frames.append(Image.fromarray(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)))
    capture.release()
    return frames


def pick_start_end(poses: list[Image.Image]) -> tuple[int, int]:
    """The two sampled frames whose skeletons differ most: the start and end of the movement."""
    small = [np.asarray(p.convert("L").resize((64, 64)), dtype=np.float32) for p in poses]
    usable = [i for i, s in enumerate(small) if (s > 20).mean() > 0.002] or list(range(len(small)))
    best, pair = -1.0, (usable[0], usable[-1])
    for a in usable:
        for b in usable:
            if a < b:
                score = float(np.abs(small[a] - small[b]).mean())
                if score > best:
                    best, pair = score, (a, b)
    return pair


def person_box(skeletons: list[Image.Image], margin: float = 0.3):
    """One square crop box around the skeletons of both frames, with room for head, feet and equipment."""
    union = np.zeros(np.asarray(skeletons[0].convert("L")).shape, bool)
    for skeleton in skeletons:
        union |= np.asarray(skeleton.convert("L")) > 20
    ys, xs = np.nonzero(union)
    if len(xs) < 20:
        return None
    height, width = union.shape
    side = max(xs.max() - xs.min(), ys.max() - ys.min()) * (1 + 2 * margin)
    side = int(min(max(side, min(width, height) * 0.25), width, height))
    cx, cy = (xs.min() + xs.max()) / 2, (ys.min() + ys.max()) / 2
    left = int(min(max(cx - side / 2, 0), width - side))
    top = int(min(max(cy - side / 2, 0), height - side))
    return left, top, left + side, top + side


def poses(args) -> None:
    clips, out = Path(args.clips), Path(args.poses)
    size = 64 if args.tiny else args.resolution
    reader = PoseReader(args.tiny)
    jobs: dict[str, dict] = {}
    for path in sorted(clips.rglob("*")):
        suffix = path.suffix.lower()
        if suffix in VIDEO_TYPES:
            jobs.setdefault(path.stem, {})["video"] = path
        elif suffix in PHOTO_TYPES and "__" in path.stem:
            name, phase = path.stem.rsplit("__", 1)
            if phase.lower() in {"start", "end"}:
                jobs.setdefault(name, {})[phase.lower()] = path
    manifest = []
    for name, job in jobs.items():
        folder = out / slugify(name)
        folder.mkdir(parents=True, exist_ok=True)
        if "start" in job and "end" in job:
            frames = [Image.open(job["start"]), Image.open(job["end"])]
            skeletons = [reader(f, size) for f in frames]
            first, last = 0, 1
        elif "video" in job:
            frames = video_frames(job["video"], args.samples)
            if len(frames) < 2:
                print("skipped (could not read video):", name)
                continue
            skeletons = [reader(f, size) for f in frames]
            first, last = pick_start_end(skeletons)
        else:
            print("skipped (need a video or both __start and __end photos):", name)
            continue
        chosen = [square(frames[i].convert("RGB"), size, (127, 127, 127)) for i in (first, last)]
        box = person_box([skeletons[first], skeletons[last]])
        if box:  # zoom both frames on the same area so the person fills the drawing
            chosen = [frame.crop(box).resize((size, size), Image.LANCZOS) for frame in chosen]
            skeleton_pair = [reader(frame, size) for frame in chosen]
        else:
            skeleton_pair = [skeletons[first], skeletons[last]]
        for phase, frame, skeleton in zip(("start", "end"), chosen, skeleton_pair):
            skeleton.save(folder / f"{phase}-pose.png")
            frame.save(folder / f"{phase}-frame.jpg")
        manifest.append({"name": name, "slug": slugify(name)})
        print("poses ready:", name)
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    print(f"{len(manifest)} exercises have start/end poses in {out}")


# --------------------------------------------------------------------------- poses from a description

# Body proportions as a fraction of standing height, and OpenPose's 18-point drawing order and colours.
LENGTHS = {"torso": 0.30, "head": 0.11, "upper_arm": 0.165, "forearm": 0.15, "thigh": 0.245, "shin": 0.245}
LIMBS = [(2, 3), (2, 6), (3, 4), (4, 5), (6, 7), (7, 8), (2, 9), (9, 10), (10, 11), (2, 12), (12, 13), (13, 14),
         (2, 1), (1, 15), (15, 17), (1, 16), (16, 18)]
COLORS = [(255, 0, 0), (255, 85, 0), (255, 170, 0), (255, 255, 0), (170, 255, 0), (85, 255, 0), (0, 255, 0),
          (0, 255, 85), (0, 255, 170), (0, 255, 255), (0, 170, 255), (0, 85, 255), (0, 0, 255), (85, 0, 255),
          (170, 0, 255), (255, 0, 255), (255, 0, 170), (255, 0, 85)]
FACING = {"up": 90, "down": -90, "left": 180, "right": 0}


def _towards(point, angle: float, length: float):
    return (point[0] + length * math.cos(math.radians(angle)), point[1] + length * math.sin(math.radians(angle)))


def spec_keypoints(frame: dict, view: str) -> list:
    """OpenPose keypoints (x right, y up) from segment angles.

    Angles are degrees in the picture: 0 = towards the side the person faces (right), 90 = up, -90 = down,
    180 = behind. "arms" and "legs" are [[upper, lower], [upper, lower]]: near side first (in front view: the
    limb on the picture's left first). "face" (up/down/left/right) overrides which way the face points.
    """
    side = view != "front"
    hip = (0.0, 0.0)
    neck = _towards(hip, frame["torso"], LENGTHS["torso"])
    head_angle = frame.get("head", frame["torso"])
    across = frame["torso"] - 90  # in front view: from the body's centre towards the picture's right
    if side:
        shoulders = [neck, neck]
        hips = [hip, hip]
    else:
        shoulders = [_towards(neck, across + 180, 0.11), _towards(neck, across, 0.11)]
        hips = [_towards(hip, across + 180, 0.07), _towards(hip, across, 0.07)]
    points: list = [None] * 18
    points[1] = neck
    face = FACING.get(frame.get("face", ""), head_angle - 90)
    top = _towards(neck, head_angle, LENGTHS["head"])
    if side:
        points[0] = _towards(top, face, 0.04)                            # nose, on the face side
        points[15] = _towards(_towards(top, head_angle, 0.025), face, 0.02)  # near eye
        points[17] = _towards(top, face + 180, 0.035)                    # near ear, back of the head
    else:
        points[0] = top
        for eye, ear, sign in ((14, 16, 180), (15, 17, 0)):
            points[eye] = _towards(_towards(top, head_angle, 0.025), across + sign, 0.025)
            points[ear] = _towards(top, across + sign, 0.055)
    # In side view the near limbs are drawn as the body's left side (OpenPose points 5-7 and 11-13).
    arm_slots = [(5, 6, 7), (2, 3, 4)] if side else [(2, 3, 4), (5, 6, 7)]
    leg_slots = [(11, 12, 13), (8, 9, 10)] if side else [(8, 9, 10), (11, 12, 13)]
    for i, (s, e, w) in enumerate(arm_slots):
        upper, lower = frame["arms"][i]
        points[s] = shoulders[i]
        points[e] = _towards(points[s], upper, LENGTHS["upper_arm"])
        points[w] = _towards(points[e], lower, LENGTHS["forearm"])
    for i, (h, k, a) in enumerate(leg_slots):
        thigh, shin = frame["legs"][i]
        points[h] = hips[i]
        points[k] = _towards(points[h], thigh, LENGTHS["thigh"])
        points[a] = _towards(points[k], shin, LENGTHS["shin"])
    if side:
        points[14] = points[16] = None  # far eye and ear are hidden behind the head
    return points


def draw_skeleton(points: list, size: int, box) -> Image.Image:
    import cv2

    (x0, y0), scale = box
    canvas = np.zeros((size, size, 3), np.uint8)
    pixels = [None if p is None else ((p[0] - x0) * scale, size - (p[1] - y0) * scale) for p in points]
    width = max(2, round(size / 128))
    for (a, b), color in zip(LIMBS, COLORS):
        p, q = pixels[a - 1], pixels[b - 1]
        if p is None or q is None:
            continue
        length = math.hypot(p[0] - q[0], p[1] - q[1])
        angle = math.degrees(math.atan2(p[1] - q[1], p[0] - q[0]))
        polygon = cv2.ellipse2Poly((int((p[0] + q[0]) / 2), int((p[1] + q[1]) / 2)), (int(length / 2), width),
                                   int(angle), 0, 360, 1)
        cv2.fillConvexPoly(canvas, polygon, [int(c * 0.6) for c in color])
    for point, color in zip(pixels, COLORS):
        if point is not None:
            cv2.circle(canvas, (int(point[0]), int(point[1])), width, color, thickness=-1)
    return Image.fromarray(canvas)


def specposes(args) -> None:
    """Make the start/end skeletons from written pose descriptions (exercise-poses.json), no video needed."""
    specs = json.loads(Path(args.specs).read_text(encoding="utf-8"))
    out = Path(args.poses)
    size = 64 if args.tiny else args.resolution
    manifest = []
    for spec in specs:
        view = spec.get("view", "side")
        frames = [spec_keypoints(spec[phase], view) for phase in ("start", "end")]
        # One shared scale for both frames, so the person keeps the same size and the loop doesn't jump.
        everything = [p for frame in frames for p in frame if p is not None]
        xs, ys = [p[0] for p in everything], [p[1] for p in everything]
        span = max(max(xs) - min(xs), max(ys) - min(ys), 0.5)
        scale = size * 0.76 / span
        x0 = (min(xs) + max(xs)) / 2 - size / 2 / scale
        y0 = (min(ys) + max(ys)) / 2 - size / 2 / scale
        folder = out / slugify(spec["name"])
        folder.mkdir(parents=True, exist_ok=True)
        for phase, points in zip(("start", "end"), frames):
            draw_skeleton(points, size, ((x0, y0), scale)).save(folder / f"{phase}-pose.png")
        manifest.append({"name": spec["name"], "slug": slugify(spec["name"]), "prompt": spec.get("prompt", "")})
    (out / "manifest.json").write_text(json.dumps(manifest, indent=1), encoding="utf-8")
    print(f"{len(manifest)} exercises have start/end poses in {out}")


# --------------------------------------------------------------------------- generate

def clean(image: Image.Image, size: int) -> Image.Image:
    """Pure black ink on white, cropped to the figure and centred like the library drawings."""
    gray = np.asarray(image.convert("L"))
    ink = gray < 128
    page = Image.fromarray(np.where(ink, 0, 255).astype(np.uint8))
    ys, xs = np.nonzero(ink)
    if len(xs):
        pad = 12
        page = page.crop((max(0, xs.min() - pad), max(0, ys.min() - pad),
                          min(page.width, xs.max() + pad), min(page.height, ys.max() + pad)))
    page = ImageOps.contain(page, (size - 24, size - 24), Image.LANCZOS)
    canvas = Image.new("L", (size, size), 255)
    canvas.paste(page, ((size - page.width) // 2, (size - page.height) // 2))
    return canvas.convert("RGB")


def clutter(image: Image.Image) -> float:
    """How much of a drawing looks like background: ink in the outer band, plus a little for ink overall."""
    ink = np.asarray(image.convert("L")) < 128
    band = max(1, ink.shape[0] // 10)
    edges = np.concatenate([ink[:band].ravel(), ink[-band:].ravel(), ink[:, :band].ravel(), ink[:, -band:].ravel()])
    return float(edges.mean() + 0.3 * ink.mean())


def generate(args) -> None:
    from diffusers import (ControlNetModel, StableDiffusionControlNetPipeline, UNet2DConditionModel,
                           UniPCMultistepScheduler)

    device, dtype = device_and_dtype(args.tiny)
    size = 64 if args.tiny else args.resolution
    if args.tiny:
        from diffusers import AutoencoderKL, DDIMScheduler
        from transformers import CLIPTextModel, CLIPTokenizer

        # The tiny test repo declares Flax classes in model_index.json, so assemble the pipeline by hand.
        unet = UNet2DConditionModel.from_pretrained(TINY_BASE, subfolder="unet")
        pipe = StableDiffusionControlNetPipeline(
            vae=AutoencoderKL.from_pretrained(TINY_BASE, subfolder="vae"),
            text_encoder=CLIPTextModel.from_pretrained(TINY_BASE, subfolder="text_encoder"),
            tokenizer=CLIPTokenizer.from_pretrained(TINY_BASE, subfolder="tokenizer"),
            # the tiny VAE only halves the image, so the pose embedding must downsample once, not 8x
            unet=unet, controlnet=ControlNetModel.from_unet(unet, conditioning_embedding_out_channels=(16, 32)),
            scheduler=DDIMScheduler.from_pretrained(TINY_BASE, subfolder="scheduler"),
            safety_checker=None, feature_extractor=None, requires_safety_checker=False)
    else:
        controlnet = ControlNetModel.from_pretrained(args.controlnet, torch_dtype=dtype)
        pipe = StableDiffusionControlNetPipeline.from_pretrained(args.base, controlnet=controlnet, torch_dtype=dtype,
                                                                 safety_checker=None, requires_safety_checker=False)
    pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config)
    pipe.load_lora_weights(args.lora)
    pipe.to(device)
    pipe.set_progress_bar_config(disable=True)

    poses_dir, out = Path(args.poses), Path(args.out)
    manifest = json.loads((poses_dir / "manifest.json").read_text(encoding="utf-8"))
    if args.limit:
        manifest = manifest[:args.limit]
    results = []
    for entry in manifest:
        folder = out / entry["slug"]
        folder.mkdir(parents=True, exist_ok=True)
        detail = f", {entry['prompt']}" if entry.get("prompt") else ""
        prompt = (f"{TRIGGER}, black ink line drawing of a muscular bald man in shorts doing "
                  f"{entry['name'].lower()}{detail}, isolated figure on plain white background, thick black outlines")
        pairs = []
        for candidate in range(args.candidates):
            seed = args.seed + candidate  # same seed for start and end keeps the two frames alike
            pair = []
            for phase in ("start", "end"):
                pose = Image.open(poses_dir / entry["slug"] / f"{phase}-pose.png").convert("RGB").resize((size, size))
                generator = torch.Generator(device="cpu").manual_seed(seed)
                image = pipe(f"{prompt}, {phase} position", image=pose, negative_prompt=NEGATIVE,
                             num_inference_steps=2 if args.tiny else args.steps, guidance_scale=args.guidance,
                             controlnet_conditioning_scale=args.pose_strength, generator=generator,
                             height=size, width=size).images[0]
                pair.append(clean(image, 480))
            pairs.append(pair)
        # Cleanest pair first: library drawings have empty margins, so ink near the edges means a background.
        pairs.sort(key=lambda pair: sum(clutter(image) for image in pair))
        for rank, pair in enumerate(pairs, 1):
            for phase, image in zip(("start", "end"), pair):
                image.save(folder / f"{phase}-{rank}.png")
        results.append({**entry, "candidates": args.candidates})
        print("drawn:", entry["name"])
    (out / "manifest.json").write_text(json.dumps(results, indent=1), encoding="utf-8")
    contact_sheet(out, results, args.candidates)
    print(f"{len(results)} exercises drawn in {out}")


def contact_sheet(out: Path, results: list[dict], candidates: int) -> None:
    """One review image: each row is an exercise, each pair of columns a start/end candidate."""
    if not results:
        return
    tile = 160
    sheet = Image.new("RGB", (tile * 2 * candidates, tile * len(results)), "white")
    for row, entry in enumerate(results):
        for candidate in range(candidates):
            for col, phase in enumerate(("start", "end")):
                path = out / entry["slug"] / f"{phase}-{candidate + 1}.png"
                if path.exists():
                    sheet.paste(Image.open(path).resize((tile, tile)), ((candidate * 2 + col) * tile, row * tile))
    sheet.save(out / "review.png")


# --------------------------------------------------------------------------- cli

def main(argv=None) -> None:
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("command", choices=["train", "poses", "specposes", "generate"])
    parser.add_argument("--data", default="/kaggle/input", help="folder that contains metadata.jsonl")
    parser.add_argument("--clips", default="/kaggle/input", help="folder with exercise clips or start/end photos")
    parser.add_argument("--specs", default="exercise-poses.json", help="written start/end poses (specposes)")
    parser.add_argument("--lora", default="/kaggle/working/lora")
    parser.add_argument("--poses", default="/kaggle/working/poses")
    parser.add_argument("--out", default="/kaggle/working/drawings")
    parser.add_argument("--base", default=BASE)
    parser.add_argument("--controlnet", default=CONTROLNET)
    parser.add_argument("--resolution", type=int, default=512)
    parser.add_argument("--steps", type=int, default=None, help="training steps (train) or denoising steps (generate)")
    parser.add_argument("--batch", type=int, default=4)
    parser.add_argument("--rank", type=int, default=16)
    parser.add_argument("--lr", type=float, default=1e-4)
    parser.add_argument("--samples", type=int, default=24, help="frames sampled from each video")
    parser.add_argument("--candidates", type=int, default=3, help="drawings per exercise; the cleanest is saved as 1")
    parser.add_argument("--limit", type=int, default=0, help="draw only the first N exercises (0 = all)")
    parser.add_argument("--guidance", type=float, default=7.0)
    parser.add_argument("--pose-strength", type=float, default=1.2)
    parser.add_argument("--seed", type=int, default=1234)
    parser.add_argument("--tiny", action="store_true", help="tiny test models, for checking the code on a CPU")
    args = parser.parse_args(argv)
    if args.steps is None:
        args.steps = {"train": 2000, "generate": 30}.get(args.command, 30)
    {"train": train, "poses": poses, "specposes": specposes, "generate": generate}[args.command](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile exercise-poses.json
[
{
"name": "Glute Bridge",
"view": "side",
"prompt": "lying on his back on the floor, knees bent, feet flat",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 200,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
],
"head": 180
}
},
{
"name": "Barbell Hip Thrust",
"view": "side",
"prompt": "upper back against a flat bench, barbell across the hips",
"start": {
"torso": 145,
"arms": [
[
-35,
-35
],
[
-35,
-35
]
],
"legs": [
[
30,
-55
],
[
30,
-55
]
],
"head": 140
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-100
],
[
0,
-100
]
],
"head": 170
}
},
{
"name": "Kettlebell Swing",
"view": "side",
"prompt": "standing, kettlebell held in both hands",
"start": {
"torso": 30,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 40
},
"end": {
"torso": 90,
"arms": [
[
5,
5
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Sumo Deadlift",
"view": "side",
"prompt": "wide stance, barbell on the floor in front of the shins",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-5,
-100
],
[
-5,
-100
]
],
"head": 60
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Step-Up",
"view": "side",
"prompt": "standing facing a flat bench, one foot on it",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-80
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Lunge",
"view": "side",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-10,
-95
],
[
-105,
-168
]
]
}
},
{
"name": "Stiff-Legged Barbell Deadlift",
"view": "side",
"prompt": "standing, holding a barbell at arm's length",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-86,
-92
],
[
-86,
-92
]
]
}
},
{
"name": "Split Squats",
"view": "side",
"prompt": "standing on a gym floor in a split stance",
"start": {
"torso": 90,
"arms": [
[
-120,
-80
],
[
-120,
-80
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-80
],
[
-120,
-80
]
],
"legs": [
[
-130,
-160
],
[
-40,
-90
]
]
}
},
{
"name": "Burpee",
"view": "side",
"prompt": "bodyweight, on the floor",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
92,
90
]
],
"legs": [
[
-95,
-100
],
[
-90,
-95
]
]
}
},
{
"name": "Mountain Climber",
"view": "side",
"prompt": "push-up plank, hands on the floor",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-40,
190
],
[
203,
203
]
],
"head": 15,
"face": "down"
}
},
{
"name": "Plank",
"view": "side",
"prompt": "forearms and toes on the floor, body straight",
"start": {
"torso": 8,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
210,
180
],
[
210,
180
]
]
},
"end": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Russian Twist",
"view": "side",
"prompt": "seated on the floor, knees bent, leaning back, holding a dumbbell",
"start": {
"torso": 120,
"arms": [
[
-20,
0
],
[
-20,
0
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 120,
"arms": [
[
-50,
-70
],
[
-50,
-70
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
}
},
{
"name": "Rear Delt Fly",
"view": "side",
"prompt": "bent forward at the hips, a dumbbell in each hand",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Side Lateral Raise",
"view": "front",
"prompt": "standing upright, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
178,
172
],
[
2,
8
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Clap Push-UP",
"view": "side",
"prompt": "push-up position on the floor, hands under the shoulders",
"start": {
"torso": 23,
"head": 31,
"arms": [
[
-100,
-77
],
[
-100,
-77
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 23,
"head": 31,
"arms": [
[
-65,
-20
],
[
-63,
-18
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
}
},
{
"name": "Incline Push-Up",
"view": "side",
"prompt": "body straight, hands on the edge of a bench, feet on the floor",
"start": {
"torso": 44,
"head": 52,
"arms": [
[
-110,
-63
],
[
-110,
-63
]
],
"legs": [
[
224,
224
],
[
224,
224
]
],
"face": "down"
},
"end": {
"torso": 30,
"head": 38,
"arms": [
[
-181,
-66
],
[
-181,
-66
]
],
"legs": [
[
210,
210
],
[
210,
210
]
],
"face": "down"
}
},
{
"name": "Pike Push Ups",
"view": "side",
"prompt": "hips high in an inverted V, hands and feet on the floor",
"start": {
"torso": -45,
"head": -45,
"arms": [
[
-45,
-45
],
[
-45,
-45
]
],
"legs": [
[
-117,
-117
],
[
-117,
-117
]
]
},
"end": {
"torso": -70,
"head": -90,
"arms": [
[
-156.7,
-35.9
],
[
-156.7,
-35.9
]
],
"legs": [
[
-117,
-117
],
[
-117,
-117
]
],
"face": "right"
}
},
{
"name": "Hanging Knee Raise",
"view": "side",
"prompt": "hanging from a pull-up bar, arms straight",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
20,
-90
],
[
20,
-90
]
]
}
},
{
"name": "Seated Calf Raise",
"view": "side",
"prompt": "seated on a bench, a dumbbell resting on each knee",
"start": {
"torso": 68,
"head": 60,
"arms": [
[
-53,
-77
],
[
-53,
-77
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 68,
"head": 60,
"arms": [
[
-33,
-93
],
[
-33,
-93
]
],
"legs": [
[
9,
-94
],
[
9,
-94
]
]
}
},
{
"name": "Box Jump",
"view": "side",
"prompt": "standing in front of a tall plyo box",
"start": {
"torso": 70,
"head": 80,
"arms": [
[
-120,
-120
],
[
-120,
-120
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
},
"end": {
"torso": 75,
"head": 85,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-30,
-110
],
[
-30,
-110
]
]
}
},
{
"name": "Hip Circles (prone)",
"view": "side",
"prompt": "on hands and knees on the floor",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-170
],
[
-90,
180
]
],
"head": 10
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
100
],
[
-90,
180
]
],
"head": 10
}
},
{
"name": "IT Band and Glute Stretch",
"view": "side",
"prompt": "lying face up on the floor, a strap looped around one foot",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
55,
50
],
[
55,
50
]
],
"legs": [
[
70,
70
],
[
0,
0
]
],
"head": 170
}
},
{
"name": "Iliotibial Tract-SMR",
"view": "side",
"prompt": "lying on his side, foam roller under the outer thigh, propped on forearm",
"start": {
"torso": 165,
"arms": [
[
-90,
180
],
[
-60,
-20
]
],
"legs": [
[
-10,
-10
],
[
10,
-30
]
],
"head": 165
},
"end": {
"torso": 172,
"arms": [
[
-75,
175
],
[
-60,
-20
]
],
"legs": [
[
-25,
-15
],
[
15,
-35
]
],
"head": 170
}
},
{
"name": "Lying Crossover",
"view": "side",
"prompt": "lying face up on the floor, arms out to the sides",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-30
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Monster Walk",
"view": "front",
"prompt": "standing in a half squat, resistance bands around knees and ankles",
"start": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-125,
-110
],
[
-85,
-95
]
]
}
},
{
"name": "Standing Hip Circles",
"view": "front",
"prompt": "standing on one leg, one hand holding a support post",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-15,
-60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-15,
-60
]
],
"legs": [
[
175,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Windmills",
"view": "side",
"prompt": "lying face up on the floor, arms out to the sides",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
80,
80
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Adductor",
"view": "side",
"prompt": "lying face down on forearms, foam roller under the inner thigh",
"start": {
"torso": 10,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-45,
190
],
[
180,
180
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 5,
"arms": [
[
-85,
0
],
[
-85,
0
]
],
"legs": [
[
0,
175
],
[
180,
180
]
],
"head": 10,
"face": "down"
}
},
{
"name": "Adductor/Groin",
"view": "side",
"prompt": "lying face up on the floor, legs raised toward the ceiling",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
90,
90
],
[
90,
90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
60
],
[
120,
120
]
],
"head": 180
}
},
{
"name": "Band Hip Adductions",
"view": "front",
"prompt": "standing, resistance band around one ankle, anchored to a post",
"start": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-90,
-90
],
[
-60,
-60
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-90,
-90
],
[
-110,
-110
]
]
}
},
{
"name": "Carioca Quick Step",
"view": "front",
"prompt": "standing in an athletic stance on the floor",
"start": {
"torso": 90,
"arms": [
[
120,
80
],
[
-60,
-100
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-80
],
[
60,
100
]
],
"legs": [
[
-95,
-90
],
[
200,
-90
]
]
}
},
{
"name": "Groin and Back Stretch",
"view": "side",
"prompt": "sitting on the floor, soles together, hands behind head",
"start": {
"torso": 90,
"arms": [
[
45,
170
],
[
45,
170
]
],
"legs": [
[
15,
-165
],
[
15,
-165
]
],
"head": 90
},
"end": {
"torso": 35,
"arms": [
[
-10,
115
],
[
-10,
115
]
],
"legs": [
[
15,
-165
],
[
15,
-165
]
],
"head": -10
}
},
{
"name": "Groiners",
"view": "side",
"prompt": "push-up position on the floor",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 30
},
"end": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
20,
-122
],
[
20,
-122
]
],
"head": 50
}
},
{
"name": "Lateral Bound",
"view": "front",
"prompt": "half squat on the floor, arms ready",
"start": {
"torso": 90,
"arms": [
[
-120,
-100
],
[
-60,
-80
]
],
"legs": [
[
-100,
-80
],
[
-80,
-100
]
]
},
"end": {
"torso": 85,
"arms": [
[
150,
120
],
[
-60,
-80
]
],
"legs": [
[
-135,
-90
],
[
-80,
-95
]
]
}
},
{
"name": "Lateral Box Jump",
"view": "front",
"prompt": "standing on the floor beside a low box",
"start": {
"torso": 90,
"arms": [
[
-110,
-100
],
[
-70,
-80
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
130,
110
],
[
50,
70
]
],
"legs": [
[
-135,
-60
],
[
-45,
-120
]
]
}
},
{
"name": "Lateral Cone Hops",
"view": "front",
"prompt": "standing beside a row of cones on the floor",
"start": {
"torso": 90,
"arms": [
[
-115,
-80
],
[
-65,
-100
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
-150,
-60
],
[
-30,
-120
]
],
"legs": [
[
-130,
-70
],
[
-50,
-110
]
]
}
},
{
"name": "Lying Bent Leg Groin",
"view": "side",
"prompt": "lying face up on the floor, knees bent, soles of feet together",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
70,
-50
],
[
70,
-50
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
25,
-20
],
[
25,
-20
]
],
"head": 180
}
},
{
"name": "Side Leg Raises",
"view": "front",
"prompt": "standing beside a chair, one hand on its back",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-40,
-60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 88,
"arms": [
[
-95,
-90
],
[
-40,
-60
]
],
"legs": [
[
-140,
-140
],
[
-90,
-90
]
]
}
},
{
"name": "Side Lying Groin Stretch",
"view": "front",
"prompt": "lying on his side on the floor, head resting on hand",
"start": {
"torso": 180,
"arms": [
[
170,
180
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 170
},
"end": {
"torso": 180,
"arms": [
[
170,
180
],
[
20,
20
]
],
"legs": [
[
0,
0
],
[
70,
70
]
],
"head": 170
}
},
{
"name": "Alternating Kettlebell Row",
"view": "side",
"prompt": "bent over at the hips, a kettlebell in each hand",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Alternating Renegade Row",
"view": "side",
"prompt": "push-up position, hands on two kettlebells on the floor",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 30
},
"end": {
"torso": 23,
"arms": [
[
190,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 30
}
},
{
"name": "Band Assisted Pull-Up",
"view": "side",
"prompt": "hanging from a pull-up bar, one knee in a looped band",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-80,
-120
],
[
-80,
-120
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-80,
-120
],
[
-80,
-120
]
]
}
},
{
"name": "Band Row",
"view": "side",
"prompt": "standing, holding a resistance band anchored in front",
"start": {
"torso": 90,
"arms": [
[
-5,
-5
],
[
-5,
-5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-130,
0
],
[
-130,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Bent Over Dumbbell Rows (Single Arm)",
"view": "side",
"prompt": "one knee and hand on a flat bench, dumbbell in the other hand",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-90
],
[
-90,
180
]
],
"head": 20
},
"end": {
"torso": 13,
"arms": [
[
195,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-90
],
[
-90,
180
]
],
"head": 20
}
},
{
"name": "Bent Over One-Arm Long Bar Row",
"view": "side",
"prompt": "bent over at the hips, one hand on the end of a landmine barbell",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
},
"end": {
"torso": 25,
"arms": [
[
200,
-85
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
}
},
{
"name": "Bent Over Two-Arm Long Bar Row",
"view": "side",
"prompt": "bent over at the hips, both hands on a landmine barbell end",
"start": {
"torso": 35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 40
},
"end": {
"torso": 35,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 40
}
},
{
"name": "Bent Over Two-Dumbbell Row",
"view": "side",
"prompt": "bent over at the hips, a dumbbell in each hand",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Bent Over Two-Dumbbell Row With Palms In",
"view": "side",
"prompt": "bent over at the hips, a dumbbell in each hand, palms facing in",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Blackroll",
"view": "side",
"prompt": "lying face up, foam roller under the back, knees bent, feet flat",
"start": {
"torso": 170,
"arms": [
[
30,
170
],
[
30,
170
]
],
"legs": [
[
45,
-60
],
[
45,
-60
]
],
"head": 165
},
"end": {
"torso": 190,
"arms": [
[
10,
170
],
[
10,
170
]
],
"legs": [
[
10,
-60
],
[
10,
-60
]
],
"head": 165
}
},
{
"name": "Bodyweight Mid Row",
"view": "side",
"prompt": "hanging from a pull-up bar, knees tucked, leaning back",
"start": {
"torso": 130,
"arms": [
[
60,
60
],
[
60,
60
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 110
},
"end": {
"torso": 130,
"arms": [
[
200,
80
],
[
200,
80
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 110
}
},
{
"name": "Catch and Overhead Throw",
"view": "side",
"prompt": "standing, holding a medicine ball with both hands",
"start": {
"torso": 95,
"arms": [
[
100,
160
],
[
100,
160
]
],
"legs": [
[
-70,
-90
],
[
-110,
-100
]
],
"head": 95
},
"end": {
"torso": 80,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
-70,
-90
],
[
-110,
-100
]
],
"head": 80
}
},
{
"name": "Chair Lower Back Stretch",
"view": "front",
"prompt": "sitting upright on a chair, feet on the floor",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
70,
20
],
[
-80,
-90
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
],
"head": 60
}
},
{
"name": "Chest-Supported Row",
"view": "side",
"prompt": "chest on an incline bench, a dumbbell in each hand",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
},
"end": {
"torso": 45,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
}
},
{
"name": "Close-Grip Front Lat Pulldown",
"view": "side",
"prompt": "seated at a pulldown machine, holding the bar overhead",
"start": {
"torso": 95,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-110,
75
],
[
-108,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Close-grip Lat Pull Down",
"view": "side",
"prompt": "seated at a pulldown machine, narrow grip on the bar",
"start": {
"torso": 95,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-110,
75
],
[
-108,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Close-grip Supinated Lat Pulldown",
"view": "side",
"prompt": "seated at a pulldown machine, narrow underhand grip on the bar",
"start": {
"torso": 95,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-110,
75
],
[
-108,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Cross-Bench Dumbbell Pullovers",
"view": "side",
"prompt": "upper back across a flat bench, feet on the floor, holding a dumbbell",
"start": {
"torso": 162,
"arms": [
[
185,
185
],
[
185,
185
]
],
"legs": [
[
-20,
-90
],
[
-20,
-90
]
],
"head": 170
},
"end": {
"torso": 162,
"arms": [
[
85,
85
],
[
85,
85
]
],
"legs": [
[
-20,
-90
],
[
-20,
-90
]
],
"head": 170
}
},
{
"name": "Dumbbell Incline Row",
"view": "side",
"prompt": "chest on an incline bench, a dumbbell in each hand",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
},
"end": {
"torso": 45,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
}
},
{
"name": "Dynamic Back Stretch",
"view": "side",
"prompt": "standing on the floor, arms straight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 100
}
},
{
"name": "Elevated Cable Rows",
"view": "side",
"prompt": "seated at a low cable row on a raised platform, holding a handle",
"start": {
"torso": 75,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 75
},
"end": {
"torso": 95,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 95
}
},
{
"name": "Face Pulls with Yellow/green Band",
"view": "side",
"prompt": "standing, holding a resistance band anchored at face height",
"start": {
"torso": 90,
"arms": [
[
5,
5
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
30
],
[
180,
30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Full Range-Of-Motion Lat Pulldown",
"view": "side",
"prompt": "seated on a high bench between two high cable pulleys",
"start": {
"torso": 95,
"arms": [
[
80,
80
],
[
80,
80
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-100,
60
],
[
-100,
60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Gironda Sternum Chins",
"view": "side",
"prompt": "hanging from a pull-up bar, underhand grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-75,
-85
],
[
-75,
-85
]
]
},
"end": {
"torso": 135,
"arms": [
[
200,
25
],
[
200,
25
]
],
"legs": [
[
-45,
-65
],
[
-45,
-65
]
],
"head": 110
}
},
{
"name": "Head Tilts",
"view": "front",
"prompt": "standing upright, arms relaxed at the sides",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 60
}
},
{
"name": "High-pulley Pullover",
"view": "side",
"prompt": "standing facing a high cable pulley, holding a straight bar",
"start": {
"torso": 80,
"arms": [
[
60,
60
],
[
60,
60
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
],
"head": 80
},
"end": {
"torso": 80,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
],
"head": 80
}
},
{
"name": "Incline Bench Pull",
"view": "side",
"prompt": "lying face down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 30
},
"end": {
"torso": 30,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 30
}
},
{
"name": "Incline Bench Pulldown",
"view": "side",
"prompt": "lying face down on an incline bench facing a cable machine",
"start": {
"torso": 40,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 40
},
"end": {
"torso": 40,
"arms": [
[
-100,
40
],
[
-100,
40
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 40
}
},
{
"name": "Inverted Row",
"view": "side",
"prompt": "hanging face up under a low bar, heels on the floor",
"start": {
"torso": 165,
"arms": [
[
75,
75
],
[
75,
75
]
],
"legs": [
[
-15,
-15
],
[
-15,
-15
]
],
"head": 165
},
"end": {
"torso": 150,
"arms": [
[
-120,
60
],
[
-120,
60
]
],
"legs": [
[
-30,
-30
],
[
-30,
-30
]
],
"head": 150
}
},
{
"name": "Inverted Row with Straps",
"view": "side",
"prompt": "hanging face up from suspension straps, heels on the floor",
"start": {
"torso": 165,
"arms": [
[
75,
75
],
[
75,
75
]
],
"legs": [
[
-15,
-15
],
[
-15,
-15
]
],
"head": 165
},
"end": {
"torso": 150,
"arms": [
[
-120,
60
],
[
-120,
60
]
],
"legs": [
[
-30,
-30
],
[
-30,
-30
]
],
"head": 150
}
},
{
"name": "Kipping Muscle Up",
"view": "side",
"prompt": "hanging from gymnastic rings, false grip",
"start": {
"torso": 85,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-110,
-110
],
[
-110,
-110
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
-80,
-80
],
[
-80,
-80
]
]
}
},
{
"name": "Kneeling High Pulley Row",
"view": "side",
"prompt": "kneeling in front of a high cable pulley, holding a rope",
"start": {
"torso": 90,
"arms": [
[
50,
50
],
[
50,
50
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
190,
45
],
[
190,
45
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Kneeling Single-Arm High Pulley Row",
"view": "side",
"prompt": "kneeling in front of a high cable pulley, holding a single handle",
"start": {
"torso": 90,
"arms": [
[
50,
50
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
190,
45
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Lat Pulldown",
"view": "side",
"prompt": "seated at a pulldown machine, holding the bar overhead",
"start": {
"torso": 95,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-110,
75
],
[
-108,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Latissimus Dorsi-SMR",
"view": "side",
"prompt": "lying on his side, foam roller under the armpit, arm overhead",
"start": {
"torso": 165,
"arms": [
[
180,
180
],
[
-30,
-10
]
],
"legs": [
[
-5,
-5
],
[
20,
-40
]
],
"head": 165
},
"end": {
"torso": 175,
"arms": [
[
175,
175
],
[
-30,
-10
]
],
"legs": [
[
-15,
-15
],
[
10,
-30
]
],
"head": 175
}
},
{
"name": "Leverage High Row",
"view": "side",
"prompt": "seated at a high row machine, handles above and in front",
"start": {
"torso": 90,
"arms": [
[
60,
60
],
[
60,
60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-130,
65
],
[
-130,
65
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Leverage Iso Row",
"view": "side",
"prompt": "seated at a row machine, chest against the pad, holding handles",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "London Bridges",
"view": "side",
"prompt": "hanging from a climbing rope, heels resting on a locked bar",
"start": {
"torso": 150,
"arms": [
[
80,
90
],
[
80,
90
]
],
"legs": [
[
-30,
-30
],
[
-30,
-30
]
],
"head": 150
},
"end": {
"torso": 105,
"arms": [
[
-100,
85
],
[
-100,
85
]
],
"legs": [
[
-75,
-75
],
[
-75,
-75
]
],
"head": 100
}
},
{
"name": "Long-Pulley (low Row)",
"view": "side",
"prompt": "seated at a low cable row, feet on the foot plates, holding a bar",
"start": {
"torso": 75,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 75
},
"end": {
"torso": 95,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 95
}
},
{
"name": "Lower Back Extensions",
"view": "side",
"prompt": "on a 45-degree back extension bench, hips on the pad, arms crossed",
"start": {
"torso": -50,
"arms": [
[
160,
10
],
[
160,
10
]
],
"legs": [
[
-135,
-135
],
[
-135,
-135
]
],
"head": -50
},
"end": {
"torso": 45,
"arms": [
[
-105,
105
],
[
-105,
105
]
],
"legs": [
[
-135,
-135
],
[
-135,
-135
]
],
"head": 50
}
},
{
"name": "Lying Cambered Barbell Row",
"view": "side",
"prompt": "lying face down on a raised flat bench, holding a barbell below",
"start": {
"torso": 0,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
165,
-90
],
[
165,
-90
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Lying T-Bar Row",
"view": "side",
"prompt": "chest on the pad of a T-bar row machine, holding the handles",
"start": {
"torso": 35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 35
},
"end": {
"torso": 35,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 35
}
},
{
"name": "Middle Back Shrug",
"view": "side",
"prompt": "lying face down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 40
},
"end": {
"torso": 40,
"arms": [
[
-120,
-120
],
[
-120,
-120
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 40
}
},
{
"name": "Middle Back Stretch",
"view": "front",
"prompt": "standing, feet shoulder width, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-15
],
[
-30,
-160
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"face": "right"
}
},
{
"name": "Mixed Grip Chin",
"view": "side",
"prompt": "hanging from a pull-up bar, one palm forward and one palm back",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 100
}
},
{
"name": "Modified Pulldown",
"view": "side",
"prompt": "lying face down on an incline bench in front of a pulldown machine",
"start": {
"torso": 40,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 40
},
"end": {
"torso": 40,
"arms": [
[
-100,
40
],
[
-100,
40
]
],
"legs": [
[
190,
-100
],
[
190,
-100
]
],
"head": 40
}
},
{
"name": "Muscle Up",
"view": "side",
"prompt": "hanging from gymnastic rings, false grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
-85,
-85
],
[
-85,
-85
]
]
}
},
{
"name": "Neutral-grip pull-ups or TRX rows",
"view": "side",
"prompt": "hanging from a pull-up bar, palms facing each other",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 100
}
},
{
"name": "One Arm Against Wall",
"view": "side",
"prompt": "standing facing a wall, one bent arm on the wall overhead",
"start": {
"torso": 90,
"arms": [
[
70,
90
],
[
-90,
-90
]
],
"legs": [
[
-100,
-95
],
[
-100,
-95
]
]
},
"end": {
"torso": 65,
"arms": [
[
105,
95
],
[
-100,
-100
]
],
"legs": [
[
-100,
-95
],
[
-100,
-95
]
],
"head": 55
}
},
{
"name": "One Arm Bent Row",
"view": "side",
"prompt": "bent over at the hips, one hand on a low cable handle",
"start": {
"torso": 25,
"arms": [
[
-45,
-45
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
},
"end": {
"torso": 25,
"arms": [
[
200,
-60
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
}
},
{
"name": "One Arm Chin-Up",
"view": "side",
"prompt": "hanging from a pull-up bar by one hand, other hand on a towel",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
80,
60
]
],
"legs": [
[
-95,
-110
],
[
-95,
-110
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-100,
70
]
],
"legs": [
[
-95,
-110
],
[
-95,
-110
]
],
"head": 100
}
},
{
"name": "One Arm Lat Pulldown",
"view": "side",
"prompt": "seated at a pulldown machine, one hand on a single handle overhead",
"start": {
"torso": 95,
"arms": [
[
85,
90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "One Handed Hang",
"view": "side",
"prompt": "one hand gripping a pull-up bar, feet resting on a step",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-115
],
[
-70,
-115
]
]
}
},
{
"name": "One-Arm Dumbbell Row",
"view": "side",
"prompt": "one knee and hand on a flat bench, dumbbell in the other hand",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-90
],
[
-90,
180
]
],
"head": 20
},
"end": {
"torso": 13,
"arms": [
[
195,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-90
],
[
-90,
180
]
],
"head": 20
}
},
{
"name": "One-Arm Kettlebell Row",
"view": "side",
"prompt": "bent over at the hips, a kettlebell in one hand",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "One-Arm Long Bar Row",
"view": "side",
"prompt": "bent over beside a landmine barbell, one hand on the bar end",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
},
"end": {
"torso": 25,
"arms": [
[
200,
-85
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-95,
-95
]
],
"head": 35
}
},
{
"name": "Overhead Lat",
"view": "side",
"prompt": "sitting upright on the floor, legs crossed",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
5,
175
],
[
5,
175
]
]
},
"end": {
"torso": 90,
"arms": [
[
105,
-85
],
[
-90,
-90
]
],
"legs": [
[
5,
175
],
[
5,
175
]
]
}
},
{
"name": "Overhead Slam",
"view": "side",
"prompt": "standing, holding a medicine ball with both hands",
"start": {
"torso": 95,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
},
"end": {
"torso": 45,
"arms": [
[
-75,
-80
],
[
-75,
-80
]
],
"legs": [
[
-45,
-105
],
[
-45,
-105
]
],
"head": 30
}
},
{
"name": "Pull Ups on Machine",
"view": "side",
"prompt": "kneeling on the pad of an assisted pull-up machine, gripping handles overhead",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 100
}
},
{
"name": "Pullups",
"view": "side",
"prompt": "hanging from a pull-up bar, overhand grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 100
}
},
{
"name": "Remo Maquina Agarre Estrecho",
"view": "side",
"prompt": "seated at a row machine, narrow grip on the handles",
"start": {
"torso": 85,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Remo Maquina Agarre Estrecho Supino",
"view": "side",
"prompt": "seated at a row machine, narrow underhand grip on the handles",
"start": {
"torso": 85,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Rhomboids-SMR",
"view": "side",
"prompt": "lying face up, foam roller under the upper back, arms crossed, knees bent",
"start": {
"torso": 172,
"arms": [
[
40,
170
],
[
40,
170
]
],
"legs": [
[
45,
-60
],
[
45,
-60
]
],
"head": 160
},
"end": {
"torso": 188,
"arms": [
[
25,
175
],
[
25,
175
]
],
"legs": [
[
10,
-60
],
[
10,
-60
]
],
"head": 165
}
},
{
"name": "Rocky Pull-Ups/Pulldowns",
"view": "side",
"prompt": "hanging from a pull-up bar, wide overhand grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 85,
"arms": [
[
-95,
105
],
[
-95,
105
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 70
}
},
{
"name": "Rope Climb",
"view": "side",
"prompt": "gripping a vertical climbing rope overhead",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
85,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
80
],
[
90,
90
]
],
"legs": [
[
-30,
-100
],
[
-40,
-110
]
]
}
},
{
"name": "Rope Pullover/row",
"view": "side",
"prompt": "chest on a flat bench in front of a low cable, holding a rope",
"start": {
"torso": 10,
"arms": [
[
-10,
-10
],
[
-10,
-10
]
],
"legs": [
[
-95,
-95
],
[
-95,
-95
]
],
"head": 15
},
"end": {
"torso": 10,
"arms": [
[
-140,
-170
],
[
-140,
-170
]
],
"legs": [
[
-95,
-95
],
[
-95,
-95
]
],
"head": 15
}
},
{
"name": "Rope Straight-Arm Pulldown",
"view": "side",
"prompt": "standing in a staggered stance, holding a rope from a high pulley",
"start": {
"torso": 75,
"arms": [
[
50,
50
],
[
50,
50
]
],
"legs": [
[
-80,
-90
],
[
-105,
-100
]
],
"head": 70
},
"end": {
"torso": 75,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-80,
-90
],
[
-105,
-100
]
],
"head": 70
}
},
{
"name": "Rowing Seated, Narrow Grip",
"view": "side",
"prompt": "seated at a low cable row, feet on the foot plates, narrow handle",
"start": {
"torso": 75,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 75
},
"end": {
"torso": 92,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 92
}
},
{
"name": "Seated Back Extension Machine",
"view": "side",
"prompt": "seated in a back extension machine, pad across the upper back",
"start": {
"torso": 50,
"arms": [
[
-100,
110
],
[
-100,
110
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 110,
"arms": [
[
-40,
170
],
[
-40,
170
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 105
}
},
{
"name": "Seated One-arm Cable Pulley Rows",
"view": "side",
"prompt": "seated at a low cable row, feet braced, one hand on a handle",
"start": {
"torso": 80,
"arms": [
[
0,
0
],
[
-80,
-20
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 80
},
"end": {
"torso": 95,
"arms": [
[
-150,
0
],
[
-80,
-20
]
],
"legs": [
[
15,
-15
],
[
15,
-15
]
],
"head": 95
}
},
{
"name": "Seated Rear Delt Rise",
"view": "front",
"prompt": "seated on the end of a bench, leaning forward, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-100,
-95
],
[
-80,
-85
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
178,
172
],
[
2,
8
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
],
"head": 80
}
},
{
"name": "Seated Row (Machine)",
"view": "side",
"prompt": "seated at a row machine, chest against the pad, holding handles",
"start": {
"torso": 88,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
-150,
0
],
[
-150,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Shotgun Row",
"view": "side",
"prompt": "standing in a wide split stance, one hand on a low cable handle",
"start": {
"torso": 80,
"arms": [
[
-30,
-30
],
[
-90,
-90
]
],
"legs": [
[
-70,
-90
],
[
-115,
-100
]
],
"head": 80
},
"end": {
"torso": 85,
"arms": [
[
200,
-60
],
[
-90,
-90
]
],
"legs": [
[
-70,
-90
],
[
-115,
-100
]
],
"head": 85
}
},
{
"name": "Side Stretch",
"view": "front",
"prompt": "standing upright, feet hip width apart",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
70,
20
],
[
-75,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 62
}
},
{
"name": "Side To Side Chins",
"view": "front",
"prompt": "hanging from a pull-up bar, wide overhand grip",
"start": {
"torso": 90,
"arms": [
[
125,
125
],
[
55,
55
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
162,
162
],
[
-10,
128
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 85
}
},
{
"name": "Side-Lying Floor Stretch",
"view": "front",
"prompt": "lying on his side on the floor, bottom knee bent",
"start": {
"torso": 180,
"arms": [
[
170,
180
],
[
-10,
0
]
],
"legs": [
[
0,
-20
],
[
0,
0
]
],
"head": 170
},
"end": {
"torso": 180,
"arms": [
[
170,
180
],
[
170,
180
]
],
"legs": [
[
0,
-20
],
[
-5,
-5
]
],
"head": 170
}
},
{
"name": "Single Arm Row",
"view": "side",
"prompt": "sitting on the floor, one hand on a high cable handle",
"start": {
"torso": 90,
"arms": [
[
70,
70
],
[
-90,
-90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
70
],
[
-90,
-90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Sled Row",
"view": "side",
"prompt": "standing facing a weighted sled, holding two rope handles",
"start": {
"torso": 80,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 80
},
"end": {
"torso": 85,
"arms": [
[
-150,
-20
],
[
-150,
-20
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 85
}
},
{
"name": "Smith Machine Bent Over Row",
"view": "side",
"prompt": "bent over at the hips, holding a smith machine bar",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Spinal Stretch",
"view": "front",
"prompt": "seated on a chair, fingers interlaced behind the head",
"start": {
"torso": 90,
"arms": [
[
150,
-5
],
[
30,
185
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
120,
-30
],
[
10,
195
]
],
"legs": [
[
-135,
-90
],
[
-45,
-90
]
],
"face": "right"
}
},
{
"name": "Straight Bar Bench Mid Rows",
"view": "side",
"prompt": "standing on a bench, bent over a barbell resting on its end",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 35
},
"end": {
"torso": 25,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 35
}
},
{
"name": "Straight-Arm Pulldown",
"view": "side",
"prompt": "standing, holding a straight bar from a high cable pulley",
"start": {
"torso": 75,
"arms": [
[
50,
50
],
[
50,
50
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
],
"head": 70
},
"end": {
"torso": 75,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
],
"head": 70
}
},
{
"name": "Suspended Row",
"view": "side",
"prompt": "leaning back holding suspension strap handles, heels on the floor",
"start": {
"torso": 120,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
-60,
-60
],
[
-60,
-60
]
],
"head": 115
},
"end": {
"torso": 115,
"arms": [
[
225,
50
],
[
225,
50
]
],
"legs": [
[
-65,
-65
],
[
-65,
-65
]
],
"head": 110
}
},
{
"name": "T-Bar Row with Handle",
"view": "side",
"prompt": "bent over a landmine barbell, holding a V-handle",
"start": {
"torso": 35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 40
},
"end": {
"torso": 35,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 40
}
},
{
"name": "Two-Arm Kettlebell Row",
"view": "side",
"prompt": "bent over at the hips, a kettlebell in each hand",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
-85
],
[
198,
-88
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Unilateral Cable row",
"view": "side",
"prompt": "seated on an upright bench, one hand on a cable handle",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
-80,
-30
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-150,
0
],
[
-80,
-30
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Upper Back Stretch",
"view": "side",
"prompt": "standing, fingers clasped together",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 65
}
},
{
"name": "V-Bar Pullup",
"view": "side",
"prompt": "hanging from a V-handle on a pull-up bar",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 105,
"arms": [
[
-120,
75
],
[
-120,
75
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 110
}
},
{
"name": "Weighted Pull Ups",
"view": "side",
"prompt": "hanging from a pull-up bar, weight plate on a dip belt",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 100
}
},
{
"name": "Wide-Grip Lat Pulldown",
"view": "side",
"prompt": "seated at a pulldown machine, wide grip on the bar overhead",
"start": {
"torso": 95,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
},
"end": {
"torso": 100,
"arms": [
[
-110,
75
],
[
-108,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 95
}
},
{
"name": "Wide-Grip Pulldown Behind The Neck",
"view": "side",
"prompt": "seated at a pulldown machine, wide grip on the bar overhead",
"start": {
"torso": 85,
"arms": [
[
85,
88
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 85
},
"end": {
"torso": 78,
"arms": [
[
-95,
105
],
[
-95,
105
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 60
}
},
{
"name": "Wide-Grip Rear Pull-Up",
"view": "side",
"prompt": "hanging from a pull-up bar, wide overhand grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
]
},
"end": {
"torso": 82,
"arms": [
[
-95,
105
],
[
-95,
105
]
],
"legs": [
[
-100,
-130
],
[
-100,
-130
]
],
"head": 65
}
},
{
"name": "Rack Pull",
"view": "side",
"prompt": "standing at a power rack, barbell on pins at knee height",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-75,
-95
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Axe Hold",
"view": "front",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
178,
178
],
[
2,
2
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Barbell Curl",
"view": "side",
"prompt": "standing, holding a barbell with an underhand grip",
"start": {
"torso": 90,
"arms": [
[
-88,
-85
],
[
-88,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Barbell Curls Lying Against An Incline",
"view": "side",
"prompt": "chest on an incline bench, holding a barbell below",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
},
"end": {
"torso": 45,
"arms": [
[
-90,
70
],
[
-90,
70
]
],
"legs": [
[
-110,
-100
],
[
-110,
-100
]
],
"head": 45
}
},
{
"name": "Barbell Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, holding a barbell",
"start": {
"torso": 60,
"arms": [
[
-80,
-15
],
[
-80,
-15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 60,
"arms": [
[
-80,
10
],
[
-80,
10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Biceps Curls With SZ-bar",
"view": "side",
"prompt": "standing, holding an EZ curl bar",
"start": {
"torso": 90,
"arms": [
[
-88,
-85
],
[
-88,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Biceps with TRX",
"view": "side",
"prompt": "leaning back holding suspension strap handles, heels on the floor",
"start": {
"torso": 120,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
-60,
-60
],
[
-60,
-60
]
],
"head": 115
},
"end": {
"torso": 105,
"arms": [
[
40,
170
],
[
40,
170
]
],
"legs": [
[
-75,
-75
],
[
-75,
-75
]
],
"head": 100
}
},
{
"name": "Brachialis-SMR",
"view": "side",
"prompt": "lying on his side, upper arm resting on a foam roller on the floor",
"start": {
"torso": 165,
"head": 170,
"arms": [
[
178,
178
],
[
-60,
-80
]
],
"legs": [
[
0,
0
],
[
2,
2
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
178,
178
],
[
-70,
-60
]
],
"legs": [
[
-12,
-12
],
[
-10,
-10
]
]
}
},
{
"name": "Cable Concentration Curl",
"view": "side",
"prompt": "seated on a bench leaning forward, one hand on a low cable handle",
"start": {
"torso": 60,
"head": 40,
"arms": [
[
-80,
-82
],
[
-40,
-60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 60,
"head": 40,
"arms": [
[
-80,
100
],
[
-40,
-60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Cable Curl",
"view": "side",
"prompt": "standing at a low cable pulley, holding a straight bar",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Hammer Curls - Rope Attachment",
"view": "side",
"prompt": "standing at a low cable pulley, holding a rope with both hands",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Close-Grip EZ-Bar Curl with Band",
"view": "side",
"prompt": "standing on a band attached to an EZ bar, holding the bar",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Close-Grip Standing Barbell Curl",
"view": "side",
"prompt": "standing upright, holding a barbell with a narrow underhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Close-grip Press-ups",
"view": "side",
"prompt": "push-up position on the floor, hands close together under the chest",
"start": {
"torso": 23,
"head": 31,
"arms": [
[
-100,
-77
],
[
-100,
-77
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 11,
"head": 19,
"arms": [
[
-162,
-42
],
[
-162,
-42
]
],
"legs": [
[
191,
191
],
[
191,
191
]
],
"face": "down"
}
},
{
"name": "Curl De Muñeca Con Barra",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, holding a barbell palms up",
"start": {
"torso": 50,
"head": 35,
"arms": [
[
-114,
-12
],
[
-112,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 50,
"head": 35,
"arms": [
[
-114,
18
],
[
-112,
20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Dumbbell Bicep Curl to Press",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Curl",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Prone Incline Curl",
"view": "side",
"prompt": "lying chest down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 45,
"head": 50,
"arms": [
[
-90,
-90
],
[
-88,
-88
]
],
"legs": [
[
-125,
-95
],
[
-120,
-95
]
],
"face": "down"
},
"end": {
"torso": 45,
"head": 50,
"arms": [
[
-90,
75
],
[
-88,
78
]
],
"legs": [
[
-125,
-95
],
[
-120,
-95
]
],
"face": "down"
}
},
{
"name": "Dumbbell Wide Bicep Curls",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand, hands wide",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "EZ-Bar Curl",
"view": "side",
"prompt": "standing upright, holding an EZ curl bar with an underhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Forearm Curls (underhand grip)",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, a dumbbell in each hand palms up",
"start": {
"torso": 50,
"head": 35,
"arms": [
[
-114,
-12
],
[
-112,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 50,
"head": 35,
"arms": [
[
-114,
18
],
[
-112,
20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Hand Grip",
"view": "side",
"prompt": "standing upright, holding a spring hand grip trainer in one hand",
"start": {
"torso": 90,
"arms": [
[
-75,
-15
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-75,
10
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Incline Chest-Supported Dumbbell Row",
"view": "side",
"prompt": "lying chest down on a 45-degree incline bench, a dumbbell in each hand",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-88,
-88
]
],
"legs": [
[
-125,
-95
],
[
-120,
-95
]
],
"face": "down"
},
"end": {
"torso": 45,
"head": 55,
"arms": [
[
-165,
-95
],
[
-162,
-95
]
],
"legs": [
[
-125,
-95
],
[
-120,
-95
]
],
"face": "down"
}
},
{
"name": "Incline Dumbbell Curl",
"view": "side",
"prompt": "seated leaning back on an incline bench, a dumbbell in each hand",
"start": {
"torso": 125,
"head": 110,
"arms": [
[
-95,
-90
],
[
-93,
-88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 125,
"head": 110,
"arms": [
[
-95,
80
],
[
-93,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Incline Hammer Curls",
"view": "side",
"prompt": "seated leaning back on an incline bench, dumbbells in a neutral grip",
"start": {
"torso": 125,
"head": 110,
"arms": [
[
-95,
-90
],
[
-93,
-88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 125,
"head": 110,
"arms": [
[
-95,
80
],
[
-93,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Lying Cable Curl",
"view": "side",
"prompt": "lying face up on the floor, feet toward a low pulley, holding a bar",
"start": {
"torso": 180,
"head": 175,
"arms": [
[
0,
0
],
[
2,
2
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 175,
"arms": [
[
0,
140
],
[
2,
142
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Lying Close-Grip Bar Curl On High Pulley",
"view": "side",
"prompt": "lying face up on a flat bench, holding a high cable bar above",
"start": {
"torso": 180,
"head": 185,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"head": 185,
"arms": [
[
90,
210
],
[
92,
212
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Lying High Bench Barbell Curl",
"view": "side",
"prompt": "lying face down on a high flat bench, holding a barbell below",
"start": {
"torso": 0,
"head": 10,
"arms": [
[
-90,
-90
],
[
-88,
-88
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"face": "down"
},
"end": {
"torso": 0,
"head": 10,
"arms": [
[
-90,
70
],
[
-88,
72
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"face": "down"
}
},
{
"name": "Lying Supine Dumbbell Curl",
"view": "side",
"prompt": "lying face up on a flat bench, a dumbbell in each hand",
"start": {
"torso": 180,
"arms": [
[
-90,
-90
],
[
-88,
-88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-90,
95
],
[
-88,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Mentzer Pulldown",
"view": "side",
"prompt": "seated at a lat pulldown machine, close underhand grip on the bar",
"start": {
"torso": 100,
"head": 100,
"arms": [
[
82,
86
],
[
84,
88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"head": 100,
"arms": [
[
-105,
80
],
[
-103,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Neutral-grip Chest Pulldown",
"view": "side",
"prompt": "seated at a lat pulldown machine, holding a neutral-grip handle",
"start": {
"torso": 100,
"head": 100,
"arms": [
[
82,
86
],
[
84,
88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"head": 100,
"arms": [
[
-105,
80
],
[
-103,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "One-handed Kettlebell Curls",
"view": "side",
"prompt": "standing upright, a kettlebell in one hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-92,
-92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
-92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Barbell Curl",
"view": "side",
"prompt": "standing upright, holding a barbell with an overhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Barbell Preacher Curls",
"view": "side",
"prompt": "seated at a preacher bench, upper arms on the pad, holding an EZ bar",
"start": {
"torso": 80,
"head": 75,
"arms": [
[
-45,
-30
],
[
-43,
-28
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 80,
"head": 75,
"arms": [
[
-45,
80
],
[
-43,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Reverse Cable Curl",
"view": "side",
"prompt": "standing at a low cable pulley, holding a bar with an overhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Grip Barbell Curls",
"view": "side",
"prompt": "standing upright, holding a barbell with an overhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Plate Curls",
"view": "side",
"prompt": "standing upright, holding a weight plate with both hands",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Seated Biceps",
"view": "side",
"prompt": "sitting on the floor, knees bent, arms straight behind him",
"start": {
"torso": 92,
"arms": [
[
-125,
-125
],
[
-123,
-123
]
],
"legs": [
[
40,
-45
],
[
40,
-45
]
]
},
"end": {
"torso": 92,
"arms": [
[
-160,
-160
],
[
-158,
-158
]
],
"legs": [
[
40,
-45
],
[
40,
-45
]
]
}
},
{
"name": "Seated Dumbbell Curl",
"view": "side",
"prompt": "seated on a flat bench, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Seated W Curl",
"view": "side",
"prompt": "seated leaning back on an incline bench, a dumbbell in each hand",
"start": {
"torso": 120,
"head": 108,
"arms": [
[
-95,
-90
],
[
-93,
-88
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 120,
"head": 108,
"arms": [
[
-95,
80
],
[
-93,
82
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Single-arm Preacher Curl",
"view": "side",
"prompt": "seated at a preacher bench, one upper arm on the pad, holding a dumbbell",
"start": {
"torso": 80,
"head": 75,
"arms": [
[
-45,
-30
],
[
-43,
-40
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 80,
"head": 75,
"arms": [
[
-45,
80
],
[
-43,
-40
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Sloper Hanging",
"view": "side",
"prompt": "hanging from the sloper holds of a fingerboard, arms overhead",
"start": {
"torso": 90,
"arms": [
[
88,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
90,
90
]
],
"legs": [
[
-70,
-115
],
[
-72,
-117
]
]
}
},
{
"name": "Spider Curl",
"view": "side",
"prompt": "lying chest down on a slanted preacher bench, holding an EZ bar",
"start": {
"torso": 45,
"head": 50,
"arms": [
[
-90,
-90
],
[
-88,
-88
]
],
"legs": [
[
-120,
-95
],
[
-115,
-95
]
],
"face": "down"
},
"end": {
"torso": 45,
"head": 50,
"arms": [
[
-90,
75
],
[
-88,
78
]
],
"legs": [
[
-120,
-95
],
[
-115,
-95
]
],
"face": "down"
}
},
{
"name": "Standing Biceps Stretch",
"view": "side",
"prompt": "standing upright, hands clasped behind his back",
"start": {
"torso": 90,
"arms": [
[
-100,
-100
],
[
-98,
-98
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-140,
-140
],
[
-138,
-138
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Concentration Curl",
"view": "side",
"prompt": "standing bent forward, a dumbbell in one hand, other hand on a bench",
"start": {
"torso": 25,
"head": 10,
"arms": [
[
-90,
-88
],
[
-80,
-90
]
],
"legs": [
[
-75,
-95
],
[
-80,
-95
]
]
},
"end": {
"torso": 25,
"head": 10,
"arms": [
[
-90,
100
],
[
-80,
-90
]
],
"legs": [
[
-75,
-95
],
[
-80,
-95
]
]
}
},
{
"name": "Standing Dumbbell Reverse Curl",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand, palms down",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing One-Arm Cable Curl",
"view": "side",
"prompt": "standing beside a low cable pulley, one handle in one hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-92,
-92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
-92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Straight Bar Cable Curls",
"view": "side",
"prompt": "standing at a cable tower, holding a straight bar",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wide-Grip Standing Barbell Curl",
"view": "side",
"prompt": "standing upright, holding a barbell with a wide underhand grip",
"start": {
"torso": 90,
"arms": [
[
-90,
-84
],
[
-88,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
75
],
[
-92,
78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wrist Rocks",
"view": "side",
"prompt": "on hands and knees on the floor, palms flat",
"start": {
"torso": 13,
"head": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"face": "down"
},
"end": {
"torso": 15,
"head": 20,
"arms": [
[
-107,
-107
],
[
-107,
-107
]
],
"legs": [
[
-115,
180
],
[
-115,
180
]
],
"face": "down"
}
},
{
"name": "Ankle Dorsiflexion Rocks",
"view": "side",
"prompt": "half-kneeling on the floor, front foot flat, hands on front knee",
"start": {
"torso": 90,
"arms": [
[
-50,
-10
],
[
-55,
-10
]
],
"legs": [
[
0,
-90
],
[
-90,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
-40,
-5
],
[
-45,
-5
]
],
"legs": [
[
-10,
-65
],
[
-100,
180
]
]
}
},
{
"name": "Anterior Tibialis-SMR",
"view": "side",
"prompt": "sitting on the floor, knees bent, rolling a muscle roller on the shin",
"start": {
"torso": 70,
"head": 50,
"arms": [
[
-96,
28
],
[
-96,
28
]
],
"legs": [
[
40,
-45
],
[
40,
-45
]
]
},
"end": {
"torso": 55,
"head": 40,
"arms": [
[
-93,
-10
],
[
-93,
-10
]
],
"legs": [
[
40,
-45
],
[
40,
-45
]
]
}
},
{
"name": "Balance Board",
"view": "front",
"prompt": "standing on a balance board on the floor",
"start": {
"torso": 90,
"arms": [
[
-100,
-95
],
[
-80,
-85
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 87,
"arms": [
[
175,
170
],
[
5,
10
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
}
},
{
"name": "Calf Press",
"view": "side",
"prompt": "seated in a leg press machine, balls of the feet on the platform",
"start": {
"torso": 130,
"head": 115,
"arms": [
[
-60,
-20
],
[
-60,
-20
]
],
"legs": [
[
30,
22
],
[
30,
22
]
]
},
"end": {
"torso": 130,
"head": 115,
"arms": [
[
-60,
-20
],
[
-60,
-20
]
],
"legs": [
[
27,
27
],
[
27,
27
]
]
}
},
{
"name": "Calf Raise On A Dumbbell",
"view": "side",
"prompt": "standing on a dumbbell handle, one hand holding a support",
"start": {
"torso": 90,
"arms": [
[
-30,
-10
],
[
-90,
-90
]
],
"legs": [
[
-91,
-92
],
[
-91,
-92
]
]
},
"end": {
"torso": 91,
"arms": [
[
-25,
-5
],
[
-90,
-90
]
],
"legs": [
[
-87,
-83
],
[
-87,
-83
]
]
}
},
{
"name": "Calf Stretch Elbows Against Wall",
"view": "side",
"prompt": "standing facing a wall in a staggered stance, forearms on the wall",
"start": {
"torso": 90,
"arms": [
[
-30,
70
],
[
-30,
70
]
],
"legs": [
[
-80,
-95
],
[
-100,
-95
]
]
},
"end": {
"torso": 55,
"head": 65,
"arms": [
[
10,
90
],
[
10,
90
]
],
"legs": [
[
-60,
-100
],
[
-115,
-115
]
]
}
},
{
"name": "Calf Stretch Hands Against Wall",
"view": "side",
"prompt": "standing facing a wall in a staggered stance, hands on the wall",
"start": {
"torso": 90,
"arms": [
[
-20,
0
],
[
-20,
0
]
],
"legs": [
[
-80,
-95
],
[
-100,
-95
]
]
},
"end": {
"torso": 60,
"head": 65,
"arms": [
[
15,
15
],
[
15,
15
]
],
"legs": [
[
-60,
-100
],
[
-115,
-115
]
]
}
},
{
"name": "Calves Foam Roller",
"view": "side",
"prompt": "sitting on the floor, hands behind him, calves on a foam roller",
"start": {
"torso": 110,
"head": 100,
"arms": [
[
-116,
-116
],
[
-116,
-116
]
],
"legs": [
[
4,
4
],
[
4,
4
]
]
},
"end": {
"torso": 125,
"head": 105,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-3,
-3
],
[
-3,
-3
]
]
}
},
{
"name": "Calves-SMR",
"view": "side",
"prompt": "sitting on the floor, hands behind him, one calf on a foam roller",
"start": {
"torso": 110,
"head": 100,
"arms": [
[
-116,
-116
],
[
-116,
-116
]
],
"legs": [
[
4,
4
],
[
10,
0
]
]
},
"end": {
"torso": 125,
"head": 105,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-3,
-3
],
[
3,
-8
]
]
}
},
{
"name": "Double Leg Calf Raise",
"view": "side",
"prompt": "standing upright on the floor, feet together",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-91,
-92
],
[
-91,
-92
]
]
},
"end": {
"torso": 91,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-87,
-83
],
[
-87,
-83
]
]
}
},
{
"name": "Foot-SMR",
"view": "side",
"prompt": "seated on a bench, barefoot, one foot on a small foot roller",
"start": {
"torso": 90,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
0,
-100
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
0,
-72
],
[
0,
-90
]
]
}
},
{
"name": "Jump Rope",
"view": "side",
"prompt": "standing, holding jump rope handles at the hips",
"start": {
"torso": 90,
"arms": [
[
-80,
-30
],
[
-82,
-30
]
],
"legs": [
[
-85,
-95
],
[
-87,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
-85,
-10
],
[
-87,
-10
]
],
"legs": [
[
-90,
-88
],
[
-92,
-88
]
]
}
},
{
"name": "Lateral Push Off",
"view": "front",
"prompt": "standing on the floor, bounding sideways from one leg to the other",
"start": {
"torso": 100,
"arms": [
[
-120,
-100
],
[
-60,
-40
]
],
"legs": [
[
-100,
-82
],
[
-45,
-80
]
]
},
"end": {
"torso": 80,
"arms": [
[
-140,
-120
],
[
-60,
-80
]
],
"legs": [
[
-135,
-100
],
[
-80,
-98
]
]
}
},
{
"name": "Peroneals Stretch",
"view": "side",
"prompt": "sitting on the floor, legs straight, a belt looped around one foot",
"start": {
"torso": 90,
"arms": [
[
-35,
-25
],
[
-35,
-25
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 97,
"arms": [
[
-110,
-5
],
[
-110,
-5
]
],
"legs": [
[
12,
12
],
[
0,
0
]
]
}
},
{
"name": "Peroneals-SMR",
"view": "side",
"prompt": "lying on his side on one forearm, lower leg on a foam roller",
"start": {
"torso": 147,
"head": 155,
"arms": [
[
-90,
180
],
[
0,
-20
]
],
"legs": [
[
9,
9
],
[
12,
8
]
]
},
"end": {
"torso": 160,
"head": 165,
"arms": [
[
-90,
180
],
[
0,
-20
]
],
"legs": [
[
2,
2
],
[
5,
0
]
]
}
},
{
"name": "Posterior Tibialis Stretch",
"view": "side",
"prompt": "sitting on the floor, legs straight, a belt looped around one foot",
"start": {
"torso": 90,
"arms": [
[
-35,
-25
],
[
-35,
-25
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 97,
"arms": [
[
-110,
-5
],
[
-110,
-5
]
],
"legs": [
[
12,
12
],
[
0,
0
]
]
}
},
{
"name": "Seated Calf Stretch",
"view": "side",
"prompt": "sitting on a mat, one leg straight, a towel around that foot",
"start": {
"torso": 90,
"arms": [
[
-30,
-25
],
[
-30,
-25
]
],
"legs": [
[
0,
0
],
[
45,
-50
]
]
},
"end": {
"torso": 85,
"arms": [
[
-100,
-10
],
[
-100,
-10
]
],
"legs": [
[
0,
0
],
[
45,
-50
]
]
}
},
{
"name": "Single-Leg Calf Raise",
"view": "side",
"prompt": "standing on one foot, other foot raised behind, hand on a wall",
"start": {
"torso": 90,
"arms": [
[
-20,
0
],
[
-90,
-90
]
],
"legs": [
[
-91,
-92
],
[
-100,
-160
]
]
},
"end": {
"torso": 91,
"arms": [
[
-15,
5
],
[
-90,
-90
]
],
"legs": [
[
-87,
-83
],
[
-100,
-160
]
]
}
},
{
"name": "Single-Leg Lunge with Kettlebell:",
"view": "side",
"prompt": "standing, holding a kettlebell at the chest with both hands",
"start": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-78,
102
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-78,
102
]
],
"legs": [
[
-15,
-90
],
[
-105,
-165
]
]
}
},
{
"name": "Single-leg Side Glute Press",
"view": "side",
"prompt": "lying back in a leg press machine, one foot high on the platform",
"start": {
"torso": 135,
"head": 120,
"arms": [
[
-60,
-20
],
[
-60,
-20
]
],
"legs": [
[
75,
-5
],
[
-20,
-90
]
]
},
"end": {
"torso": 135,
"head": 120,
"arms": [
[
-60,
-20
],
[
-60,
-20
]
],
"legs": [
[
37,
32
],
[
-20,
-90
]
]
}
},
{
"name": "Sitting Calf Raises",
"view": "side",
"prompt": "seated on a bench, feet on a raised step, hands on the knees",
"start": {
"torso": 68,
"head": 60,
"arms": [
[
-53,
-77
],
[
-53,
-77
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 68,
"head": 60,
"arms": [
[
-33,
-93
],
[
-33,
-93
]
],
"legs": [
[
9,
-94
],
[
9,
-94
]
]
}
},
{
"name": "Sitting Calf Stretch (Dorsiflexion)",
"view": "side",
"prompt": "sitting on the floor with legs straight, a towel around the feet",
"start": {
"torso": 90,
"arms": [
[
-25,
-20
],
[
-25,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 45,
"head": 30,
"arms": [
[
-35,
-35
],
[
-35,
-35
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Smith Machine Calf Raise",
"view": "side",
"prompt": "standing under a Smith machine bar resting on the shoulders",
"start": {
"torso": 90,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-91,
-92
],
[
-91,
-92
]
]
},
"end": {
"torso": 91,
"arms": [
[
-110,
80
],
[
-108,
82
]
],
"legs": [
[
-87,
-83
],
[
-87,
-83
]
]
}
},
{
"name": "Standing Calf Raise",
"view": "side",
"prompt": "standing upright on the floor, hands at his sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-91,
-92
],
[
-91,
-92
]
]
},
"end": {
"torso": 91,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-87,
-83
],
[
-87,
-83
]
]
}
},
{
"name": "Standing Calf Stretch",
"view": "side",
"prompt": "standing on the floor, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-110,
-10
],
[
-110,
-10
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"head": 80,
"arms": [
[
-110,
-10
],
[
-110,
-10
]
],
"legs": [
[
-60,
-100
],
[
-115,
-115
]
]
}
},
{
"name": "Standing Dumbbell Calf Raise",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand at his sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-91,
-92
],
[
-91,
-92
]
]
},
"end": {
"torso": 91,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-87,
-83
],
[
-87,
-83
]
]
}
},
{
"name": "Standing Gastrocnemius Calf Stretch",
"view": "side",
"prompt": "standing with one heel on a low step, leg straight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-50,
-50
],
[
-90,
-90
]
]
},
"end": {
"torso": 15,
"head": 0,
"arms": [
[
-80,
-80
],
[
-82,
-82
]
],
"legs": [
[
-50,
-50
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Soleus And Achilles Stretch",
"view": "side",
"prompt": "standing in a short staggered stance, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-110,
-10
],
[
-110,
-10
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
},
"end": {
"torso": 88,
"arms": [
[
-110,
-10
],
[
-110,
-10
]
],
"legs": [
[
-60,
-110
],
[
-95,
-120
]
]
}
},
{
"name": "Supported Calf Stretch",
"view": "side",
"prompt": "standing in a staggered stance, hands resting on a wall",
"start": {
"torso": 90,
"arms": [
[
-20,
0
],
[
-20,
0
]
],
"legs": [
[
-80,
-95
],
[
-100,
-95
]
]
},
"end": {
"torso": 60,
"head": 65,
"arms": [
[
15,
15
],
[
15,
15
]
],
"legs": [
[
-60,
-100
],
[
-115,
-115
]
]
}
},
{
"name": "Talons Fesses",
"view": "side",
"prompt": "standing on the floor, jogging in place",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 88,
"arms": [
[
-70,
60
],
[
-120,
-60
]
],
"legs": [
[
-95,
85
],
[
-90,
-90
]
]
}
},
{
"name": "Alternating Floor Press",
"view": "side",
"prompt": "lying face up on the floor, knees bent, a kettlebell in each hand",
"start": {
"torso": 180,
"arms": [
[
0,
90
],
[
90,
90
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
0,
90
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
}
},
{
"name": "Around The Worlds",
"view": "side",
"prompt": "lying face up on a flat bench, a dumbbell in each hand",
"start": {
"torso": 180,
"arms": [
[
-5,
-5
],
[
-3,
-3
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
185,
185
],
[
183,
183
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Band Chest Press",
"view": "side",
"prompt": "standing, resistance band behind his back, handles at the chest",
"start": {
"torso": 90,
"arms": [
[
-165,
5
],
[
-163,
7
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
2,
2
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
}
},
{
"name": "Barbell Bench Press - Medium Grip",
"view": "side",
"prompt": "lying face up on a flat bench, holding a barbell above the chest",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-75,
95
],
[
-73,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Barbell Guillotine Bench Press",
"view": "side",
"prompt": "lying face up on a flat bench, holding a barbell above the neck",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-95,
92
],
[
-93,
94
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Barbell Incline Bench Press - Medium Grip",
"view": "side",
"prompt": "lying back on an incline bench, holding a barbell above the chest",
"start": {
"torso": 135,
"head": 125,
"arms": [
[
85,
90
],
[
87,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 135,
"head": 125,
"arms": [
[
-100,
75
],
[
-98,
77
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Behind Head Chest Stretch",
"view": "side",
"prompt": "sitting on the floor, legs straight, hands behind his head",
"start": {
"torso": 90,
"arms": [
[
60,
170
],
[
62,
172
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
30
],
[
152,
32
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Bench Press - With Bands",
"view": "side",
"prompt": "lying face up on a flat bench, holding a banded barbell",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-75,
95
],
[
-73,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Bodyweight Flyes",
"view": "side",
"prompt": "push-up position, hands on two EZ bars lying on the floor",
"start": {
"torso": 23,
"head": 31,
"arms": [
[
-119,
-56
],
[
-119,
-56
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 13,
"head": 21,
"arms": [
[
-40,
-140
],
[
-40,
-140
]
],
"legs": [
[
193,
193
],
[
193,
193
]
],
"face": "down"
}
},
{
"name": "Cable Chest Press",
"view": "side",
"prompt": "seated in a cable chest press machine, holding the handles",
"start": {
"torso": 95,
"arms": [
[
-170,
5
],
[
-168,
7
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
0,
0
],
[
2,
2
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Cable Iron Cross",
"view": "front",
"prompt": "standing between two high cable pulleys, a handle in each hand",
"start": {
"torso": 90,
"arms": [
[
150,
150
],
[
30,
30
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
-72
],
[
-100,
-108
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
}
},
{
"name": "Chain Press",
"view": "side",
"prompt": "lying face up on a flat bench, holding handles with hanging chains",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-75,
95
],
[
-73,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Chest And Front Of Shoulder Stretch",
"view": "side",
"prompt": "standing upright, holding a broomstick with a wide grip",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-78,
-78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
135,
135
],
[
137,
137
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Chest Press",
"view": "side",
"prompt": "seated in a chest press machine, holding the handles",
"start": {
"torso": 95,
"arms": [
[
-170,
5
],
[
-168,
7
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
0,
0
],
[
2,
2
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Chest Push (multiple response)",
"view": "side",
"prompt": "kneeling on the floor, holding a medicine ball at the chest",
"start": {
"torso": 90,
"arms": [
[
-120,
30
],
[
-118,
32
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 60,
"head": 65,
"arms": [
[
10,
10
],
[
12,
12
]
],
"legs": [
[
-110,
180
],
[
-110,
180
]
]
}
},
{
"name": "Chest Push (single response)",
"view": "side",
"prompt": "kneeling on the floor, holding a medicine ball at the chest",
"start": {
"torso": 90,
"arms": [
[
-120,
30
],
[
-118,
32
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 60,
"head": 65,
"arms": [
[
10,
10
],
[
12,
12
]
],
"legs": [
[
-110,
180
],
[
-110,
180
]
]
}
},
{
"name": "Chest Push from 3 point stance",
"view": "side",
"prompt": "crouched in a three-point stance, a medicine ball on the floor ahead",
"start": {
"torso": -20,
"head": 0,
"arms": [
[
-85,
-85
],
[
-80,
-80
]
],
"legs": [
[
-60,
-120
],
[
-70,
-115
]
],
"face": "down"
},
"end": {
"torso": 70,
"head": 80,
"arms": [
[
5,
5
],
[
7,
7
]
],
"legs": [
[
-50,
-100
],
[
-120,
-120
]
]
}
},
{
"name": "Chest Push with Run Release",
"view": "side",
"prompt": "athletic stance, knees bent, holding a medicine ball near the legs",
"start": {
"torso": 45,
"head": 60,
"arms": [
[
-80,
-80
],
[
-78,
-78
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
]
},
"end": {
"torso": 65,
"head": 75,
"arms": [
[
5,
5
],
[
7,
7
]
],
"legs": [
[
-50,
-100
],
[
-120,
-120
]
]
}
},
{
"name": "Chest Stretch on Stability Ball",
"view": "side",
"prompt": "on hands and knees beside an exercise ball, one elbow on the ball",
"start": {
"torso": 15,
"head": 20,
"arms": [
[
10,
20
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"face": "down"
},
"end": {
"torso": -5,
"head": 0,
"arms": [
[
40,
40
],
[
-40,
-140
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"face": "down"
}
},
{
"name": "Clock Push-Up",
"view": "side",
"prompt": "push-up position on the floor, hands under the shoulders",
"start": {
"torso": 23,
"head": 31,
"arms": [
[
-100,
-77
],
[
-100,
-77
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 11,
"head": 19,
"arms": [
[
-162,
-42
],
[
-162,
-42
]
],
"legs": [
[
191,
191
],
[
191,
191
]
],
"face": "down"
}
},
{
"name": "Decline Push-Up",
"view": "side",
"prompt": "push-up position, feet up on a bench, hands on the floor",
"start": {
"torso": -5,
"head": 3,
"arms": [
[
-100,
-82
],
[
-100,
-82
]
],
"legs": [
[
175,
175
],
[
175,
175
]
],
"face": "down"
},
"end": {
"torso": -12,
"head": -4,
"arms": [
[
-132,
-38
],
[
-132,
-38
]
],
"legs": [
[
168,
168
],
[
168,
168
]
],
"face": "down"
}
},
{
"name": "Decline Smith Press",
"view": "side",
"prompt": "lying face up on a decline bench under a Smith machine bar",
"start": {
"torso": 200,
"head": 195,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
},
"end": {
"torso": 200,
"head": 195,
"arms": [
[
-70,
100
],
[
-68,
102
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
}
},
{
"name": "Dips - Chest Version",
"view": "side",
"prompt": "supported on straight arms between parallel dip bars, leaning forward",
"start": {
"torso": 75,
"head": 80,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
},
"end": {
"torso": 60,
"head": 70,
"arms": [
[
175,
-90
],
[
175,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
}
},
{
"name": "Drop Push",
"view": "side",
"prompt": "push-up position, hands on two low boxes on the floor",
"start": {
"torso": 30,
"head": 38,
"arms": [
[
-102,
-63
],
[
-102,
-63
]
],
"legs": [
[
210,
210
],
[
210,
210
]
],
"face": "down"
},
"end": {
"torso": 12,
"head": 20,
"arms": [
[
-155,
-39
],
[
-155,
-39
]
],
"legs": [
[
192,
192
],
[
192,
192
]
],
"face": "down"
}
},
{
"name": "Dumbbell Bench Press with Neutral Grip",
"view": "side",
"prompt": "lying face up on a flat bench, dumbbells held with palms facing in",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-75,
95
],
[
-73,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Dumbbell Hex Press",
"view": "side",
"prompt": "lying face up on a flat bench, two dumbbells pressed together",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
92,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
-75,
95
],
[
-73,
97
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Dumbbell Pullover",
"view": "side",
"prompt": "lying face up on a flat bench, holding one dumbbell with both hands",
"start": {
"torso": 180,
"arms": [
[
95,
100
],
[
97,
102
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"arms": [
[
185,
190
],
[
187,
192
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Dynamic Chest Stretch",
"view": "side",
"prompt": "standing upright, arms straight out in front",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
2,
2
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
170,
170
],
[
172,
172
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Elbows Back",
"view": "side",
"prompt": "standing upright, both hands on the lower back",
"start": {
"torso": 90,
"arms": [
[
-140,
-50
],
[
-138,
-48
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 95,
"head": 100,
"arms": [
[
-150,
-55
],
[
-148,
-53
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Explosive Push-Ups",
"view": "side",
"prompt": "push-up position on the floor, hands under the shoulders",
"start": {
"torso": 11,
"head": 19,
"arms": [
[
-162,
-42
],
[
-162,
-42
]
],
"legs": [
[
191,
191
],
[
191,
191
]
],
"face": "down"
},
"end": {
"torso": 23,
"head": 31,
"arms": [
[
-100,
-77
],
[
-100,
-77
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
}
},
{
"name": "Extended Range One-Arm Kettlebell Floor Press",
"view": "side",
"prompt": "lying face up on the floor, knees bent, a kettlebell in one hand",
"start": {
"torso": 180,
"arms": [
[
0,
90
],
[
0,
0
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
0,
0
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
}
},
{
"name": "Finger Pushup",
"view": "side",
"prompt": "push-up position on the fingertips, hands under the shoulders",
"start": {
"torso": 23,
"head": 31,
"arms": [
[
-100,
-77
],
[
-100,
-77
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 11,
"head": 19,
"arms": [
[
-162,
-42
],
[
-162,
-42
]
],
"legs": [
[
191,
191
],
[
191,
191
]
],
"face": "down"
}
},
{
"name": "Forward Drag with Press",
"view": "side",
"prompt": "leaning forward mid-stride, holding sled handles at the chest",
"start": {
"torso": 60,
"head": 70,
"arms": [
[
-130,
20
],
[
-128,
22
]
],
"legs": [
[
-50,
-100
],
[
-120,
-120
]
]
},
"end": {
"torso": 60,
"head": 70,
"arms": [
[
5,
5
],
[
7,
7
]
],
"legs": [
[
-50,
-100
],
[
-120,
-120
]
]
}
},
{
"name": "Hammerstrength Decline Chest Press",
"view": "side",
"prompt": "seated upright in a decline chest press machine, holding the handles",
"start": {
"torso": 95,
"arms": [
[
-170,
5
],
[
-168,
7
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
-20,
-20
],
[
-18,
-18
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Heavy Bag Thrust",
"view": "side",
"prompt": "standing beside a heavy bag, one hand on the bag at chest height",
"start": {
"torso": 85,
"arms": [
[
-150,
20
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-105,
-90
]
]
},
"end": {
"torso": 78,
"head": 85,
"arms": [
[
0,
0
],
[
-100,
-90
]
],
"legs": [
[
-75,
-95
],
[
-105,
-90
]
]
}
},
{
"name": "Incline Cable Chest Press",
"view": "side",
"prompt": "seated in an incline cable press, holding the handles",
"start": {
"torso": 100,
"arms": [
[
-170,
10
],
[
-168,
12
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"arms": [
[
30,
30
],
[
32,
32
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Incline Dumbbell Bench With Palms Facing In",
"view": "side",
"prompt": "lying back on an incline bench, dumbbells held with palms facing in",
"start": {
"torso": 135,
"head": 125,
"arms": [
[
85,
90
],
[
87,
92
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 135,
"head": 125,
"arms": [
[
-100,
75
],
[
-98,
77
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Incline Push-Up Depth Jump",
"view": "side",
"prompt": "feet on a low box, hands on the floor between two steps",
"start": {
"torso": -2,
"head": 6,
"arms": [
[
-146,
-32
],
[
-146,
-32
]
],
"legs": [
[
178,
178
],
[
178,
178
]
],
"face": "down"
},
"end": {
"torso": 12,
"head": 20,
"arms": [
[
-115,
-50
],
[
-115,
-50
]
],
"legs": [
[
192,
192
],
[
192,
192
]
],
"face": "down"
}
},
{
"name": "Incline Push-Up Medium",
"view": "side",
"prompt": "body straight, hands on a Smith machine bar, feet on the floor",
"start": {
"torso": 44,
"head": 52,
"arms": [
[
-110,
-63
],
[
-110,
-63
]
],
"legs": [
[
224,
224
],
[
224,
224
]
],
"face": "down"
},
"end": {
"torso": 30,
"head": 38,
"arms": [
[
-181,
-66
],
[
-181,
-66
]
],
"legs": [
[
210,
210
],
[
210,
210
]
],
"face": "down"
}
},
{
"name": "Incline Push-Up Reverse Grip",
"view": "side",
"prompt": "body straight, hands underhand on a Smith machine bar, feet on the floor",
"start": {
"torso": 44,
"head": 52,
"arms": [
[
-110,
-63
],
[
-110,
-63
]
],
"legs": [
[
224,
224
],
[
224,
224
]
],
"face": "down"
},
"end": {
"torso": 30,
"head": 38,
"arms": [
[
-181,
-66
],
[
-181,
-66
]
],
"legs": [
[
210,
210
],
[
210,
210
]
],
"face": "down"
}
},
{
"name": "Incline Push-Up Wide",
"view": "side",
"prompt": "body straight, hands wide on a Smith machine bar, feet on the floor",
"start": {
"torso": 44,
"head": 52,
"arms": [
[
-110,
-63
],
[
-110,
-63
]
],
"legs": [
[
224,
224
],
[
224,
224
]
],
"face": "down"
},
"end": {
"torso": 30,
"head": 38,
"arms": [
[
-181,
-66
],
[
-181,
-66
]
],
"legs": [
[
210,
210
],
[
210,
210
]
],
"face": "down"
}
},
{
"name": "Isometric Chest Squeezes",
"view": "side",
"prompt": "standing upright, palms pressed together in front of the chest",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-30,
60
],
[
-28,
62
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Isometric Wipers",
"view": "side",
"prompt": "push-up position with hands wide, on hands and toes on the floor",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 15,
"arms": [
[
-155.6,
-65.4
],
[
-155.6,
-65.4
]
],
"legs": [
[
195,
195
],
[
195,
195
]
]
}
},
{
"name": "Leg-Over Floor Press",
"view": "side",
"prompt": "lying face up on the floor, one leg crossed over, kettlebell in one hand",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
90
],
[
0,
0
]
],
"legs": [
[
8,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
90,
90
],
[
0,
0
]
],
"legs": [
[
8,
0
],
[
0,
0
]
]
}
},
{
"name": "Leverage Chest Press",
"view": "side",
"prompt": "seated in a chest press machine, gripping the handles",
"start": {
"torso": 100,
"arms": [
[
-165,
5
],
[
-165,
5
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Leverage Decline Chest Press",
"view": "side",
"prompt": "seated in a decline chest press machine, gripping the handles",
"start": {
"torso": 100,
"arms": [
[
-170,
-15
],
[
-170,
-15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"arms": [
[
-25,
-25
],
[
-25,
-25
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Leverage Incline Chest Press",
"view": "side",
"prompt": "seated in an incline chest press machine, gripping the handles",
"start": {
"torso": 100,
"arms": [
[
-160,
25
],
[
-160,
25
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 100,
"arms": [
[
35,
35
],
[
35,
35
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Low Cable Crossover",
"view": "side",
"prompt": "standing between two low cable pulleys, a handle in each hand",
"start": {
"torso": 85,
"arms": [
[
-110,
-105
],
[
-110,
-105
]
],
"legs": [
[
-80,
-90
],
[
-100,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
15,
25
],
[
15,
25
]
],
"legs": [
[
-80,
-90
],
[
-100,
-95
]
]
}
},
{
"name": "Low-Cable Cross-Over - NB",
"view": "side",
"prompt": "standing between two low cable pulleys, a handle in each hand",
"start": {
"torso": 85,
"arms": [
[
-110,
-105
],
[
-110,
-105
]
],
"legs": [
[
-80,
-90
],
[
-100,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
15,
25
],
[
15,
25
]
],
"legs": [
[
-80,
-90
],
[
-100,
-95
]
]
}
},
{
"name": "Machine Chest Fly",
"view": "side",
"prompt": "seated upright in a pec deck machine, forearms against the pads",
"start": {
"torso": 90,
"arms": [
[
180,
90
],
[
180,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
90
],
[
0,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Medicine Ball Chest Pass",
"view": "side",
"prompt": "standing in a split stance, holding a medicine ball at the chest",
"start": {
"torso": 90,
"arms": [
[
-120,
30
],
[
-120,
30
]
],
"legs": [
[
-75,
-95
],
[
-105,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
5,
5
],
[
5,
5
]
],
"legs": [
[
-75,
-95
],
[
-105,
-95
]
]
}
},
{
"name": "One-Arm Flat Bench Dumbbell Flye",
"view": "side",
"prompt": "lying face up on a flat bench, a dumbbell in one hand",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
90,
90
],
[
-60,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
-25,
-45
],
[
-60,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Floor Press",
"view": "side",
"prompt": "lying face up on the floor, knees bent, a kettlebell in one hand",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
90
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
90,
90
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Plyo Kettlebell Pushups",
"view": "side",
"prompt": "push-up position, one hand on a kettlebell, the other on the floor",
"start": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
},
"end": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
210,
210
],
[
210,
210
]
]
}
},
{
"name": "Plyo Push-up",
"view": "side",
"prompt": "push-up position on the floor, exploding up so the hands leave the floor",
"start": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
},
"end": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
210,
210
],
[
210,
210
]
]
}
},
{
"name": "Push Up to Side Plank",
"view": "side",
"prompt": "on hands and toes on the floor, turning into a side plank",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 23,
"arms": [
[
90,
90
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
}
},
{
"name": "Push-Up Wide",
"view": "side",
"prompt": "push-up position, hands wide apart on the floor",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Push-Ups With Feet Elevated",
"view": "side",
"prompt": "push-up position, hands on the floor, feet up on a bench",
"start": {
"torso": 5,
"arms": [
[
-94.8,
-85.0
],
[
-94.8,
-85.0
]
],
"legs": [
[
185,
185
],
[
185,
185
]
]
},
"end": {
"torso": -6,
"arms": [
[
-173.5,
-73.4
],
[
-173.5,
-73.4
]
],
"legs": [
[
174,
174
],
[
174,
174
]
]
}
},
{
"name": "Push-Ups | Decline",
"view": "side",
"prompt": "push-up position, hands on the floor, feet up on a bench",
"start": {
"torso": 5,
"arms": [
[
-94.8,
-85.0
],
[
-94.8,
-85.0
]
],
"legs": [
[
185,
185
],
[
185,
185
]
]
},
"end": {
"torso": -6,
"arms": [
[
-173.5,
-73.4
],
[
-173.5,
-73.4
]
],
"legs": [
[
174,
174
],
[
174,
174
]
]
}
},
{
"name": "Pushups (Close and Wide Hand Positions)",
"view": "side",
"prompt": "in a push-up position, hands and toes on the floor",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Russian Push Ups",
"view": "side",
"prompt": "push-up position on the floor, shifting back onto the forearms",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Scapular Pushups",
"view": "side",
"prompt": "straight-arm plank on the floor, arms locked",
"start": {
"torso": 17,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
207.6,
207.6
],
[
207.6,
207.6
]
]
},
"end": {
"torso": 27,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
201.4,
201.4
],
[
201.4,
201.4
]
]
}
},
{
"name": "Seated Cable chest fly",
"view": "side",
"prompt": "seated on a bench between two cable pulleys, a handle in each hand",
"start": {
"torso": 90,
"arms": [
[
180,
175
],
[
180,
175
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
5
],
[
0,
5
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Single-Arm Cable Crossover",
"view": "side",
"prompt": "split stance beside a high cable pulley, a handle in one hand",
"start": {
"torso": 80,
"arms": [
[
140,
130
],
[
-90,
-90
]
],
"legs": [
[
-75,
-90
],
[
-105,
-95
]
]
},
"end": {
"torso": 80,
"arms": [
[
-60,
-45
],
[
-90,
-90
]
],
"legs": [
[
-75,
-90
],
[
-105,
-95
]
]
}
},
{
"name": "Single-Arm Push-Up",
"view": "side",
"prompt": "push-up position on one hand and toes, feet wide, other arm behind the back",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
205,
25
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
205,
25
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Smith Machine Slight Incline Press",
"view": "side",
"prompt": "lying on a slight incline bench in a Smith machine, holding the bar",
"start": {
"torso": 160,
"head": 160,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 160,
"head": 160,
"arms": [
[
70,
70
],
[
70,
70
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Standing Cable Chest Press",
"view": "side",
"prompt": "split stance in front of two cables, a handle in each hand",
"start": {
"torso": 85,
"arms": [
[
-165,
10
],
[
-165,
10
]
],
"legs": [
[
-75,
-90
],
[
-105,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-75,
-90
],
[
-105,
-95
]
]
}
},
{
"name": "Suspended Crossess",
"view": "side",
"prompt": "leaning forward holding suspension strap handles, feet on the floor",
"start": {
"torso": 60,
"arms": [
[
175,
180
],
[
175,
180
]
],
"legs": [
[
-105,
-105
],
[
-105,
-105
]
]
},
"end": {
"torso": 75,
"arms": [
[
40,
40
],
[
40,
40
]
],
"legs": [
[
-100,
-100
],
[
-100,
-100
]
]
}
},
{
"name": "Suspended Push-Up",
"view": "side",
"prompt": "leaning forward with hands in suspension strap handles, feet on the floor",
"start": {
"torso": 45,
"arms": [
[
-45,
-45
],
[
-45,
-45
]
],
"legs": [
[
225,
225
],
[
225,
225
]
]
},
"end": {
"torso": 35,
"arms": [
[
-165,
10
],
[
-165,
10
]
],
"legs": [
[
215,
215
],
[
215,
215
]
]
}
},
{
"name": "Svend Press",
"view": "side",
"prompt": "standing, squeezing two weight plates together at the chest",
"start": {
"torso": 90,
"arms": [
[
-115,
40
],
[
-115,
40
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Weighted Push-ups",
"view": "side",
"prompt": "push-up position on the floor, a weight plate on the upper back",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "3/4 Sit-Up",
"view": "side",
"prompt": "lying face up on the floor, knees bent, feet anchored, hands behind the head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
120,
230
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 125,
"head": 125,
"arms": [
[
65,
175
],
[
65,
175
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Ab Crunch Machine",
"view": "side",
"prompt": "seated in an ab crunch machine, feet under pads, gripping the top handles",
"start": {
"torso": 95,
"arms": [
[
35,
175
],
[
35,
175
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 55,
"head": 40,
"arms": [
[
-5,
135
],
[
-5,
135
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Ab Roller",
"view": "side",
"prompt": "kneeling on the floor, both hands on an ab roller",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
180
],
[
-60,
180
]
]
},
"end": {
"torso": 8,
"arms": [
[
-24,
-24
],
[
-24,
-24
]
],
"legs": [
[
200,
180
],
[
200,
180
]
]
}
},
{
"name": "Ab Wheel",
"view": "side",
"prompt": "kneeling on the floor, both hands on an ab wheel",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
180
],
[
-60,
180
]
]
},
"end": {
"torso": 8,
"arms": [
[
-24,
-24
],
[
-24,
-24
]
],
"legs": [
[
200,
180
],
[
200,
180
]
]
}
},
{
"name": "Ab Wheel Rollout",
"view": "side",
"prompt": "kneeling on the floor, both hands on an ab wheel",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
180
],
[
-60,
180
]
]
},
"end": {
"torso": 8,
"arms": [
[
-24,
-24
],
[
-24,
-24
]
],
"legs": [
[
200,
180
],
[
200,
180
]
]
}
},
{
"name": "Advanced Kettlebell Windmill",
"view": "front",
"prompt": "standing feet apart, a kettlebell locked out overhead in one hand",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
-95,
-92
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
},
"end": {
"torso": 35,
"head": 60,
"arms": [
[
90,
90
],
[
-95,
-90
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
}
},
{
"name": "Alternate Heel Touchers",
"view": "side",
"prompt": "lying face up on the floor, knees bent, shoulders lifted",
"start": {
"torso": 165,
"head": 150,
"arms": [
[
-5,
-5
],
[
-5,
-5
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 160,
"head": 145,
"arms": [
[
-25,
-30
],
[
5,
5
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Barbell Rollout from Bench",
"view": "side",
"prompt": "kneeling on a bench, hands on a barbell on the floor in front",
"start": {
"torso": -35,
"head": -35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"face": "down"
},
"end": {
"torso": -15,
"head": -10,
"arms": [
[
-23,
-23
],
[
-23,
-23
]
],
"legs": [
[
170,
180
],
[
170,
180
]
],
"face": "down"
}
},
{
"name": "Barbell Side Bend",
"view": "front",
"prompt": "standing, a barbell across the back of the shoulders",
"start": {
"torso": 90,
"arms": [
[
200,
90
],
[
-20,
90
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
180,
70
],
[
-40,
70
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
}
},
{
"name": "Bent Press",
"view": "front",
"prompt": "standing feet apart, a kettlebell in one hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-60,
100
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
},
"end": {
"torso": 135,
"head": 110,
"arms": [
[
-100,
-95
],
[
90,
90
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
}
},
{
"name": "Bird Dog",
"view": "side",
"prompt": "on hands and knees on the floor",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 13,
"arms": [
[
13,
13
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
185,
185
]
]
}
},
{
"name": "Bosu Ball Cable Crunch With Side Bends",
"view": "side",
"prompt": "lying back on a BOSU ball, holding low cable handles beside the hips",
"start": {
"torso": 175,
"head": 175,
"arms": [
[
-10,
-5
],
[
-10,
-5
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
},
"end": {
"torso": 140,
"head": 125,
"arms": [
[
-40,
-35
],
[
-40,
-35
]
],
"legs": [
[
50,
-60
],
[
50,
-60
]
]
}
},
{
"name": "Bottoms Up",
"view": "side",
"prompt": "lying face up on the floor, arms by the sides",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 200,
"head": 190,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
100,
90
],
[
100,
90
]
]
}
},
{
"name": "Butt-Ups",
"view": "side",
"prompt": "forearm plank on the floor, elbows under the shoulders",
"start": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
},
"end": {
"torso": -40,
"head": -40,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-133,
-133
],
[
-133,
-133
]
],
"face": "down"
}
},
{
"name": "Cable Crunch",
"view": "side",
"prompt": "kneeling in front of a high cable, holding a rope beside the head",
"start": {
"torso": 80,
"arms": [
[
-40,
90
],
[
-40,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 15,
"head": 0,
"arms": [
[
-105,
25
],
[
-105,
25
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Cable Judo Flip",
"view": "side",
"prompt": "standing beside a low cable, holding a rope with both hands",
"start": {
"torso": 55,
"arms": [
[
-60,
-55
],
[
-60,
-55
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
110,
150
],
[
110,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Reverse Crunch",
"view": "side",
"prompt": "lying face up on a mat, ankle straps attached to a low cable",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
-20
],
[
20,
-20
]
]
},
"end": {
"torso": 190,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
115,
0
],
[
115,
0
]
]
}
},
{
"name": "Cable Russian Twists",
"view": "side",
"prompt": "upper back on a stability ball, hips up, holding a cable handle",
"start": {
"torso": 175,
"head": 175,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 175,
"head": 175,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Cable Seated Crunch",
"view": "side",
"prompt": "seated on a bench facing away from a high cable, holding a rope",
"start": {
"torso": 95,
"arms": [
[
-25,
105
],
[
-25,
105
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 45,
"head": 30,
"arms": [
[
-75,
55
],
[
-75,
55
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Captain's Chair Knee Raise",
"view": "side",
"prompt": "back against a vertical knee raise station, forearms on the pads",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
10,
-90
],
[
10,
-90
]
]
}
},
{
"name": "Captain's Chair Oblique Knee Raise",
"view": "side",
"prompt": "back against a vertical knee raise station, forearms on the pads",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
5,
-70
],
[
5,
-70
]
]
}
},
{
"name": "Cobra Stretch",
"view": "side",
"prompt": "lying face down on the floor, hands under the shoulders",
"start": {
"torso": 0,
"head": 0,
"arms": [
[
150,
-60
],
[
150,
-60
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"face": "down"
},
"end": {
"torso": 65,
"head": 80,
"arms": [
[
-80,
-50
],
[
-80,
-50
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
}
},
{
"name": "Cocoons",
"view": "side",
"prompt": "lying face up on the floor, arms stretched overhead",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
},
"end": {
"torso": 135,
"head": 125,
"arms": [
[
15,
15
],
[
15,
15
]
],
"legs": [
[
115,
10
],
[
115,
10
]
]
}
},
{
"name": "Crunch - Hands Overhead",
"view": "side",
"prompt": "lying face up on the floor, knees bent, arms straight overhead",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 150,
"head": 140,
"arms": [
[
150,
150
],
[
150,
150
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Crunch - Legs On Exercise Ball",
"view": "side",
"prompt": "lying face up on the floor, calves resting on an exercise ball",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
120,
230
]
],
"legs": [
[
80,
-5
],
[
80,
-5
]
]
},
"end": {
"torso": 145,
"head": 135,
"arms": [
[
85,
195
],
[
85,
195
]
],
"legs": [
[
80,
-5
],
[
80,
-5
]
]
}
},
{
"name": "Dead Bug",
"view": "side",
"prompt": "lying face up on the floor, arms up, knees bent over the hips",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
90,
0
],
[
90,
0
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
175,
175
],
[
90,
90
]
],
"legs": [
[
90,
0
],
[
5,
5
]
]
}
},
{
"name": "Decline Bench Leg Raise",
"view": "side",
"prompt": "lying face up on a decline bench, head at the top, gripping the handles",
"start": {
"torso": 160,
"head": 160,
"arms": [
[
165,
100
],
[
165,
100
]
],
"legs": [
[
-15,
-15
],
[
-15,
-15
]
]
},
"end": {
"torso": 160,
"head": 160,
"arms": [
[
165,
100
],
[
165,
100
]
],
"legs": [
[
75,
75
],
[
75,
75
]
]
}
},
{
"name": "Decline Reverse Crunch",
"view": "side",
"prompt": "lying face up on a decline bench, head at the top, holding the bench",
"start": {
"torso": 160,
"head": 160,
"arms": [
[
165,
100
],
[
165,
100
]
],
"legs": [
[
10,
-40
],
[
10,
-40
]
]
},
"end": {
"torso": 175,
"head": 160,
"arms": [
[
180,
115
],
[
180,
115
]
],
"legs": [
[
105,
40
],
[
105,
40
]
]
}
},
{
"name": "Double Kettlebell Windmill",
"view": "front",
"prompt": "standing feet apart, one kettlebell overhead, one on the floor",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
-95,
-92
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
},
"end": {
"torso": 35,
"head": 60,
"arms": [
[
90,
90
],
[
-95,
-90
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
}
},
{
"name": "Elbow to Knee",
"view": "side",
"prompt": "lying face up on the floor, ankle crossed over bent knee, hands behind head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
120,
230
]
],
"legs": [
[
70,
-5
],
[
60,
-60
]
]
},
"end": {
"torso": 135,
"head": 125,
"arms": [
[
45,
175
],
[
75,
185
]
],
"legs": [
[
70,
-5
],
[
60,
-60
]
]
}
},
{
"name": "Exercise Ball Crunch",
"view": "side",
"prompt": "lying back on an exercise ball, feet flat on the floor",
"start": {
"torso": 190,
"head": 190,
"arms": [
[
20,
130
],
[
20,
130
]
],
"legs": [
[
10,
-85
],
[
10,
-85
]
]
},
"end": {
"torso": 145,
"head": 135,
"arms": [
[
-25,
85
],
[
-25,
85
]
],
"legs": [
[
10,
-85
],
[
10,
-85
]
]
}
},
{
"name": "Flat Bench Leg Pull-In",
"view": "side",
"prompt": "lying face up on a flat bench, legs off the end, hands under hips",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
110,
10
],
[
110,
10
]
]
}
},
{
"name": "Flat Bench Lying Leg Raise",
"view": "side",
"prompt": "lying face up on a flat bench, legs off the end, hands under hips",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
90,
90
],
[
90,
90
]
]
}
},
{
"name": "Frog Sit-Ups",
"view": "side",
"prompt": "lying face up on the floor, soles together, knees out, hands behind head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
120,
230
]
],
"legs": [
[
35,
-30
],
[
35,
-30
]
]
},
"end": {
"torso": 100,
"head": 100,
"arms": [
[
40,
150
],
[
40,
150
]
],
"legs": [
[
35,
-30
],
[
35,
-30
]
]
}
},
{
"name": "Front Plank",
"view": "side",
"prompt": "forearms and toes on the floor, body straight",
"start": {
"torso": 8,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
210,
180
],
[
210,
180
]
]
},
"end": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
}
},
{
"name": "Gorilla Chin/Crunch",
"view": "side",
"prompt": "hanging from a pull-up bar with an underhand grip, knees bent",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-40,
105
],
[
-40,
105
]
],
"legs": [
[
40,
-70
],
[
40,
-70
]
]
}
},
{
"name": "Hanging Leg Raise",
"view": "side",
"prompt": "hanging from a pull-up bar, arms straight",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Hanging Pike",
"view": "side",
"prompt": "hanging from a pull-up bar, legs together",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 105,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
75,
75
],
[
75,
75
]
]
}
},
{
"name": "Hollow Hold",
"view": "side",
"prompt": "lying face up on the floor, arms stretched overhead",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 162,
"head": 150,
"arms": [
[
165,
165
],
[
165,
165
]
],
"legs": [
[
15,
15
],
[
15,
15
]
]
}
},
{
"name": "Jackknife Sit-Up",
"view": "side",
"prompt": "lying face up on the floor, arms overhead, legs straight",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 135,
"head": 125,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
45,
45
],
[
45,
45
]
]
}
},
{
"name": "Janda Sit-Up",
"view": "side",
"prompt": "lying face up on the floor, knees bent 90 degrees, arms extended",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 110,
"head": 110,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Kettlebell Figure 8",
"view": "side",
"prompt": "standing wide, bent at the hips, a kettlebell between the legs",
"start": {
"torso": 20,
"head": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
]
},
"end": {
"torso": 20,
"head": 30,
"arms": [
[
-115,
-125
],
[
-75,
-80
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
]
}
},
{
"name": "Kettlebell Pass Between The Legs",
"view": "side",
"prompt": "standing, bent at the hips, a kettlebell between the legs",
"start": {
"torso": 20,
"head": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
]
},
"end": {
"torso": 20,
"head": 30,
"arms": [
[
-115,
-125
],
[
-75,
-80
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
]
}
},
{
"name": "Kettlebell Windmill",
"view": "front",
"prompt": "standing feet apart, a kettlebell overhead in one hand",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
-95,
-92
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
},
"end": {
"torso": 35,
"head": 60,
"arms": [
[
90,
90
],
[
-95,
-90
]
],
"legs": [
[
-100,
-90
],
[
-80,
-90
]
]
}
},
{
"name": "Knee Raises",
"view": "side",
"prompt": "hanging from a pull-up bar, arms straight",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
10,
-80
],
[
10,
-80
]
]
}
},
{
"name": "Knee/Hip Raise On Parallel Bars",
"view": "side",
"prompt": "back against a vertical leg raise station, forearms on the pads",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
10,
-90
],
[
10,
-90
]
]
}
},
{
"name": "Kneeling Cable Crunch With Alternating Oblique Twists",
"view": "side",
"prompt": "kneeling in front of a high cable, holding a rope beside the head",
"start": {
"torso": 80,
"arms": [
[
-40,
90
],
[
-40,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 20,
"head": 5,
"arms": [
[
-100,
30
],
[
-100,
30
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Landmine 180's",
"view": "front",
"prompt": "standing, arms straight, holding the end of a landmine barbell",
"start": {
"torso": 90,
"arms": [
[
60,
70
],
[
120,
110
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-50,
-50
],
[
-70,
-70
]
],
"legs": [
[
-100,
-95
],
[
-85,
-95
]
]
}
},
{
"name": "Leg Pull-In",
"view": "side",
"prompt": "lying face up on a mat, hands under the hips, legs straight",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
},
"end": {
"torso": 170,
"head": 170,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
110,
10
],
[
110,
10
]
]
}
},
{
"name": "Leg Raises Pull Up Bar",
"view": "side",
"prompt": "hanging from a pull-up bar, arms straight",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 95,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Lower Back Curl",
"view": "side",
"prompt": "lying face down on the floor, arms by the sides",
"start": {
"torso": 0,
"head": 0,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"face": "down"
},
"end": {
"torso": 20,
"head": 30,
"arms": [
[
195,
195
],
[
195,
195
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"face": "right"
}
},
{
"name": "Medicine Ball Booklet Crunch",
"view": "side",
"prompt": "lying face up on the floor, holding a medicine ball overhead",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
},
"end": {
"torso": 125,
"head": 115,
"arms": [
[
50,
50
],
[
50,
50
]
],
"legs": [
[
60,
0
],
[
60,
0
]
]
}
},
{
"name": "Medicine Ball Full Twist",
"view": "side",
"prompt": "standing, holding a medicine ball at chest height",
"start": {
"torso": 90,
"arms": [
[
-40,
10
],
[
-40,
10
]
],
"legs": [
[
-85,
-92
],
[
-95,
-88
]
]
},
"end": {
"torso": 90,
"arms": [
[
175,
180
],
[
175,
180
]
],
"legs": [
[
-85,
-92
],
[
-95,
-88
]
]
}
},
{
"name": "Medicine Ball Twist",
"view": "side",
"prompt": "seated on the floor, knees bent, leaning back, holding a ball",
"start": {
"torso": 120,
"arms": [
[
-20,
0
],
[
-20,
0
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 120,
"arms": [
[
-50,
-70
],
[
-50,
-70
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
}
},
{
"name": "Negative Crunches",
"view": "side",
"prompt": "lying on a decline bench, feet hooked at the top, arms crossed",
"start": {
"torso": 190,
"head": 190,
"arms": [
[
20,
130
],
[
20,
130
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
},
"end": {
"torso": 100,
"arms": [
[
-70,
40
],
[
-70,
40
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
}
},
{
"name": "Oblique Crunches",
"view": "side",
"prompt": "lying face up on the floor, knees bent, one hand beside the head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 145,
"head": 135,
"arms": [
[
85,
195
],
[
-5,
-5
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Oblique Crunches - On The Floor",
"view": "front",
"prompt": "lying on one side on the floor, knees slightly bent, top hand behind head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
90,
225
]
],
"legs": [
[
-5,
-25
],
[
-5,
-25
]
]
},
"end": {
"torso": 150,
"head": 150,
"arms": [
[
-30,
-30
],
[
60,
195
]
],
"legs": [
[
-5,
-25
],
[
-5,
-25
]
]
}
},
{
"name": "One-Arm High-Pulley Cable Side Bends",
"view": "front",
"prompt": "standing beside a high cable, one hand holding the handle overhead",
"start": {
"torso": 90,
"arms": [
[
85,
90
],
[
-45,
-150
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
65,
70
],
[
-65,
-170
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
}
},
{
"name": "One-Arm Medicine Ball Slam",
"view": "side",
"prompt": "standing in a split stance, a medicine ball in one hand",
"start": {
"torso": 95,
"arms": [
[
100,
100
],
[
-90,
-90
]
],
"legs": [
[
-75,
-90
],
[
-105,
-95
]
]
},
"end": {
"torso": 50,
"head": 40,
"arms": [
[
-70,
-70
],
[
-100,
-95
]
],
"legs": [
[
-55,
-105
],
[
-115,
-110
]
]
}
},
{
"name": "Otis-Up",
"view": "side",
"prompt": "lying face up on the floor, knees bent, holding a weight at the chest",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
10,
120
],
[
10,
120
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 100,
"head": 100,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Overhead Stretch",
"view": "side",
"prompt": "standing straight, fingers laced together",
"start": {
"torso": 90,
"arms": [
[
-100,
70
],
[
-100,
70
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
92
],
[
90,
92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Pallof Press",
"view": "side",
"prompt": "standing sideways to a chest-height cable, both hands on the handle",
"start": {
"torso": 90,
"arms": [
[
-120,
25
],
[
-120,
25
]
],
"legs": [
[
-85,
-95
],
[
-95,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-85,
-95
],
[
-95,
-85
]
]
}
},
{
"name": "Pallof Press With Rotation",
"view": "side",
"prompt": "standing sideways to a chest-height cable, both hands on the handle",
"start": {
"torso": 90,
"arms": [
[
-120,
25
],
[
-120,
25
]
],
"legs": [
[
-85,
-95
],
[
-95,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-85,
-95
],
[
-95,
-85
]
]
}
},
{
"name": "Plate Twist",
"view": "side",
"prompt": "seated on the floor, legs straight, leaning back, holding a plate",
"start": {
"torso": 115,
"arms": [
[
-25,
0
],
[
-25,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 115,
"arms": [
[
-55,
-75
],
[
-55,
-75
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Press Sit-Up",
"view": "side",
"prompt": "lying on a decline bench, legs secured, a barbell on the chest",
"start": {
"torso": 190,
"head": 190,
"arms": [
[
20,
130
],
[
20,
130
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
},
"end": {
"torso": 100,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
20,
-70
],
[
20,
-70
]
]
}
},
{
"name": "Reverse Crunch",
"view": "side",
"prompt": "lying face up on the floor, palms down beside the hips",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
10,
10
],
[
10,
10
]
]
},
"end": {
"torso": 195,
"head": 182,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
115,
30
],
[
115,
30
]
]
}
},
{
"name": "Rope Crunch",
"view": "side",
"prompt": "kneeling in front of a high cable, holding a rope beside the head",
"start": {
"torso": 80,
"arms": [
[
-40,
90
],
[
-40,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 15,
"head": 0,
"arms": [
[
-105,
25
],
[
-105,
25
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Scissor Kick",
"view": "side",
"prompt": "lying face up on the floor, hands under the hips, legs raised",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
15,
15
],
[
15,
15
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
35,
35
],
[
3,
3
]
]
}
},
{
"name": "Seated Barbell Twist",
"view": "side",
"prompt": "seated at the end of a flat bench, a barbell across the shoulders",
"start": {
"torso": 90,
"arms": [
[
-100,
100
],
[
-100,
100
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-45,
130
],
[
-150,
75
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Seated Flat Bench Leg Pull-In",
"view": "side",
"prompt": "seated on a bench, leaning back, hands gripping the bench sides",
"start": {
"torso": 125,
"arms": [
[
-135,
-120
],
[
-135,
-120
]
],
"legs": [
[
-15,
-15
],
[
-15,
-15
]
]
},
"end": {
"torso": 110,
"arms": [
[
-120,
-105
],
[
-120,
-105
]
],
"legs": [
[
45,
-60
],
[
45,
-60
]
]
}
},
{
"name": "Seated Knee Tuck",
"view": "side",
"prompt": "seated on the floor, leaning back, hands on the floor behind",
"start": {
"torso": 125,
"arms": [
[
-135,
-120
],
[
-135,
-120
]
],
"legs": [
[
10,
10
],
[
10,
10
]
]
},
"end": {
"torso": 110,
"arms": [
[
-120,
-105
],
[
-120,
-105
]
],
"legs": [
[
50,
-55
],
[
50,
-55
]
]
}
},
{
"name": "Seated Leg Tucks",
"view": "side",
"prompt": "seated on a bench, leaning back, hands gripping the bench sides",
"start": {
"torso": 125,
"arms": [
[
-135,
-120
],
[
-135,
-120
]
],
"legs": [
[
-15,
-15
],
[
-15,
-15
]
]
},
"end": {
"torso": 110,
"arms": [
[
-120,
-105
],
[
-120,
-105
]
],
"legs": [
[
45,
-60
],
[
45,
-60
]
]
}
},
{
"name": "Seated Overhead Stretch",
"view": "side",
"prompt": "seated upright on a mat, soles of the feet together",
"start": {
"torso": 90,
"arms": [
[
-60,
-70
],
[
-60,
-70
]
],
"legs": [
[
20,
-154
],
[
20,
-154
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
92
],
[
90,
92
]
],
"legs": [
[
20,
-154
],
[
20,
-154
]
]
}
},
{
"name": "Side Bridge",
"view": "front",
"prompt": "lying on one side on the floor, propped on the forearm",
"start": {
"torso": 145,
"head": 145,
"arms": [
[
-90,
180
],
[
-25,
-175
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 160,
"head": 160,
"arms": [
[
-90,
205
],
[
-10,
-160
]
],
"legs": [
[
-20,
-20
],
[
-20,
-20
]
]
}
},
{
"name": "Side Jackknife",
"view": "front",
"prompt": "lying on one side on the floor, top hand behind the head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
90,
225
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 145,
"head": 145,
"arms": [
[
-35,
-35
],
[
55,
190
]
],
"legs": [
[
30,
30
],
[
30,
30
]
]
}
},
{
"name": "Sit Up Elbow Thrust",
"view": "side",
"prompt": "lying face up on the floor, knees bent, fists by the chest",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
10,
120
],
[
10,
120
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 100,
"arms": [
[
-10,
170
],
[
-160,
30
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Sit-Up",
"view": "side",
"prompt": "lying face up on the floor, knees bent, feet anchored, hands behind head",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
120,
230
],
[
120,
230
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 95,
"head": 95,
"arms": [
[
35,
145
],
[
35,
145
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
}
},
{
"name": "Sledgehammer Swings",
"view": "side",
"prompt": "standing in a split stance in front of a tire, holding a sledgehammer",
"start": {
"torso": 95,
"arms": [
[
115,
125
],
[
115,
125
]
],
"legs": [
[
-80,
-90
],
[
-100,
-95
]
]
},
"end": {
"torso": 40,
"head": 30,
"arms": [
[
-55,
-50
],
[
-55,
-50
]
],
"legs": [
[
-55,
-105
],
[
-110,
-115
]
]
}
},
{
"name": "Smith Machine Hip Raise",
"view": "side",
"prompt": "lying face up on a bench, feet pressing up against a Smith machine bar",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
85,
95
],
[
85,
95
]
]
},
"end": {
"torso": 200,
"head": 185,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
100,
100
],
[
100,
100
]
]
}
},
{
"name": "Spell Caster",
"view": "front",
"prompt": "standing with feet wide, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-100,
-100
],
[
-130,
-140
]
],
"legs": [
[
-105,
-90
],
[
-75,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
35,
45
],
[
50,
60
]
],
"legs": [
[
-105,
-90
],
[
-75,
-90
]
]
}
},
{
"name": "Spider Crawl",
"view": "side",
"prompt": "push-up position on the floor, hands and toes down",
"start": {
"torso": 23,
"arms": [
[
-101.9,
-78.9
],
[
-101.9,
-78.9
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 12,
"arms": [
[
-172.7,
-72.9
],
[
-172.7,
-72.9
]
],
"legs": [
[
-5.0,
-170.0
],
[
192,
192
]
]
}
},
{
"name": "Standing Cable Lift",
"view": "side",
"prompt": "standing sideways to a low cable, both hands on the handle",
"start": {
"torso": 65,
"head": 60,
"arms": [
[
-80,
-75
],
[
-80,
-75
]
],
"legs": [
[
-60,
-105
],
[
-60,
-105
]
]
},
"end": {
"torso": 95,
"arms": [
[
60,
70
],
[
60,
70
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Cable Wood Chop",
"view": "side",
"prompt": "standing sideways to a high cable, both hands on the handle",
"start": {
"torso": 95,
"arms": [
[
120,
130
],
[
120,
130
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 65,
"head": 55,
"arms": [
[
-70,
-60
],
[
-70,
-60
]
],
"legs": [
[
-60,
-105
],
[
-60,
-105
]
]
}
},
{
"name": "Standing Lateral Stretch",
"view": "front",
"prompt": "standing feet apart, one hand on the hip, other arm overhead",
"start": {
"torso": 90,
"arms": [
[
200,
-30
],
[
85,
95
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 115,
"head": 110,
"arms": [
[
225,
-5
],
[
110,
120
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
}
},
{
"name": "Standing Rope Crunch",
"view": "side",
"prompt": "standing with back to a high cable, holding a rope beside the head",
"start": {
"torso": 90,
"arms": [
[
-30,
100
],
[
-30,
100
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
]
},
"end": {
"torso": 45,
"head": 30,
"arms": [
[
-75,
55
],
[
-75,
55
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
]
}
},
{
"name": "Stomach Vacuum",
"view": "side",
"prompt": "standing upright, feet shoulder width, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-100,
-60
],
[
-100,
-60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 85
}
},
{
"name": "Supine One-Arm Overhead Throw",
"view": "side",
"prompt": "lying face up on the floor, knees bent, medicine ball in one hand",
"start": {
"torso": 180,
"arms": [
[
180,
180
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 120,
"arms": [
[
25,
25
],
[
-60,
-60
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 115
}
},
{
"name": "Supine Two-Arm Overhead Throw",
"view": "side",
"prompt": "lying face up on the floor, knees bent, medicine ball held overhead",
"start": {
"torso": 180,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 120,
"arms": [
[
25,
25
],
[
25,
25
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 115
}
},
{
"name": "Suspended Fallout",
"view": "side",
"prompt": "leaning forward into suspension straps, hands gripping the handles",
"start": {
"torso": 70,
"arms": [
[
-15,
-15
],
[
-15,
-15
]
],
"legs": [
[
-110,
-110
],
[
-110,
-110
]
],
"head": 70
},
"end": {
"torso": 40,
"arms": [
[
55,
55
],
[
55,
55
]
],
"legs": [
[
-140,
-140
],
[
-140,
-140
]
],
"head": 40
}
},
{
"name": "Suspended Reverse Crunch",
"view": "side",
"prompt": "push-up plank, feet in suspension strap handles behind",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
175
],
[
-60,
175
]
],
"head": 10,
"face": "down"
}
},
{
"name": "Toe Touchers",
"view": "side",
"prompt": "lying face up on the floor, legs straight up",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
90,
90
],
[
90,
90
]
],
"head": 180
},
"end": {
"torso": 145,
"arms": [
[
50,
50
],
[
50,
50
]
],
"legs": [
[
90,
90
],
[
90,
90
]
],
"head": 135
}
},
{
"name": "Torso Rotation",
"view": "front",
"prompt": "standing upright, holding an exercise ball with both hands",
"start": {
"torso": 90,
"arms": [
[
-80,
-30
],
[
-100,
-150
]
],
"legs": [
[
-95,
-95
],
[
-85,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
-5,
0
],
[
5,
5
]
],
"legs": [
[
-95,
-95
],
[
-85,
-85
]
]
}
},
{
"name": "Torso Twist",
"view": "front",
"prompt": "standing upright, feet planted, arms out",
"start": {
"torso": 90,
"arms": [
[
178,
172
],
[
2,
8
]
],
"legs": [
[
-95,
-95
],
[
-85,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
-10,
0
],
[
10,
10
]
],
"legs": [
[
-95,
-95
],
[
-85,
-85
]
]
}
},
{
"name": "Tuck Crunch",
"view": "side",
"prompt": "lying face up on the floor, hands behind head, knees raised",
"start": {
"torso": 180,
"arms": [
[
135,
-100
],
[
135,
-100
]
],
"legs": [
[
90,
0
],
[
90,
0
]
],
"head": 180
},
"end": {
"torso": 140,
"arms": [
[
95,
-140
],
[
95,
-140
]
],
"legs": [
[
110,
-30
],
[
110,
-30
]
],
"head": 130
}
},
{
"name": "Weighted Crunches",
"view": "side",
"prompt": "lying face up on the floor, knees bent, medicine ball held above chest",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 145,
"arms": [
[
55,
55
],
[
55,
55
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 135
}
},
{
"name": "Weighted Sit-Ups - With Bands",
"view": "side",
"prompt": "lying on a decline bench, feet hooked, band handles at collarbone",
"start": {
"torso": 200,
"arms": [
[
60,
200
],
[
60,
200
]
],
"legs": [
[
30,
-60
],
[
30,
-60
]
],
"head": 200
},
"end": {
"torso": 100,
"arms": [
[
-40,
100
],
[
-40,
100
]
],
"legs": [
[
30,
-60
],
[
30,
-60
]
],
"head": 95
}
},
{
"name": "Wind Sprints",
"view": "side",
"prompt": "hanging from a pull-up bar, overhand grip",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
0,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Bottoms-Up Clean From The Hang Position",
"view": "side",
"prompt": "standing, kettlebell in one hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 50
},
"end": {
"torso": 90,
"arms": [
[
-80,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Wrist Curl",
"view": "side",
"prompt": "seated on a bench facing a low cable, forearms on thighs, palms up",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Dumbbell Lying Pronation",
"view": "side",
"prompt": "lying face down on a flat bench, dumbbell in one hand",
"start": {
"torso": 0,
"arms": [
[
-90,
0
],
[
-15,
165
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
-90,
35
],
[
-15,
165
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Dumbbell Lying Supination",
"view": "side",
"prompt": "lying on a flat bench, dumbbell in one hand",
"start": {
"torso": 0,
"arms": [
[
-90,
0
],
[
-15,
165
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
-90,
-35
],
[
-15,
165
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Farmer's Walk",
"view": "side",
"prompt": "standing tall, a heavy weight in each hand at the sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-95
],
[
-85,
-85
]
],
"legs": [
[
-70,
-95
],
[
-110,
-115
]
]
}
},
{
"name": "Finger Curls",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, barbell palms up",
"start": {
"torso": 50,
"arms": [
[
-90,
-25
],
[
-90,
-25
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
10
],
[
-90,
10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Kneeling Forearm Stretch",
"view": "side",
"prompt": "kneeling on a mat, palms flat, fingers pointing back",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 5,
"face": "down"
},
"end": {
"torso": 15,
"arms": [
[
-67,
-67
],
[
-67,
-67
]
],
"legs": [
[
-60,
180
],
[
-60,
180
]
],
"head": 10,
"face": "down"
}
},
{
"name": "Palms-Down Dumbbell Wrist Curl Over A Bench",
"view": "side",
"prompt": "kneeling at a flat bench, forearms on it, dumbbells palms down",
"start": {
"torso": 50,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-80,
15
],
[
-80,
15
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
}
},
{
"name": "Palms-Down Wrist Curl Over A Bench",
"view": "side",
"prompt": "kneeling at a flat bench, forearms on it, barbell palms down",
"start": {
"torso": 50,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-80,
15
],
[
-80,
15
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
}
},
{
"name": "Palms-Up Barbell Wrist Curl Over A Bench",
"view": "side",
"prompt": "kneeling at a flat bench, forearms on it, barbell palms up",
"start": {
"torso": 50,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-80,
15
],
[
-80,
15
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
}
},
{
"name": "Palms-Up Dumbbell Wrist Curl Over A Bench",
"view": "side",
"prompt": "kneeling at a flat bench, forearms on it, dumbbells palms up",
"start": {
"torso": 50,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-80,
15
],
[
-80,
15
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 40
}
},
{
"name": "Plate Pinch",
"view": "side",
"prompt": "standing, two weight plates pinched together in each hand",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-110
],
[
-60,
-110
]
],
"head": 40
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, dumbbell palms down",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Rickshaw Carry",
"view": "side",
"prompt": "standing inside a loaded carry frame, gripping its handles",
"start": {
"torso": 50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-100
],
[
-20,
-100
]
],
"head": 60
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-95
],
[
-110,
-115
]
]
}
},
{
"name": "Seated Dumbbell Palms-Down Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, dumbbells palms down",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated Dumbbell Palms-Up Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, dumbbells palms up",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated One-Arm Dumbbell Palms-Down Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearm on thigh, dumbbell palm down",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated One-Arm Dumbbell Palms-Up Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearm on thigh, dumbbell palm up",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated Palm-Up Barbell Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, barbell palms up",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated Palms-Down Barbell Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, barbell palms down",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Seated Two-Arm Palms-Up Low-Pulley Wrist Curl",
"view": "side",
"prompt": "seated on a bench facing a low cable, forearms on thighs",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Standing Olympic Plate Hand Squeeze",
"view": "side",
"prompt": "standing, a weight plate gripped by the rim in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
-75
],
[
-80,
-75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Palms-Up Barbell Behind The Back Wrist Curl",
"view": "side",
"prompt": "standing, barbell held behind the glutes at arm's length",
"start": {
"torso": 90,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-100,
-75
],
[
-100,
-75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wrist Circles",
"view": "side",
"prompt": "standing, arms extended straight forward",
"start": {
"torso": 90,
"arms": [
[
0,
-15
],
[
0,
-15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
15
],
[
0,
15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wrist Curl",
"view": "side",
"prompt": "seated on a bench, forearms on thighs, dumbbells palms up",
"start": {
"torso": 50,
"arms": [
[
-90,
-20
],
[
-90,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
},
"end": {
"torso": 50,
"arms": [
[
-90,
15
],
[
-90,
15
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 40
}
},
{
"name": "Wrist Roller",
"view": "side",
"prompt": "standing, arms straight forward holding a wrist roller with hanging weight",
"start": {
"torso": 90,
"arms": [
[
0,
-10
],
[
0,
-10
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
15
],
[
0,
15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wrist Rotations with Straight Bar",
"view": "side",
"prompt": "standing, holding a barbell in front, arms extended",
"start": {
"torso": 90,
"arms": [
[
-30,
-15
],
[
-30,
-15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-30,
15
],
[
-30,
15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "High Knees",
"view": "side",
"prompt": "standing, running in place",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-130,
-40
],
[
-40,
60
]
],
"legs": [
[
5,
-95
],
[
-90,
-90
]
]
}
},
{
"name": "Jumping Jack",
"view": "front",
"prompt": "standing upright, bodyweight",
"start": {
"torso": 90,
"arms": [
[
-95,
-95
],
[
-85,
-85
]
],
"legs": [
[
-95,
-95
],
[
-85,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
110,
95
],
[
70,
85
]
],
"legs": [
[
-110,
-110
],
[
-70,
-70
]
]
}
},
{
"name": "Kettlebell Clean",
"view": "side",
"prompt": "standing, kettlebell in one hand",
"start": {
"torso": 40,
"arms": [
[
-100,
-100
],
[
-90,
-90
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 50
},
"end": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Stationary Bike Cardio",
"view": "side",
"prompt": "seated on a stationary bike, hands on the handlebars",
"start": {
"torso": 70,
"arms": [
[
-25,
-5
],
[
-25,
-5
]
],
"legs": [
[
20,
-95
],
[
-50,
-80
]
],
"head": 75
},
"end": {
"torso": 70,
"arms": [
[
-25,
-5
],
[
-25,
-5
]
],
"legs": [
[
-50,
-80
],
[
20,
-95
]
],
"head": 75
}
},
{
"name": "Treadmill Cardio",
"view": "side",
"prompt": "running on a treadmill",
"start": {
"torso": 80,
"arms": [
[
-125,
-40
],
[
-50,
40
]
],
"legs": [
[
-40,
-110
],
[
-115,
-160
]
]
},
"end": {
"torso": 80,
"arms": [
[
-50,
40
],
[
-125,
-40
]
],
"legs": [
[
-115,
-160
],
[
-40,
-110
]
]
}
},
{
"name": "Treadmill Run",
"view": "side",
"prompt": "running on a treadmill",
"start": {
"torso": 80,
"arms": [
[
-125,
-40
],
[
-50,
40
]
],
"legs": [
[
-40,
-110
],
[
-115,
-160
]
]
},
"end": {
"torso": 80,
"arms": [
[
-50,
40
],
[
-125,
-40
]
],
"legs": [
[
-115,
-160
],
[
-40,
-110
]
]
}
},
{
"name": "Treadmill Walk",
"view": "side",
"prompt": "walking on a treadmill",
"start": {
"torso": 90,
"arms": [
[
-105,
-100
],
[
-75,
-70
]
],
"legs": [
[
-70,
-95
],
[
-110,
-110
]
]
},
"end": {
"torso": 90,
"arms": [
[
-75,
-70
],
[
-105,
-100
]
],
"legs": [
[
-110,
-110
],
[
-70,
-95
]
]
}
},
{
"name": "Ankle On The Knee",
"view": "side",
"prompt": "lying face up on the floor, knees bent, feet flat",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
130
],
[
60,
-60
]
],
"head": 180
}
},
{
"name": "Barbell Glute Bridge",
"view": "side",
"prompt": "lying face up on the floor, barbell across the hips",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 200,
"arms": [
[
20,
20
],
[
20,
20
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
],
"head": 180
}
},
{
"name": "Bulgarian Split Squats Left",
"view": "side",
"prompt": "standing, rear foot resting on a chair behind",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-92
],
[
-120,
185
]
]
},
"end": {
"torso": 88,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-100
],
[
-100,
128
]
]
}
},
{
"name": "Bulgarian Squat with Dumbbells",
"view": "side",
"prompt": "dumbbells in hands, rear foot on a bench behind",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-92
],
[
-120,
185
]
]
},
"end": {
"torso": 88,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-100
],
[
-100,
128
]
]
}
},
{
"name": "Butt Lift (Bridge)",
"view": "side",
"prompt": "lying face up on the floor, knees bent, arms at sides",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 200,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
],
"head": 180
}
},
{
"name": "Cable Glute Extension",
"view": "side",
"prompt": "standing facing a cable machine, ankle strap on one leg",
"start": {
"torso": 80,
"arms": [
[
-15,
-15
],
[
-15,
-15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-15,
-15
],
[
-15,
-15
]
],
"legs": [
[
-130,
-130
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Pull-Through",
"view": "side",
"prompt": "standing facing away from a low cable, rope between legs",
"start": {
"torso": 20,
"arms": [
[
-105,
-110
],
[
-105,
-110
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Downward Facing Balance",
"view": "side",
"prompt": "lying face down over an exercise ball, hands on floor",
"start": {
"torso": -40,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
220,
220
],
[
220,
220
]
],
"head": -40,
"face": "down"
},
"end": {
"torso": -10,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
175
],
[
175,
175
]
],
"head": -5,
"face": "down"
}
},
{
"name": "Dumbbell Front Squat",
"view": "side",
"prompt": "standing, a dumbbell in each hand resting on the shoulders",
"start": {
"torso": 90,
"arms": [
[
-30,
120
],
[
-30,
120
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-30,
120
],
[
-30,
120
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Dumbbell Hip Thrust",
"view": "side",
"prompt": "upper back against a flat bench, dumbbell resting on the hips",
"start": {
"torso": 145,
"arms": [
[
-35,
-35
],
[
-35,
-35
]
],
"legs": [
[
30,
-55
],
[
30,
-55
]
],
"head": 140
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-100
],
[
0,
-100
]
],
"head": 170
}
},
{
"name": "Dumbbell Side Squat",
"view": "front",
"prompt": "standing in a wide stance, dumbbell held in front",
"start": {
"torso": 90,
"arms": [
[
-92,
-92
],
[
-88,
-88
]
],
"legs": [
[
-102,
-100
],
[
-78,
-80
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-135,
-90
],
[
-58,
-58
]
]
}
},
{
"name": "Foam Roller Gluteus",
"view": "side",
"prompt": "sitting on a foam roller, hands on the floor behind",
"start": {
"torso": 130,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
45,
-80
],
[
45,
-80
]
],
"head": 110
},
"end": {
"torso": 140,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
30,
-60
],
[
30,
-60
]
],
"head": 120
}
},
{
"name": "Glute Kickback",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 5,
"face": "down"
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
175
],
[
-90,
180
]
],
"head": 5,
"face": "down"
}
},
{
"name": "Hip Extension with Bands",
"view": "side",
"prompt": "standing facing a post, band attached to one ankle",
"start": {
"torso": 85,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-125,
-125
],
[
-90,
-90
]
]
}
},
{
"name": "Hip Lift with Band",
"view": "side",
"prompt": "lying face up on the floor, band across the hips",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 200,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
],
"head": 180
}
},
{
"name": "Isometric Squat to Failure",
"view": "side",
"prompt": "standing, bodyweight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-100
],
[
0,
-100
]
],
"head": 85
}
},
{
"name": "Knee Across The Body",
"view": "side",
"prompt": "lying face up on the floor",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
50,
-30
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Knee to Chest Stretch",
"view": "side",
"prompt": "lying face up on the floor",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
70,
30
],
[
70,
30
]
],
"legs": [
[
120,
10
],
[
120,
10
]
],
"head": 170
}
},
{
"name": "Kneeling Jump Squat",
"view": "side",
"prompt": "kneeling on the floor, barbell across the upper back",
"start": {
"torso": 75,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
-25,
180
],
[
-25,
180
]
],
"head": 85
},
"end": {
"torso": 65,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 80
}
},
{
"name": "Kneeling Kickbacks",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 5,
"face": "down"
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
175
],
[
-90,
180
]
],
"head": 5,
"face": "down"
}
},
{
"name": "Kneeling Squat",
"view": "side",
"prompt": "kneeling on a mat in a power rack, barbell across the upper back",
"start": {
"torso": 75,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
-25,
180
],
[
-25,
180
]
],
"head": 85
},
"end": {
"torso": 90,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Leg Lift",
"view": "side",
"prompt": "standing, hands holding the back of a chair in front",
"start": {
"torso": 85,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-100,
170
],
[
-90,
-90
]
]
}
},
{
"name": "Lying Figure Four Stretch",
"view": "side",
"prompt": "lying face up on the floor, one ankle crossed over the other knee",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
20,
130
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
35,
22
],
[
35,
22
]
],
"legs": [
[
45,
162
],
[
100,
10
]
],
"head": 170
}
},
{
"name": "Lying Glute",
"view": "side",
"prompt": "lying face up on the floor, partner kneeling beside",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
80,
170
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Machine Hip Abduction",
"view": "front",
"prompt": "seated in a hip abduction machine, knees against the pads",
"start": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-80,
-95
],
[
-100,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-140,
-95
],
[
-40,
-85
]
]
}
},
{
"name": "One Knee To Chest",
"view": "side",
"prompt": "lying face up on the floor",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
70,
30
],
[
70,
30
]
],
"legs": [
[
120,
10
],
[
0,
0
]
],
"head": 170
}
},
{
"name": "Physioball Hip Bridge",
"view": "side",
"prompt": "upper back resting on an exercise ball, feet flat on the floor",
"start": {
"torso": 145,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
30,
-55
],
[
30,
-55
]
],
"head": 140
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-100
],
[
0,
-100
]
],
"head": 170
}
},
{
"name": "Piriformis-SMR",
"view": "side",
"prompt": "sitting on a foam roller, hands on the floor behind",
"start": {
"torso": 130,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
45,
-80
],
[
45,
-80
]
],
"head": 110
},
"end": {
"torso": 130,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
15,
120
],
[
45,
-80
]
],
"head": 110
}
},
{
"name": "Reverse V-Squat",
"view": "side",
"prompt": "in a V-squat machine, chest against the pad",
"start": {
"torso": 80,
"arms": [
[
-30,
120
],
[
-30,
120
]
],
"legs": [
[
-95,
-90
],
[
-95,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
-30,
120
],
[
-30,
120
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Rubber Band Glute Kickback",
"view": "side",
"prompt": "on hands and knees, resistance band around one foot",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 5,
"face": "down"
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
175
],
[
-90,
180
]
],
"head": 5,
"face": "down"
}
},
{
"name": "Seated Glute",
"view": "side",
"prompt": "seated on the floor, one ankle crossed over the other knee",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
15,
120
],
[
45,
-60
]
]
},
"end": {
"torso": 50,
"arms": [
[
-20,
-20
],
[
-20,
-20
]
],
"legs": [
[
15,
120
],
[
45,
-60
]
],
"head": 45
}
},
{
"name": "Seated Hip Adduction",
"view": "front",
"prompt": "seated in a hip adduction machine, knees against the pads",
"start": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-140,
-95
],
[
-40,
-85
]
]
},
"end": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-80,
-95
],
[
-100,
-85
]
]
}
},
{
"name": "Single Leg Glute Bridge",
"view": "side",
"prompt": "lying face up on the floor, one knee pulled up",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
100,
10
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 200,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
80,
-10
],
[
20,
-50
]
],
"head": 180
}
},
{
"name": "Step-up with Knee Raise",
"view": "side",
"prompt": "standing facing a box, one foot on it",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
0,
-80
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-40,
60
],
[
-130,
-40
]
],
"legs": [
[
-90,
-90
],
[
0,
-90
]
]
}
},
{
"name": "90/90 Hamstring",
"view": "side",
"prompt": "lying face up on the floor, hands holding behind one thigh",
"start": {
"torso": 180,
"arms": [
[
45,
20
],
[
45,
20
]
],
"legs": [
[
90,
0
],
[
0,
0
]
],
"head": 170
},
"end": {
"torso": 180,
"arms": [
[
45,
20
],
[
45,
20
]
],
"legs": [
[
90,
90
],
[
0,
0
]
],
"head": 170
}
},
{
"name": "Alternating Hang Clean",
"view": "side",
"prompt": "standing, a kettlebell in each hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 50
},
"end": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Arabesque",
"view": "side",
"prompt": "standing on one leg, bodyweight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"arms": [
[
10,
10
],
[
10,
10
]
],
"legs": [
[
180,
180
],
[
-85,
-90
]
],
"head": 15,
"face": "down"
}
},
{
"name": "Ball Leg Curl",
"view": "side",
"prompt": "lying face up on the floor, heels on an exercise ball",
"start": {
"torso": 202,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
22,
22
],
[
22,
22
]
],
"head": 180
},
"end": {
"torso": 210,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-35
],
[
60,
-35
]
],
"head": 180
}
},
{
"name": "Band Good Morning",
"view": "side",
"prompt": "standing on a resistance band looped around the neck",
"start": {
"torso": 90,
"arms": [
[
-60,
135
],
[
-60,
135
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 5,
"arms": [
[
-145,
50
],
[
-145,
50
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 15,
"face": "down"
}
},
{
"name": "Band Good Morning (Pull Through)",
"view": "side",
"prompt": "standing, band from a post looped around the neck",
"start": {
"torso": 90,
"arms": [
[
-60,
135
],
[
-60,
135
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 5,
"arms": [
[
-145,
50
],
[
-145,
50
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 15,
"face": "down"
}
},
{
"name": "Box Jump (Multiple Response)",
"view": "side",
"prompt": "standing facing a plyo box",
"start": {
"torso": 60,
"arms": [
[
-140,
-140
],
[
-140,
-140
]
],
"legs": [
[
-30,
-110
],
[
-30,
-110
]
],
"head": 70
},
"end": {
"torso": 80,
"arms": [
[
40,
40
],
[
40,
40
]
],
"legs": [
[
20,
-100
],
[
20,
-100
]
],
"head": 85
}
},
{
"name": "Box Skip",
"view": "side",
"prompt": "bounding between plyo boxes",
"start": {
"torso": 85,
"arms": [
[
-100,
-30
],
[
-80,
20
]
],
"legs": [
[
-80,
-100
],
[
-110,
-120
]
]
},
"end": {
"torso": 80,
"arms": [
[
-130,
-40
],
[
-30,
60
]
],
"legs": [
[
10,
-90
],
[
-130,
-145
]
]
}
},
{
"name": "Chair Leg Extended Stretch",
"view": "side",
"prompt": "seated upright in a chair, hands gripping the seat sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
5,
5
],
[
0,
-90
]
]
}
},
{
"name": "Clean",
"view": "side",
"prompt": "barbell on the floor close to the shins",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
],
"head": 45
},
"end": {
"torso": 85,
"arms": [
[
-10,
165
],
[
-10,
165
]
],
"legs": [
[
-40,
-100
],
[
-40,
-100
]
]
}
},
{
"name": "Clean Deadlift",
"view": "side",
"prompt": "barbell on the floor close to the shins",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Double Kettlebell Alternating Hang Clean",
"view": "side",
"prompt": "standing, a kettlebell in each hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 50
},
"end": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Clean",
"view": "side",
"prompt": "a dumbbell in each hand, lowered to the floor",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-105
],
[
-15,
-105
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
-60,
105
],
[
-60,
105
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Donkey Kick",
"view": "side",
"prompt": "on hands and knees, dumbbell behind one knee",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 5,
"face": "down"
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
170,
80
],
[
-90,
180
]
],
"head": 5,
"face": "down"
}
},
{
"name": "Dumbbell Hang Power Cleans",
"view": "side",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-100
],
[
-75,
-100
]
],
"head": 55
},
"end": {
"torso": 85,
"arms": [
[
-60,
105
],
[
-60,
105
]
],
"legs": [
[
-60,
-105
],
[
-60,
-105
]
]
}
},
{
"name": "Dumbbell Romanian Deadlift",
"view": "side",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Dumbbell Snatch",
"view": "side",
"prompt": "dumbbell on the floor between the feet, gripped in one hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-105
],
[
-15,
-105
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Sumo Deadlift",
"view": "side",
"prompt": "wide stance, one dumbbell held between the legs",
"start": {
"torso": 50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-5,
-105
],
[
-5,
-105
]
],
"head": 65
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Floor Glute-Ham Raise",
"view": "side",
"prompt": "kneeling upright on a mat, feet anchored behind",
"start": {
"torso": 90,
"arms": [
[
-60,
120
],
[
-60,
120
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 30,
"arms": [
[
-60,
-60
],
[
-60,
-60
]
],
"legs": [
[
210,
180
],
[
210,
180
]
],
"head": 30,
"face": "down"
}
},
{
"name": "Front Box Jump",
"view": "side",
"prompt": "standing facing a plyo box",
"start": {
"torso": 60,
"arms": [
[
-140,
-140
],
[
-140,
-140
]
],
"legs": [
[
-30,
-110
],
[
-30,
-110
]
],
"head": 70
},
"end": {
"torso": 80,
"arms": [
[
40,
40
],
[
40,
40
]
],
"legs": [
[
20,
-100
],
[
20,
-100
]
],
"head": 85
}
},
{
"name": "Front Leg Raises",
"view": "side",
"prompt": "standing, one hand holding a chair for support",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-80,
-80
]
],
"legs": [
[
20,
20
],
[
-90,
-90
]
]
}
},
{
"name": "Glute Ham Raise",
"view": "side",
"prompt": "on a glute-ham raise machine, feet against the footplate",
"start": {
"torso": 0,
"arms": [
[
-150,
45
],
[
-150,
45
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 90,
"arms": [
[
-60,
135
],
[
-60,
135
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Good Morning off Pins",
"view": "side",
"prompt": "bent over in a rack, barbell across the upper back",
"start": {
"torso": 5,
"arms": [
[
145,
15
],
[
145,
15
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 90,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Hamstring Kicks",
"view": "side",
"prompt": "standing, one arm extended straight forward",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
0,
0
]
],
"legs": [
[
50,
50
],
[
-90,
-90
]
]
}
},
{
"name": "Hamstring Stretch",
"view": "side",
"prompt": "lying face up on the floor",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
45,
25
],
[
45,
25
]
],
"legs": [
[
95,
95
],
[
0,
0
]
],
"head": 170
}
},
{
"name": "Hamstring-SMR",
"view": "side",
"prompt": "seated, foam roller under the thighs, hands on the floor behind",
"start": {
"torso": 120,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
10,
-15
],
[
10,
-15
]
],
"head": 105
},
"end": {
"torso": 140,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-5,
-10
],
[
-5,
-10
]
],
"head": 120
}
},
{
"name": "Hang Snatch",
"view": "side",
"prompt": "standing, wide grip barbell at mid-thigh",
"start": {
"torso": 70,
"arms": [
[
-95,
-95
],
[
-95,
-95
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 75
},
"end": {
"torso": 85,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-40,
-100
],
[
-40,
-100
]
]
}
},
{
"name": "Hang Snatch - Below Knees",
"view": "side",
"prompt": "bent over, wide grip barbell just below the knees",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 55
},
"end": {
"torso": 85,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-40,
-100
],
[
-40,
-100
]
]
}
},
{
"name": "Hanging Bar Good Morning",
"view": "side",
"prompt": "bent over, barbell suspended from a rack across the upper back",
"start": {
"torso": 5,
"arms": [
[
145,
15
],
[
145,
15
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 90,
"arms": [
[
-130,
100
],
[
-130,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "High Knee Jumps",
"view": "side",
"prompt": "bodyweight, feet slightly wider than shoulders",
"start": {
"torso": 65,
"arms": [
[
-140,
-140
],
[
-140,
-140
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 80
},
"end": {
"torso": 85,
"arms": [
[
-10,
-10
],
[
-10,
-10
]
],
"legs": [
[
50,
-90
],
[
50,
-90
]
],
"head": 85
}
},
{
"name": "Hurdle Hops",
"view": "side",
"prompt": "standing behind a small hurdle on the floor",
"start": {
"torso": 70,
"head": 80,
"arms": [
[
-120,
-120
],
[
-120,
-120
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
},
"end": {
"torso": 85,
"arms": [
[
20,
40
],
[
20,
40
]
],
"legs": [
[
30,
-100
],
[
30,
-100
]
]
}
},
{
"name": "Inchworm",
"view": "side",
"prompt": "on a gym floor, hands reaching to the floor",
"start": {
"torso": -40,
"head": -60,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
}
},
{
"name": "Intermediate Groin Stretch",
"view": "side",
"prompt": "lying face up on a mat, a strap looped around one foot",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
30,
10
],
[
30,
10
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
70,
80
],
[
30,
10
]
],
"legs": [
[
85,
85
],
[
0,
0
]
]
}
},
{
"name": "Kettlebell Dead Clean",
"view": "side",
"prompt": "a kettlebell on the floor between his feet",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-80,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Deadlifts",
"view": "side",
"prompt": "a kettlebell on the floor between his feet",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Hang Clean",
"view": "side",
"prompt": "standing, holding a kettlebell hanging between his knees",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-80,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell One-Legged Deadlift",
"view": "side",
"prompt": "standing on one leg, a kettlebell in one hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 5,
"head": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
185,
185
],
[
-85,
-95
]
]
}
},
{
"name": "Kettlebell Sumo Deadlift",
"view": "side",
"prompt": "wide stance, a kettlebell on the floor between his feet",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-100
],
[
-20,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Knee Tuck Jump",
"view": "side",
"prompt": "standing on a gym floor, hands held out at waist height",
"start": {
"torso": 80,
"head": 85,
"arms": [
[
-30,
0
],
[
-30,
0
]
],
"legs": [
[
-60,
-110
],
[
-60,
-110
]
]
},
"end": {
"torso": 90,
"arms": [
[
-30,
0
],
[
-30,
0
]
],
"legs": [
[
50,
-90
],
[
50,
-90
]
]
}
},
{
"name": "Leg-Up Hamstring Stretch",
"view": "side",
"prompt": "lying face up on a mat, one knee bent with foot flat",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
60,
-60
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
60,
30
],
[
60,
30
]
],
"legs": [
[
88,
88
],
[
60,
-60
]
]
}
},
{
"name": "Linear 3-Part Start Technique",
"view": "side",
"prompt": "on a running track, crouched in a sprint start",
"start": {
"torso": 35,
"head": 45,
"arms": [
[
-80,
-80
],
[
-110,
-100
]
],
"legs": [
[
-50,
-100
],
[
-120,
-110
]
]
},
"end": {
"torso": 60,
"head": 65,
"arms": [
[
-60,
60
],
[
-130,
-80
]
],
"legs": [
[
20,
-90
],
[
-115,
-120
]
]
}
},
{
"name": "Linear Acceleration Wall Drill",
"view": "side",
"prompt": "leaning forward with both hands on a wall",
"start": {
"torso": 45,
"head": 50,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
225,
225
],
[
225,
225
]
]
},
"end": {
"torso": 45,
"head": 50,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
5,
-85
],
[
225,
225
]
]
}
},
{
"name": "Lunge Pass Through",
"view": "side",
"prompt": "standing, holding a kettlebell in one hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Lying Hamstring",
"view": "side",
"prompt": "lying face up on a mat, a partner kneeling beside him",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
90,
95
],
[
0,
0
]
]
}
},
{
"name": "Lying Leg Curl",
"view": "side",
"prompt": "lying face down on a leg curl machine, ankles under the pad",
"start": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
-100,
-60
],
[
-100,
-60
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
},
"end": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
-100,
-60
],
[
-100,
-60
]
],
"legs": [
[
180,
70
],
[
180,
70
]
]
}
},
{
"name": "Moving Claw Series",
"view": "side",
"prompt": "standing tall on a running track",
"start": {
"torso": 90,
"arms": [
[
-120,
-80
],
[
-60,
60
]
],
"legs": [
[
0,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-60,
60
],
[
-120,
-80
]
],
"legs": [
[
-60,
-70
],
[
-90,
-90
]
]
}
},
{
"name": "Muscle Snatch",
"view": "side",
"prompt": "standing, holding a barbell at mid thigh with a wide grip",
"start": {
"torso": 75,
"head": 85,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Natural Glute Ham Raise",
"view": "side",
"prompt": "kneeling on a pad, ankles hooked under a padded bar",
"start": {
"torso": 90,
"arms": [
[
290,
90
],
[
290,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 25,
"head": 30,
"arms": [
[
-50,
-20
],
[
-50,
-20
]
],
"legs": [
[
205,
180
],
[
205,
180
]
]
}
},
{
"name": "Nordic Hamstring Curl",
"view": "side",
"prompt": "kneeling on a mat, a partner holding his ankles",
"start": {
"torso": 90,
"arms": [
[
290,
90
],
[
290,
90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 25,
"head": 30,
"arms": [
[
-50,
-20
],
[
-50,
-20
]
],
"legs": [
[
205,
180
],
[
205,
180
]
]
}
},
{
"name": "One-Arm Kettlebell Clean",
"view": "side",
"prompt": "a kettlebell on the floor between his feet",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Swings",
"view": "side",
"prompt": "standing with feet apart, a kettlebell in one hand",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-120,
-120
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Open Palm Kettlebell Clean",
"view": "side",
"prompt": "a kettlebell on the floor between his feet",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Open Palm Kettlebell Clean",
"view": "side",
"prompt": "a kettlebell on the floor between his feet",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-60,
90
],
[
-60,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Pistol Squat",
"view": "side",
"prompt": "standing on one leg, arms reaching forward",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-30,
-30
],
[
-90,
-90
]
]
},
"end": {
"torso": 60,
"head": 75,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
10,
10
],
[
5,
-110
]
]
}
},
{
"name": "Platform Hamstring Slides",
"view": "side",
"prompt": "lying face up on a wooden floor, heels on a towel",
"start": {
"torso": 190,
"head": 185,
"arms": [
[
-5,
0
],
[
-5,
0
]
],
"legs": [
[
-5,
-5
],
[
-5,
-5
]
]
},
"end": {
"torso": 200,
"head": 195,
"arms": [
[
-5,
0
],
[
-5,
0
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
]
}
},
{
"name": "Power Clean",
"view": "side",
"prompt": "a loaded barbell on the floor in front of his shins",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-10,
120
],
[
-10,
120
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Power Clean from Blocks",
"view": "side",
"prompt": "a loaded barbell resting on blocks at knee height",
"start": {
"torso": 40,
"head": 50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-10,
120
],
[
-10,
120
]
],
"legs": [
[
-80,
-100
],
[
-80,
-100
]
]
}
},
{
"name": "Power Snatch",
"view": "side",
"prompt": "a loaded barbell on the floor, wide grip",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 85,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-60,
-110
],
[
-60,
-110
]
]
}
},
{
"name": "Power Stairs",
"view": "side",
"prompt": "gripping a heavy two-handled implement at the foot of stairs",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-85,
-95
],
[
-85,
-95
]
]
}
},
{
"name": "Prone Manual Hamstring",
"view": "side",
"prompt": "lying face down on a mat, a partner holding his ankle",
"start": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
},
"end": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
180,
80
],
[
180,
180
]
]
}
},
{
"name": "Prowler Sprint",
"view": "side",
"prompt": "pushing a loaded sled, hands on its tall upright poles",
"start": {
"torso": 30,
"head": 40,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-40,
-100
],
[
-115,
-140
]
]
},
"end": {
"torso": 30,
"head": 40,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-115,
-140
],
[
-40,
-100
]
]
}
},
{
"name": "Reverse Band Sumo Deadlift",
"view": "side",
"prompt": "wide stance in a power rack, barbell on the floor with bands",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-100
],
[
-20,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Hyperextension",
"view": "side",
"prompt": "lying face down on a reverse hyper machine, gripping the handles",
"start": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
-90,
-30
],
[
-90,
-30
]
],
"legs": [
[
-80,
-80
],
[
-80,
-80
]
]
},
"end": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
-90,
-30
],
[
-90,
-30
]
],
"legs": [
[
178,
178
],
[
178,
178
]
]
}
},
{
"name": "Romanian Deadlift from Deficit",
"view": "side",
"prompt": "standing on a low platform, holding a barbell at arm's length",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-82,
-95
],
[
-82,
-95
]
]
}
},
{
"name": "Runner's Stretch",
"view": "side",
"prompt": "on a mat in a split stance",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-90
],
[
-115,
-100
]
]
},
"end": {
"torso": 20,
"head": 35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
210,
210
]
]
}
},
{
"name": "Seated Band Hamstring Curl",
"view": "side",
"prompt": "seated on a bench, a band anchored low around one ankle",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
0
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-125
],
[
0,
-90
]
]
}
},
{
"name": "Seated Floor Hamstring Stretch",
"view": "side",
"prompt": "sitting on a mat, one leg straight, other knee bent",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
0,
0
],
[
15,
195
]
]
},
"end": {
"torso": 30,
"head": 20,
"arms": [
[
-15,
-20
],
[
-15,
-20
]
],
"legs": [
[
0,
0
],
[
15,
195
]
]
}
},
{
"name": "Seated Hamstring",
"view": "side",
"prompt": "sitting on a mat, legs straight, partner standing behind",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 30,
"head": 20,
"arms": [
[
-15,
-20
],
[
-15,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Seated Hamstring Stretch",
"view": "side",
"prompt": "sitting on a mat with both legs straight",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 30,
"head": 20,
"arms": [
[
-15,
-20
],
[
-15,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Seated Hamstring and Calf Stretch",
"view": "side",
"prompt": "sitting on a mat, legs straight, band looped around one foot",
"start": {
"torso": 90,
"arms": [
[
-20,
-10
],
[
-20,
-10
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 50,
"head": 45,
"arms": [
[
-10,
-20
],
[
-10,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Single-Leg Deadlift with Dumbbell",
"view": "side",
"prompt": "standing on one leg, a dumbbell in one hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 5,
"head": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
185,
185
],
[
-85,
-95
]
]
}
},
{
"name": "Sliding Lateral Lunge",
"view": "front",
"prompt": "standing, holding a kettlebell at the chest, one foot on a slider",
"start": {
"torso": 90,
"arms": [
[
-85,
60
],
[
-95,
120
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-85,
60
],
[
-95,
120
]
],
"legs": [
[
-125,
-55
],
[
-50,
-50
]
]
}
},
{
"name": "Smith Machine Hang Power Clean",
"view": "side",
"prompt": "holding a Smith machine bar at knee height",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-10,
120
],
[
-10,
120
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Smith Machine Split Squat",
"view": "side",
"prompt": "in a Smith machine, bar across upper back, split stance",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-75,
-90
],
[
-115,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Smith Machine Stiff-Legged Deadlift",
"view": "side",
"prompt": "in a Smith machine, holding the bar at arm's length",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-82,
-95
],
[
-82,
-95
]
]
}
},
{
"name": "Snatch Deadlift",
"view": "side",
"prompt": "a loaded barbell on the floor, wide grip",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Snatch Pull",
"view": "side",
"prompt": "a loaded barbell on the floor, wide grip",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 95,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-80
],
[
-95,
-80
]
]
}
},
{
"name": "Split Snatch",
"view": "side",
"prompt": "a loaded barbell on the floor, wide grip",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Standing Hamstring and Calf Stretch",
"view": "side",
"prompt": "standing, a strap looped around the front foot",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-75,
-75
],
[
-95,
-90
]
]
},
"end": {
"torso": 50,
"head": 55,
"arms": [
[
-30,
-40
],
[
-30,
-40
]
],
"legs": [
[
-60,
-60
],
[
-120,
-70
]
]
}
},
{
"name": "Standing Toe Touches",
"view": "side",
"prompt": "standing on a gym floor, feet together",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": -40,
"head": -60,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Stiff-Leg Deadlift",
"view": "side",
"prompt": "standing, holding a barbell at arm's length",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-86,
-92
],
[
-86,
-92
]
]
}
},
{
"name": "Stiff-Legged Dumbbell Deadlift",
"view": "side",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-86,
-92
],
[
-86,
-92
]
]
}
},
{
"name": "Sumo Deadlift with Bands",
"view": "side",
"prompt": "wide stance, barbell on the floor with bands looped over it",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-100
],
[
-20,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Sumo Deadlift with Chains",
"view": "side",
"prompt": "wide stance, barbell on the floor with chains draped on it",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-100
],
[
-20,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "The Straddle",
"view": "side",
"prompt": "sitting on a mat, legs straight and spread wide",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 30,
"head": 20,
"arms": [
[
-15,
-20
],
[
-15,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Upper Back-Leg Grab",
"view": "side",
"prompt": "sitting on the floor, knees bent, feet flat",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-60,
-30
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 40,
"head": 30,
"arms": [
[
-95,
170
],
[
-95,
170
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
}
},
{
"name": "Vertical Swing",
"view": "side",
"prompt": "standing with feet apart, one dumbbell held in both hands",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Wide Stance Stiff Legs",
"view": "side",
"prompt": "wide stance, barbell on the floor in front",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-82,
-95
],
[
-82,
-95
]
]
}
},
{
"name": "World's Greatest Stretch",
"view": "side",
"prompt": "in a deep lunge on a mat, hands on the floor",
"start": {
"torso": 20,
"head": 35,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
210,
210
]
]
},
"end": {
"torso": 20,
"head": 35,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
210,
210
]
]
}
},
{
"name": "Atlas Stone Trainer",
"view": "side",
"prompt": "squatting over a round stone on the floor, arms wrapped around it",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-60,
60
],
[
-60,
60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Atlas Stones",
"view": "side",
"prompt": "squatting over a heavy round stone, a platform in front",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-10,
30
],
[
-10,
30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Axle Deadlift",
"view": "side",
"prompt": "a loaded thick axle bar on the floor in front of his shins",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Back Extension",
"view": "side",
"prompt": "on a 45-degree back extension bench, hips on the pad",
"start": {
"torso": -45,
"head": -50,
"face": "down",
"arms": [
[
155,
-45
],
[
155,
-45
]
],
"legs": [
[
225,
225
],
[
225,
225
]
]
},
"end": {
"torso": 45,
"head": 45,
"face": "down",
"arms": [
[
245,
45
],
[
245,
45
]
],
"legs": [
[
225,
225
],
[
225,
225
]
]
}
},
{
"name": "Cat Stretch",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 13,
"head": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 18,
"head": -50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Cat-Cow Stretch",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 8,
"head": 70,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 18,
"head": -50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
}
},
{
"name": "Child's Pose",
"view": "side",
"prompt": "kneeling on a mat",
"start": {
"torso": 13,
"head": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": -15,
"head": -20,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-25,
180
],
[
-25,
180
]
]
}
},
{
"name": "Conventional Deadlift",
"view": "side",
"prompt": "a loaded barbell on the floor in front of his shins",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Crossover Reverse Lunge",
"view": "side",
"prompt": "standing on a gym floor, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Dancer's Stretch",
"view": "side",
"prompt": "sitting on a mat, legs straight",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 90,
"arms": [
[
-20,
-60
],
[
-110,
-95
]
],
"legs": [
[
55,
-60
],
[
0,
0
]
]
}
},
{
"name": "Deadlift with Bands",
"view": "side",
"prompt": "a loaded barbell on the floor with bands looped over it",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Deadlift with Chains",
"view": "side",
"prompt": "a loaded barbell on the floor with chains draped on it",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Deficit Deadlift",
"view": "side",
"prompt": "standing on a low platform, a loaded barbell in front of his shins",
"start": {
"torso": 25,
"head": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-10,
-110
],
[
-10,
-110
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Hug A Ball",
"view": "side",
"prompt": "sitting on a mat, legs wide around an exercise ball",
"start": {
"torso": 90,
"arms": [
[
-60,
-20
],
[
-60,
-20
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
},
"end": {
"torso": 45,
"head": 40,
"arms": [
[
0,
-60
],
[
0,
-60
]
],
"legs": [
[
5,
5
],
[
5,
5
]
]
}
},
{
"name": "Hug Knees To Chest",
"view": "side",
"prompt": "lying face up on a mat",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 180,
"head": 170,
"arms": [
[
50,
20
],
[
50,
20
]
],
"legs": [
[
120,
10
],
[
120,
10
]
]
}
},
{
"name": "Hyperextensions With No Hyperextension Bench",
"view": "side",
"prompt": "face down on a flat bench, hips over the edge, partner holding legs",
"start": {
"torso": -80,
"head": -85,
"face": "down",
"arms": [
[
120,
-80
],
[
120,
-80
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
},
"end": {
"torso": 0,
"head": 0,
"face": "down",
"arms": [
[
200,
0
],
[
200,
0
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
}
},
{
"name": "Keg Load",
"view": "side",
"prompt": "squatting over a heavy keg on the ground, a platform in front",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-20,
60
],
[
-20,
60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Lower Back-SMR",
"view": "side",
"prompt": "sitting on a mat, a foam roller behind his lower back, knees bent",
"start": {
"torso": 90,
"arms": [
[
-30,
0
],
[
-30,
0
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 160,
"head": 150,
"arms": [
[
80,
60
],
[
80,
60
]
],
"legs": [
[
30,
-50
],
[
30,
-50
]
]
}
},
{
"name": "Pelvic Tilt Into Bridge",
"view": "side",
"prompt": "lying face up on a mat, knees bent, feet flat",
"start": {
"torso": 180,
"head": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
]
},
"end": {
"torso": 200,
"head": 195,
"arms": [
[
-5,
0
],
[
-5,
0
]
],
"legs": [
[
20,
-50
],
[
20,
-50
]
]
}
},
{
"name": "Pyramid",
"view": "side",
"prompt": "hips on top of an exercise ball, hands and toes on the floor",
"start": {
"torso": 0,
"head": -5,
"face": "down",
"arms": [
[
-75,
-90
],
[
-75,
-90
]
],
"legs": [
[
229,
229
],
[
229,
229
]
]
},
"end": {
"torso": -30,
"head": -40,
"face": "down",
"arms": [
[
-45,
-45
],
[
-45,
-45
]
],
"legs": [
[
229,
229
],
[
229,
229
]
]
}
},
{
"name": "Rack Pull with Bands",
"view": "side",
"prompt": "in a power rack, banded barbell on pins below the knees",
"start": {
"torso": 45,
"head": 55,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Reverse Band Deadlift",
"view": "side",
"prompt": "in a power rack, barbell on the floor, bands from the top",
"start": {
"torso": 30,
"head": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Seated Good Mornings",
"view": "side",
"prompt": "seated on a box in a power rack, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 15,
"head": 25,
"arms": [
[
-195,
5
],
[
-195,
5
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Standing Pelvic Tilt",
"view": "side",
"prompt": "standing, knees soft, hands on hips",
"start": {
"torso": 84,
"head": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-80,
-100
],
[
-80,
-100
]
]
},
"end": {
"torso": 98,
"head": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-80,
-100
],
[
-80,
-100
]
]
}
},
{
"name": "Stiff Leg Barbell Good Morning",
"view": "side",
"prompt": "standing, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"head": 20,
"arms": [
[
-200,
0
],
[
-200,
0
]
],
"legs": [
[
-84,
-94
],
[
-84,
-94
]
]
}
},
{
"name": "Weighted Ball Hyperextension",
"view": "side",
"prompt": "hips on an exercise ball, toes on floor, plate held at chest",
"start": {
"torso": -40,
"head": -45,
"face": "down",
"arms": [
[
160,
-40
],
[
160,
-40
]
],
"legs": [
[
229,
229
],
[
229,
229
]
]
},
"end": {
"torso": 10,
"head": 15,
"face": "down",
"arms": [
[
210,
10
],
[
210,
10
]
],
"legs": [
[
229,
229
],
[
229,
229
]
]
}
},
{
"name": "Chin To Chest Stretch",
"view": "side",
"prompt": "sitting on the floor, hands clasped behind head",
"start": {
"torso": 90,
"head": 90,
"arms": [
[
60,
150
],
[
60,
150
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 90,
"head": 30,
"arms": [
[
10,
140
],
[
10,
140
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Isometric Neck Exercise - Front And Back",
"view": "side",
"prompt": "seated on a bench, hands pressed on the head",
"start": {
"torso": 90,
"head": 90,
"arms": [
[
-20,
120
],
[
-20,
120
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"head": 90,
"arms": [
[
60,
150
],
[
60,
150
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Isometric Neck Exercise - Sides",
"view": "front",
"prompt": "standing upright, one hand pressed on the side of the head",
"start": {
"torso": 90,
"head": 90,
"arms": [
[
150,
40
],
[
-88,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"head": 90,
"arms": [
[
-92,
-90
],
[
30,
140
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Lying Face Down Plate Neck Resistance",
"view": "side",
"prompt": "face down on a flat bench, head off the end, plate behind head",
"start": {
"torso": 0,
"head": -40,
"face": "down",
"arms": [
[
20,
40
],
[
20,
40
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
},
"end": {
"torso": 0,
"head": 20,
"face": "down",
"arms": [
[
20,
40
],
[
20,
40
]
],
"legs": [
[
180,
180
],
[
180,
180
]
]
}
},
{
"name": "Lying Face Up Plate Neck Resistance",
"view": "side",
"prompt": "face up on a flat bench, head off the end, plate on forehead",
"start": {
"torso": 180,
"head": 215,
"face": "up",
"arms": [
[
70,
170
],
[
70,
170
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 180,
"head": 140,
"face": "up",
"arms": [
[
70,
170
],
[
70,
170
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Neck-SMR",
"view": "side",
"prompt": "standing, holding a small roller against the back of the neck",
"start": {
"torso": 90,
"head": 90,
"arms": [
[
60,
150
],
[
60,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"head": 70,
"arms": [
[
40,
160
],
[
40,
160
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Seated Head Harness Neck Resistance",
"view": "side",
"prompt": "sitting at the end of a bench, weighted harness on the head",
"start": {
"torso": 50,
"head": -30,
"arms": [
[
-75,
-75
],
[
-75,
-75
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 50,
"head": 60,
"arms": [
[
-75,
-75
],
[
-75,
-75
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Side Neck Stretch",
"view": "front",
"prompt": "standing upright, shoulders relaxed",
"start": {
"torso": 90,
"head": 90,
"arms": [
[
-92,
-90
],
[
-88,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"head": 125,
"arms": [
[
110,
10
],
[
-88,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "All Fours Quad Stretch",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 13,
"head": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 13,
"head": 20,
"arms": [
[
175,
165
],
[
-90,
-90
]
],
"legs": [
[
190,
60
],
[
-90,
180
]
]
}
},
{
"name": "Alternate Leg Diagonal Bound",
"view": "side",
"prompt": "on an open field in a staggered stance",
"start": {
"torso": 75,
"head": 80,
"arms": [
[
-60,
30
],
[
-120,
-90
]
],
"legs": [
[
-70,
-95
],
[
-110,
-110
]
]
},
"end": {
"torso": 75,
"head": 80,
"arms": [
[
-120,
-90
],
[
-30,
40
]
],
"legs": [
[
10,
-80
],
[
-140,
-150
]
]
}
},
{
"name": "Ankle Roll",
"view": "side",
"prompt": "sitting on the floor, one leg crossed over the other knee",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
25,
-40
],
[
45,
-45
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
25,
-5
],
[
45,
-45
]
]
}
},
{
"name": "Backward Drag",
"view": "side",
"prompt": "holding straps of a loaded sled in front, walking backward",
"start": {
"torso": 105,
"head": 95,
"arms": [
[
-20,
-10
],
[
-20,
-10
]
],
"legs": [
[
-60,
-110
],
[
-110,
-80
]
]
},
"end": {
"torso": 105,
"head": 95,
"arms": [
[
-20,
-10
],
[
-20,
-10
]
],
"legs": [
[
-110,
-80
],
[
-60,
-110
]
]
}
},
{
"name": "Band Squat",
"view": "side",
"prompt": "standing on a resistance band, band ends at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-80,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 65,
"head": 80,
"arms": [
[
-80,
100
],
[
-80,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
]
}
},
{
"name": "Band Terminal Knee extension",
"view": "side",
"prompt": "standing, a band looped behind one knee anchored in front",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-65,
-110
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Banded Ankle Mobility",
"view": "side",
"prompt": "half kneeling on a mat, a band around the front ankle",
"start": {
"torso": 90,
"arms": [
[
-50,
-60
],
[
-50,
-60
]
],
"legs": [
[
0,
-90
],
[
-90,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
-40,
-60
],
[
-40,
-60
]
],
"legs": [
[
-10,
-55
],
[
-90,
180
]
]
}
},
{
"name": "Barbell Back Squat",
"view": "side",
"prompt": "in a squat rack, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 65,
"head": 80,
"arms": [
[
-145,
55
],
[
-145,
55
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
]
}
},
{
"name": "Barbell Full Squat",
"view": "side",
"prompt": "in a squat rack, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"head": 80,
"arms": [
[
-140,
60
],
[
-140,
60
]
],
"legs": [
[
15,
-115
],
[
15,
-115
]
]
}
},
{
"name": "Barbell Lunges Standing",
"view": "side",
"prompt": "standing, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Barbell Side Split Squat",
"view": "front",
"prompt": "wide stance, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-150,
80
],
[
-30,
100
]
],
"legs": [
[
-105,
-90
],
[
-75,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-150,
80
],
[
-30,
100
]
],
"legs": [
[
-125,
-55
],
[
-50,
-50
]
]
}
},
{
"name": "Barbell Step Back Lunge",
"view": "side",
"prompt": "standing, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Barbell Walking Lunge",
"view": "side",
"prompt": "standing, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Bear Crawl Sled Drags",
"view": "side",
"prompt": "on hands and feet, a sled chained to a belt behind",
"start": {
"torso": 0,
"head": 10,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-95,
-160
],
[
-95,
-160
]
]
},
"end": {
"torso": 0,
"head": 10,
"arms": [
[
-70,
-100
],
[
-100,
-85
]
],
"legs": [
[
-75,
-150
],
[
-110,
-160
]
]
}
},
{
"name": "Bench Jump",
"view": "side",
"prompt": "standing beside a low bench on the floor",
"start": {
"torso": 70,
"head": 80,
"arms": [
[
-120,
-120
],
[
-120,
-120
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
},
"end": {
"torso": 85,
"arms": [
[
20,
40
],
[
20,
40
]
],
"legs": [
[
30,
-100
],
[
30,
-100
]
]
}
},
{
"name": "Bench Sprint",
"view": "side",
"prompt": "standing, one foot on a low box",
"start": {
"torso": 85,
"arms": [
[
-60,
60
],
[
-120,
-80
]
],
"legs": [
[
-5,
-100
],
[
-90,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-120,
-80
],
[
-60,
60
]
],
"legs": [
[
-90,
-90
],
[
-5,
-100
]
]
}
},
{
"name": "Bicycling",
"view": "side",
"prompt": "seated on a road bicycle, hands on the handlebars",
"start": {
"torso": 60,
"head": 70,
"arms": [
[
-10,
-30
],
[
-10,
-30
]
],
"legs": [
[
-10,
-110
],
[
-60,
-90
]
]
},
"end": {
"torso": 60,
"head": 70,
"arms": [
[
-10,
-30
],
[
-10,
-30
]
],
"legs": [
[
-60,
-90
],
[
-10,
-110
]
]
}
},
{
"name": "Bicycling, Stationary",
"view": "side",
"prompt": "seated on a stationary exercise bike, hands on the handlebars",
"start": {
"torso": 60,
"head": 70,
"arms": [
[
-10,
-30
],
[
-10,
-30
]
],
"legs": [
[
-10,
-110
],
[
-60,
-90
]
]
},
"end": {
"torso": 60,
"head": 70,
"arms": [
[
-10,
-30
],
[
-10,
-30
]
],
"legs": [
[
-60,
-90
],
[
-10,
-110
]
]
}
},
{
"name": "Bodyweight Squat",
"view": "side",
"prompt": "standing on a gym floor, hands behind head",
"start": {
"torso": 90,
"arms": [
[
60,
150
],
[
60,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 65,
"head": 80,
"arms": [
[
60,
150
],
[
60,
150
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
]
}
},
{
"name": "Bodyweight Walking Lunge",
"view": "side",
"prompt": "standing on a gym floor, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
-40
],
[
-110,
-40
]
],
"legs": [
[
-40,
-90
],
[
-130,
-160
]
]
}
},
{
"name": "Box Squat",
"view": "side",
"prompt": "in a squat rack, barbell across upper back, box behind",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"head": 80,
"arms": [
[
-140,
60
],
[
-140,
60
]
],
"legs": [
[
0,
-95
],
[
0,
-95
]
]
}
},
{
"name": "Box Squat with Bands",
"view": "side",
"prompt": "in a squat rack, banded barbell across upper back, box behind",
"start": {
"torso": 90,
"arms": [
[
-120,
80
],
[
-120,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"head": 80,
"arms": [
[
-140,
60
],
[
-140,
60
]
],
"legs": [
[
0,
-95
],
[
0,
-95
]
]
}
},
{
"name": "Box Squat with Chains",
"view": "side",
"prompt": "standing in a power rack, barbell with chains on upper back, box behind",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Bulgarian Split Squat",
"view": "side",
"prompt": "standing, rear foot on a bench behind, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-120,
-175
]
]
},
"end": {
"torso": 82,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-10,
-88
],
[
-105,
130
]
],
"head": 88
}
},
{
"name": "Cable Deadlifts",
"view": "side",
"prompt": "standing between two low cable pulleys, a handle in each hand",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
],
"head": 40
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Car Deadlift",
"view": "side",
"prompt": "standing inside a loaded car-deadlift frame, gripping side handles",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-35,
-100
],
[
-35,
-100
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Chair Squat",
"view": "side",
"prompt": "standing in a Smith machine, bar across upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 68,
"arms": [
[
-132,
53
],
[
-132,
53
]
],
"legs": [
[
0,
-100
],
[
0,
-100
]
],
"head": 82
}
},
{
"name": "Clean Pull",
"view": "side",
"prompt": "barbell on the floor in front of the shins, overhand grip",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-105
],
[
-15,
-105
]
],
"head": 45
},
"end": {
"torso": 95,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-88,
-78
],
[
-88,
-78
]
],
"head": 95
}
},
{
"name": "Clean from Blocks",
"view": "side",
"prompt": "barbell resting on blocks at knee height, overhand grip",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
],
"head": 60
},
"end": {
"torso": 78,
"arms": [
[
-12,
138
],
[
-12,
138
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Conan's Wheel",
"view": "side",
"prompt": "holding a long loaded lever bar in the crook of the elbows",
"start": {
"torso": 85,
"arms": [
[
-75,
15
],
[
-75,
15
]
],
"legs": [
[
-75,
-95
],
[
-105,
-120
]
]
},
"end": {
"torso": 85,
"arms": [
[
-75,
15
],
[
-75,
15
]
],
"legs": [
[
-105,
-120
],
[
-75,
-95
]
]
}
},
{
"name": "Cossack Squat",
"view": "front",
"prompt": "standing in a very wide stance, hands together in front of chest",
"start": {
"torso": 90,
"arms": [
[
-60,
10
],
[
-120,
170
]
],
"legs": [
[
-118,
-118
],
[
-62,
-62
]
]
},
"end": {
"torso": 98,
"arms": [
[
-60,
10
],
[
-120,
170
]
],
"legs": [
[
-160,
-95
],
[
-42,
-42
]
]
}
},
{
"name": "Depth Jump Leap",
"view": "side",
"prompt": "standing on a low box, a higher box a few feet ahead",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-30,
-120
],
[
-30,
-120
]
]
}
},
{
"name": "Double Leg Butt Kick",
"view": "side",
"prompt": "standing on the floor, arms at sides",
"start": {
"torso": 70,
"arms": [
[
-125,
-110
],
[
-125,
-110
]
],
"legs": [
[
-50,
-110
],
[
-50,
-110
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
60,
80
],
[
60,
80
]
],
"legs": [
[
-80,
110
],
[
-80,
110
]
]
}
},
{
"name": "Dumbbell Lunges Standing",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand at the sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
-100,
180
]
]
}
},
{
"name": "Dumbbell Seated Box Jump",
"view": "side",
"prompt": "seated on a bench facing a box, dumbbell held at the chest",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-75,
85
],
[
-75,
85
]
],
"legs": [
[
-40,
-110
],
[
-40,
-110
]
],
"head": 85
}
},
{
"name": "Elevated Back Lunge",
"view": "side",
"prompt": "standing on a low platform, barbell across upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-5,
-90
],
[
-110,
170
]
]
}
},
{
"name": "Elliptical Trainer",
"view": "side",
"prompt": "standing on an elliptical trainer, hands on the moving handles",
"start": {
"torso": 85,
"arms": [
[
-40,
20
],
[
-75,
-10
]
],
"legs": [
[
-70,
-100
],
[
-105,
-95
]
]
},
"end": {
"torso": 85,
"arms": [
[
-75,
-10
],
[
-40,
20
]
],
"legs": [
[
-105,
-95
],
[
-70,
-100
]
]
}
},
{
"name": "Fast Skipping",
"view": "side",
"prompt": "standing on the floor, light athletic stance",
"start": {
"torso": 88,
"arms": [
[
-125,
-60
],
[
-50,
40
]
],
"legs": [
[
0,
-90
],
[
-95,
-95
]
]
},
"end": {
"torso": 88,
"arms": [
[
-50,
40
],
[
-125,
-60
]
],
"legs": [
[
-95,
-95
],
[
0,
-90
]
]
}
},
{
"name": "Foam Roller Adductors",
"view": "side",
"prompt": "lying face down on forearms, foam roller under the inner thigh",
"start": {
"torso": 16,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
188,
188
],
[
188,
188
]
],
"head": 20,
"face": "down"
},
"end": {
"torso": 18,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
-10,
180
],
[
190,
190
]
],
"head": 22,
"face": "down"
}
},
{
"name": "Foam Roller Anterior tibialis",
"view": "side",
"prompt": "on hands and knees, shins resting on a foam roller",
"start": {
"torso": 0,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 15,
"arms": [
[
-75,
-75
],
[
-75,
-75
]
],
"legs": [
[
-120,
180
],
[
-120,
180
]
],
"head": 15,
"face": "down"
}
},
{
"name": "Foam Roller Iliotibial band",
"view": "side",
"prompt": "lying on one side on a forearm, foam roller under outer thigh",
"start": {
"torso": 12,
"arms": [
[
-60,
-80
],
[
-90,
0
]
],
"legs": [
[
-45,
-95
],
[
195,
195
]
],
"head": 15
},
"end": {
"torso": 25,
"arms": [
[
-60,
-80
],
[
-90,
0
]
],
"legs": [
[
-45,
-95
],
[
185,
185
]
],
"head": 25
}
},
{
"name": "Foam Roller quadriceps",
"view": "side",
"prompt": "lying face down on forearms, foam roller under the thighs",
"start": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
188,
188
],
[
188,
188
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 22,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
168,
185
],
[
168,
185
]
],
"head": 25,
"face": "down"
}
},
{
"name": "Frankenstein Squat",
"view": "side",
"prompt": "barbell on front of shoulders, arms straight out in front, no hands",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Freehand Jump Squat",
"view": "side",
"prompt": "standing on the floor, arms crossed over the chest",
"start": {
"torso": 65,
"arms": [
[
-95,
95
],
[
-95,
95
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
-70,
120
],
[
-70,
120
]
],
"legs": [
[
-90,
-78
],
[
-90,
-78
]
]
}
},
{
"name": "Frog Hops",
"view": "side",
"prompt": "on the floor, hands behind the head",
"start": {
"torso": 70,
"arms": [
[
100,
0
],
[
100,
0
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 82
},
"end": {
"torso": 70,
"arms": [
[
100,
0
],
[
100,
0
]
],
"legs": [
[
-115,
-110
],
[
-115,
-110
]
],
"head": 80
}
},
{
"name": "Front Cone Hops (or hurdle hops)",
"view": "side",
"prompt": "standing behind a row of small cones on the floor",
"start": {
"torso": 70,
"arms": [
[
-125,
-110
],
[
-125,
-110
]
],
"legs": [
[
-50,
-110
],
[
-50,
-110
]
],
"head": 80
},
"end": {
"torso": 80,
"arms": [
[
-20,
-10
],
[
-20,
-10
]
],
"legs": [
[
-15,
-115
],
[
-15,
-115
]
]
}
},
{
"name": "Front Squat (Clean Grip)",
"view": "side",
"prompt": "standing, barbell racked on the front of the shoulders, elbows high",
"start": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 78,
"arms": [
[
-12,
138
],
[
-12,
138
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Front Squats With Two Kettlebells",
"view": "side",
"prompt": "standing, a kettlebell racked at each shoulder",
"start": {
"torso": 90,
"arms": [
[
-80,
100
],
[
-80,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 78,
"arms": [
[
-92,
88
],
[
-92,
88
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Goblet Squat",
"view": "side",
"prompt": "standing, holding one dumbbell vertically against the chest",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-75,
85
],
[
-75,
85
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Hamstring Foam roller",
"view": "side",
"prompt": "sitting on the floor, foam roller under the hamstrings, hands behind",
"start": {
"torso": 130,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-8,
-8
],
[
-8,
-8
]
],
"head": 110
},
"end": {
"torso": 125,
"arms": [
[
-95,
-95
],
[
-95,
-95
]
],
"legs": [
[
10,
-15
],
[
10,
-15
]
],
"head": 105
}
},
{
"name": "Hang Clean",
"view": "side",
"prompt": "standing, barbell held at mid-thigh, overhand grip",
"start": {
"torso": 70,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 80
},
"end": {
"torso": 78,
"arms": [
[
-12,
138
],
[
-12,
138
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Hang Clean - Below the Knees",
"view": "side",
"prompt": "standing bent over, barbell held just below the knees",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-65,
-95
],
[
-65,
-95
]
],
"head": 55
},
"end": {
"torso": 78,
"arms": [
[
-12,
138
],
[
-12,
138
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Heaving Snatch Balance",
"view": "side",
"prompt": "standing, barbell across upper back with a wide grip",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Hip Crossover",
"view": "side",
"prompt": "lying face up on the floor, knees bent, arms out to sides",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
25,
-20
],
[
30,
-15
]
],
"head": 180
}
},
{
"name": "Hip Flexion with Band",
"view": "side",
"prompt": "standing, resistance band from a low post looped around one ankle",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-40,
-40
],
[
-90,
-90
]
]
}
},
{
"name": "Hip Flexor Stretch",
"view": "side",
"prompt": "half-kneeling on the floor, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-115,
-50
],
[
-115,
-50
]
],
"legs": [
[
0,
-90
],
[
-90,
180
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
-45
],
[
-110,
-45
]
],
"legs": [
[
0,
-65
],
[
-115,
180
]
]
}
},
{
"name": "Hip-Out and Hip-In Lateral Steps",
"view": "front",
"prompt": "standing, resistance band around the legs, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-30
],
[
-60,
-150
]
],
"legs": [
[
-118,
-100
],
[
-62,
-80
]
]
}
},
{
"name": "Intermediate Hip Flexor and Quad Stretch",
"view": "side",
"prompt": "lying face down on the floor, a strap looped around one foot",
"start": {
"torso": 0,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 5,
"arms": [
[
160,
150
],
[
180,
180
]
],
"legs": [
[
180,
70
],
[
180,
180
]
],
"head": 10,
"face": "down"
}
},
{
"name": "Iron Crosses (stretch)",
"view": "side",
"prompt": "lying face down on the floor, arms out to the sides",
"start": {
"torso": 0,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
165,
95
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Jerk Dip Squat",
"view": "side",
"prompt": "standing, barbell racked on the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
}
},
{
"name": "Jogging, Treadmill",
"view": "side",
"prompt": "jogging on a treadmill",
"start": {
"torso": 85,
"arms": [
[
-120,
-30
],
[
-60,
30
]
],
"legs": [
[
-65,
-100
],
[
-110,
-150
]
]
},
"end": {
"torso": 85,
"arms": [
[
-60,
30
],
[
-120,
-30
]
],
"legs": [
[
-110,
-150
],
[
-65,
-100
]
]
}
},
{
"name": "Kettlebell Goblet Squat",
"view": "side",
"prompt": "standing, holding a kettlebell by the horns at the chest",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-75,
85
],
[
-75,
85
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Kettlebell Pistol Squat",
"view": "side",
"prompt": "standing on one leg, kettlebell held by the horns at the chest",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
-60,
-60
],
[
-90,
-90
]
]
},
"end": {
"torso": 60,
"arms": [
[
-70,
60
],
[
-70,
60
]
],
"legs": [
[
3,
3
],
[
0,
-110
]
],
"head": 75
}
},
{
"name": "Kneeling Hip Flexor",
"view": "side",
"prompt": "half-kneeling on a mat, back foot top-down on the floor",
"start": {
"torso": 90,
"arms": [
[
-60,
-70
],
[
-60,
-70
]
],
"legs": [
[
0,
-90
],
[
-90,
180
]
]
},
"end": {
"torso": 95,
"arms": [
[
-55,
-65
],
[
-55,
-65
]
],
"legs": [
[
0,
-65
],
[
-115,
180
]
]
}
},
{
"name": "Leverage Deadlift",
"view": "side",
"prompt": "standing inside a lever deadlift machine, gripping low side handles",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-30,
-105
],
[
-30,
-105
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Linear Depth Jump",
"view": "side",
"prompt": "standing on a box, a second box a few feet ahead",
"start": {
"torso": 90,
"arms": [
[
-100,
-95
],
[
-100,
-95
]
],
"legs": [
[
-60,
-80
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-30,
-120
],
[
-30,
-120
]
]
}
},
{
"name": "Looking At Ceiling",
"view": "side",
"prompt": "kneeling upright on the floor, toes tucked",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 145,
"arms": [
[
-95,
-95
],
[
-95,
-95
]
],
"legs": [
[
-100,
180
],
[
-100,
180
]
],
"head": 200,
"face": "up"
}
},
{
"name": "Lunge Sprint",
"view": "side",
"prompt": "in a Smith machine, bar across upper back, split stance",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
0,
-90
],
[
-100,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-100,
180
],
[
0,
-90
]
]
}
},
{
"name": "Lunge with Twist Stretch",
"view": "side",
"prompt": "standing on the floor, arms at sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
95
],
[
-70,
-80
]
],
"legs": [
[
0,
-90
],
[
-100,
180
]
]
}
},
{
"name": "Lying Hamstring Stretch with Band",
"view": "side",
"prompt": "lying face up on the floor, band looped around one foot",
"start": {
"torso": 180,
"arms": [
[
15,
5
],
[
15,
5
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
55,
60
],
[
20,
10
]
],
"legs": [
[
85,
85
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Lying Prone Quadriceps",
"view": "side",
"prompt": "lying face down on the floor, head resting on folded arms",
"start": {
"torso": 0,
"arms": [
[
0,
180
],
[
0,
180
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
0,
180
],
[
0,
180
]
],
"legs": [
[
180,
60
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Marching High Knees",
"view": "side",
"prompt": "standing tall on the floor",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
-80
],
[
-40,
60
]
],
"legs": [
[
5,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Olympic Squat",
"view": "side",
"prompt": "standing, barbell high on the upper back, feet hip width",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 78,
"arms": [
[
-122,
63
],
[
-122,
63
]
],
"legs": [
[
10,
-115
],
[
10,
-115
]
],
"head": 88
}
},
{
"name": "On Your Side Quad Stretch",
"view": "side",
"prompt": "lying on one side on the floor, bottom knee bent in front",
"start": {
"torso": 180,
"arms": [
[
10,
0
],
[
160,
0
]
],
"legs": [
[
0,
0
],
[
40,
-40
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
5,
-5
],
[
160,
0
]
],
"legs": [
[
0,
170
],
[
40,
-40
]
],
"head": 180
}
},
{
"name": "On-Your-Back Quad Stretch",
"view": "side",
"prompt": "lying face up on a flat bench, one leg hanging off the side",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-70,
-90
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
-20,
15
],
[
0,
0
]
],
"legs": [
[
-80,
110
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "One Half Locust",
"view": "side",
"prompt": "lying face down on the floor, one hand under the hip",
"start": {
"torso": 0,
"arms": [
[
180,
180
],
[
180,
180
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 10,
"arms": [
[
165,
165
],
[
180,
180
]
],
"legs": [
[
180,
35
],
[
180,
180
]
],
"head": 20
}
},
{
"name": "One-Arm Overhead Kettlebell Squats",
"view": "side",
"prompt": "standing, one kettlebell locked out overhead",
"start": {
"torso": 90,
"arms": [
[
92,
92
],
[
-60,
-30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
95,
95
],
[
-30,
-10
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Pendular Hack",
"view": "side",
"prompt": "standing on a pendulum squat machine, shoulders under the pads",
"start": {
"torso": 80,
"arms": [
[
-40,
70
],
[
-40,
70
]
],
"legs": [
[
-80,
-85
],
[
-80,
-85
]
]
},
"end": {
"torso": 72,
"arms": [
[
-50,
60
],
[
-50,
60
]
],
"legs": [
[
0,
-110
],
[
0,
-110
]
],
"head": 82
}
},
{
"name": "Pigeon Stretch",
"view": "side",
"prompt": "on hands and knees on a mat",
"start": {
"torso": 10,
"arms": [
[
-75,
-75
],
[
-75,
-75
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 10,
"face": "down"
},
"end": {
"torso": 80,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-15,
180
],
[
190,
190
]
],
"head": 85
}
},
{
"name": "Plantarflexion Stretch with Band",
"view": "side",
"prompt": "sitting on the floor, legs straight, band around one foot",
"start": {
"torso": 90,
"arms": [
[
-30,
-20
],
[
-30,
-20
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 85,
"arms": [
[
-60,
30
],
[
-60,
30
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Plie Dumbbell Squat",
"view": "side",
"prompt": "standing in a wide stance, one dumbbell held between the legs",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-100
],
[
-15,
-100
]
]
}
},
{
"name": "Power Jerk",
"view": "side",
"prompt": "standing, barbell racked on the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
92,
92
],
[
92,
92
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
}
},
{
"name": "Power Snatch from Blocks",
"view": "side",
"prompt": "barbell resting on blocks at knee height, wide grip",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
],
"head": 60
},
"end": {
"torso": 85,
"arms": [
[
97,
97
],
[
97,
97
]
],
"legs": [
[
-45,
-115
],
[
-45,
-115
]
],
"head": 90
}
},
{
"name": "Quad Stretch",
"view": "side",
"prompt": "lying on one side on the floor, strap looped around the top foot",
"start": {
"torso": 180,
"arms": [
[
10,
0
],
[
160,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
-10
],
[
160,
0
]
],
"legs": [
[
-5,
170
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Quadriceps Stretch",
"view": "side",
"prompt": "standing on one leg on the floor",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-100
],
[
-20,
-10
]
],
"legs": [
[
-90,
100
],
[
-90,
-90
]
]
}
},
{
"name": "Quadriceps-SMR",
"view": "side",
"prompt": "lying face down on forearms, foam roller under one thigh",
"start": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
188,
188
],
[
-20,
180
]
],
"head": 15,
"face": "down"
},
"end": {
"torso": 22,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
168,
185
],
[
-20,
180
]
],
"head": 25,
"face": "down"
}
},
{
"name": "Quick Leap",
"view": "side",
"prompt": "standing facing a box a foot away",
"start": {
"torso": 70,
"arms": [
[
-125,
-110
],
[
-125,
-110
]
],
"legs": [
[
-50,
-110
],
[
-50,
-110
]
],
"head": 80
},
"end": {
"torso": 78,
"arms": [
[
-30,
-20
],
[
-30,
-20
]
],
"legs": [
[
-15,
-115
],
[
-15,
-115
]
]
}
},
{
"name": "Rear Leg Raises",
"view": "side",
"prompt": "on hands and knees on an exercise mat",
"start": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 13,
"face": "down"
},
"end": {
"torso": 13,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
175,
170
],
[
-90,
180
]
],
"head": 13,
"face": "down"
}
},
{
"name": "Recumbent Bike",
"view": "side",
"prompt": "seated on a recumbent bike, hands on side handles",
"start": {
"torso": 110,
"arms": [
[
-80,
-10
],
[
-80,
-10
]
],
"legs": [
[
10,
-20
],
[
35,
-60
]
]
},
"end": {
"torso": 110,
"arms": [
[
-80,
-10
],
[
-80,
-10
]
],
"legs": [
[
35,
-60
],
[
10,
-20
]
]
}
},
{
"name": "Reverse Band Box Squat",
"view": "side",
"prompt": "standing in a power rack, banded barbell on upper back, box behind",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Reverse Band Power Squat",
"view": "side",
"prompt": "standing in a power rack, banded barbell on upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 60,
"arms": [
[
-140,
45
],
[
-140,
45
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 78
}
},
{
"name": "Reverse Nordic Curl",
"view": "side",
"prompt": "kneeling upright on a mat, arms straight out in front",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
]
},
"end": {
"torso": 125,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
-55,
180
],
[
-55,
180
]
],
"head": 120
}
},
{
"name": "Rickshaw Deadlift",
"view": "side",
"prompt": "standing inside a loaded rickshaw frame, gripping side handles",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-35,
-100
],
[
-35,
-100
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Rocket Jump",
"view": "side",
"prompt": "standing on the floor, arms close to the body",
"start": {
"torso": 70,
"arms": [
[
-110,
-100
],
[
-110,
-100
]
],
"legs": [
[
-50,
-110
],
[
-50,
-110
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
92,
92
],
[
92,
92
]
],
"legs": [
[
-90,
-78
],
[
-90,
-78
]
]
}
},
{
"name": "Rope Jumping",
"view": "side",
"prompt": "holding a jump rope handle in each hand",
"start": {
"torso": 88,
"arms": [
[
-75,
-10
],
[
-75,
-10
]
],
"legs": [
[
-80,
-100
],
[
-80,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
-40
],
[
-80,
-40
]
],
"legs": [
[
-90,
-78
],
[
-90,
-78
]
]
}
},
{
"name": "Rowing, Stationary",
"view": "side",
"prompt": "seated on a rowing machine, feet strapped in, holding the handle",
"start": {
"torso": 70,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
50,
-80
],
[
50,
-80
]
],
"head": 75
},
"end": {
"torso": 110,
"arms": [
[
-150,
10
],
[
-150,
10
]
],
"legs": [
[
-5,
-5
],
[
-5,
-5
]
],
"head": 100
}
},
{
"name": "Runners Lunge Stretch",
"view": "side",
"prompt": "on the floor, hands down beside the front foot",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 10,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-10,
-85
],
[
-145,
-145
]
],
"head": 20
}
},
{
"name": "Running, Treadmill",
"view": "side",
"prompt": "running on a treadmill",
"start": {
"torso": 80,
"arms": [
[
-130,
-40
],
[
-50,
50
]
],
"legs": [
[
-50,
-100
],
[
-125,
-170
]
]
},
"end": {
"torso": 80,
"arms": [
[
-50,
50
],
[
-130,
-40
]
],
"legs": [
[
-125,
-170
],
[
-50,
-100
]
]
}
},
{
"name": "Sandbag Load",
"view": "side",
"prompt": "heavy sandbag on the floor, a waist-high platform ahead",
"start": {
"torso": 35,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-10,
-110
],
[
-10,
-110
]
],
"head": 50
},
"end": {
"torso": 95,
"arms": [
[
-40,
90
],
[
-40,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Scissors Jump",
"view": "side",
"prompt": "on the floor in a deep lunge stance",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-120,
-90
]
],
"legs": [
[
0,
-90
],
[
-100,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-90
],
[
-60,
-30
]
],
"legs": [
[
-100,
180
],
[
0,
-90
]
]
}
},
{
"name": "Side Hop-Sprint",
"view": "front",
"prompt": "standing beside a small cone on the floor",
"start": {
"torso": 90,
"arms": [
[
-110,
-100
],
[
-70,
-80
]
],
"legs": [
[
-97,
-88
],
[
-83,
-92
]
]
},
"end": {
"torso": 85,
"arms": [
[
-130,
-120
],
[
-50,
-60
]
],
"legs": [
[
-110,
-130
],
[
-80,
-100
]
]
}
},
{
"name": "Side Standing Long Jump",
"view": "front",
"prompt": "standing on the floor in an athletic stance",
"start": {
"torso": 90,
"arms": [
[
-100,
-95
],
[
-80,
-85
]
],
"legs": [
[
-100,
-85
],
[
-80,
-95
]
]
},
"end": {
"torso": 105,
"arms": [
[
150,
140
],
[
120,
110
]
],
"legs": [
[
-110,
-120
],
[
-95,
-110
]
]
}
},
{
"name": "Side to Side Box Shuffle",
"view": "front",
"prompt": "standing beside a low box, one foot on top of it",
"start": {
"torso": 90,
"arms": [
[
-110,
-100
],
[
-60,
-50
]
],
"legs": [
[
-150,
-90
],
[
-80,
-80
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
-130
],
[
-70,
-80
]
],
"legs": [
[
-100,
-100
],
[
-30,
-90
]
]
}
},
{
"name": "Single Leg Butt Kick",
"view": "side",
"prompt": "standing on one leg, other knee raised",
"start": {
"torso": 90,
"arms": [
[
-120,
-60
],
[
-50,
40
]
],
"legs": [
[
0,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
70,
85
],
[
70,
85
]
],
"legs": [
[
-80,
110
],
[
-90,
-78
]
]
}
},
{
"name": "Single Leg Push-off",
"view": "side",
"prompt": "standing beside a low box, one foot on top of it",
"start": {
"torso": 85,
"arms": [
[
-115,
-100
],
[
-115,
-100
]
],
"legs": [
[
-20,
-100
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
100,
95
],
[
100,
95
]
],
"legs": [
[
-88,
-82
],
[
-50,
-120
]
]
}
},
{
"name": "Single-Cone Sprint Drill",
"view": "side",
"prompt": "standing next to a single cone, one arm forward and one back",
"start": {
"torso": 80,
"arms": [
[
-40,
50
],
[
-130,
-80
]
],
"legs": [
[
-75,
-100
],
[
-100,
-120
]
]
},
"end": {
"torso": 80,
"arms": [
[
-130,
-80
],
[
-40,
50
]
],
"legs": [
[
-100,
-120
],
[
-75,
-100
]
]
}
},
{
"name": "Single-Leg High Box Squat",
"view": "side",
"prompt": "standing on one leg on a high box, holding a band overhead",
"start": {
"torso": 90,
"arms": [
[
75,
85
],
[
75,
85
]
],
"legs": [
[
-80,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 65,
"arms": [
[
85,
92
],
[
85,
92
]
],
"legs": [
[
-80,
-80
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Single-Leg Hop Progression",
"view": "side",
"prompt": "balancing on one leg in front of a line of cones",
"start": {
"torso": 85,
"arms": [
[
-120,
-60
],
[
-50,
40
]
],
"legs": [
[
-10,
-95
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
-40,
50
],
[
-130,
-70
]
],
"legs": [
[
-10,
-95
],
[
-115,
-105
]
]
}
},
{
"name": "Single-Leg Lateral Hop",
"view": "front",
"prompt": "balancing on one leg beside a small cone",
"start": {
"torso": 90,
"arms": [
[
-110,
-100
],
[
-70,
-80
]
],
"legs": [
[
-95,
-85
],
[
-75,
-125
]
]
},
"end": {
"torso": 80,
"arms": [
[
-140,
-150
],
[
-40,
-30
]
],
"legs": [
[
-75,
-60
],
[
-60,
-110
]
]
}
},
{
"name": "Single-Leg Leg Extension",
"view": "side",
"prompt": "seated in a leg extension machine, pad on one lower shin",
"start": {
"torso": 90,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
-20
],
[
-80,
-20
]
],
"legs": [
[
0,
5
],
[
0,
-90
]
]
}
},
{
"name": "Single-Leg Soft-Landing Holds",
"view": "side",
"prompt": "standing on one leg on the floor",
"start": {
"torso": 90,
"arms": [
[
-100,
-95
],
[
-100,
-95
]
],
"legs": [
[
-5,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
-30,
-20
],
[
-30,
-20
]
],
"legs": [
[
-100,
-160
],
[
-45,
-115
]
],
"head": 80
}
},
{
"name": "Single-Leg Stride Jump",
"view": "side",
"prompt": "standing beside a box, one foot on top of it",
"start": {
"torso": 85,
"arms": [
[
-115,
-100
],
[
-115,
-100
]
],
"legs": [
[
-20,
-100
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
100,
95
],
[
100,
95
]
],
"legs": [
[
-88,
-82
],
[
-30,
-110
]
]
}
},
{
"name": "Sissy Squat",
"view": "side",
"prompt": "standing, one hand holding a support in front",
"start": {
"torso": 90,
"arms": [
[
-30,
-20
],
[
-40,
-30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 120,
"arms": [
[
-5,
0
],
[
-15,
-10
]
],
"legs": [
[
-60,
-150
],
[
-60,
-150
]
],
"head": 110
}
},
{
"name": "Sit Squats",
"view": "side",
"prompt": "standing on the floor, feet shoulder width",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 55,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 75
}
},
{
"name": "Skating",
"view": "side",
"prompt": "roller skating on a smooth surface",
"start": {
"torso": 55,
"arms": [
[
-40,
-20
],
[
-140,
-160
]
],
"legs": [
[
-50,
-110
],
[
-130,
-130
]
],
"head": 70
},
"end": {
"torso": 55,
"arms": [
[
-140,
-160
],
[
-40,
-20
]
],
"legs": [
[
-130,
-130
],
[
-50,
-110
]
],
"head": 70
}
},
{
"name": "Sled Drag - Harness",
"view": "side",
"prompt": "wearing a harness attached by a strap to a loaded sled behind",
"start": {
"torso": 65,
"arms": [
[
-130,
-60
],
[
-40,
40
]
],
"legs": [
[
-50,
-110
],
[
-125,
-125
]
],
"head": 75
},
"end": {
"torso": 65,
"arms": [
[
-40,
40
],
[
-130,
-60
]
],
"legs": [
[
-125,
-125
],
[
-50,
-110
]
],
"head": 75
}
},
{
"name": "Sled Push",
"view": "side",
"prompt": "leaning into a loaded sled, hands on its upright handles",
"start": {
"torso": 35,
"arms": [
[
-5,
-5
],
[
-5,
-5
]
],
"legs": [
[
-40,
-110
],
[
-140,
-140
]
],
"head": 50
},
"end": {
"torso": 35,
"arms": [
[
-5,
-5
],
[
-5,
-5
]
],
"legs": [
[
-140,
-140
],
[
-40,
-110
]
],
"head": 50
}
},
{
"name": "Slow Squat",
"view": "side",
"prompt": "standing on the floor, arms straight out in front",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 82
}
},
{
"name": "Smith Machine Leg Press",
"view": "side",
"prompt": "lying face up under a Smith machine bar, feet on the bar",
"start": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
120,
45
],
[
120,
45
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
88,
88
],
[
88,
88
]
],
"head": 180
}
},
{
"name": "Smith Machine Pistol Squat",
"view": "side",
"prompt": "standing on one leg in a Smith machine, bar across upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-60,
-60
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
3,
3
],
[
0,
-110
]
],
"head": 78
}
},
{
"name": "Smith Single-Leg Split Squat",
"view": "side",
"prompt": "in a Smith machine, bar on upper back, rear foot on a bench",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-75,
-95
],
[
-120,
-175
]
]
},
"end": {
"torso": 82,
"arms": [
[
-118,
67
],
[
-118,
67
]
],
"legs": [
[
-10,
-88
],
[
-105,
130
]
],
"head": 88
}
},
{
"name": "Snatch",
"view": "side",
"prompt": "barbell on the floor in front of the shins, wide grip",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-105
],
[
-15,
-105
]
],
"head": 45
},
"end": {
"torso": 75,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Snatch Balance",
"view": "side",
"prompt": "standing, barbell across upper back with a wide grip",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Snatch from Blocks",
"view": "side",
"prompt": "barbell resting on blocks at knee height, wide grip",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-100
],
[
-60,
-100
]
],
"head": 60
},
"end": {
"torso": 75,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Speed Box Squat",
"view": "side",
"prompt": "standing in a rack, banded barbell on upper back, box behind",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Split Clean",
"view": "side",
"prompt": "barbell on the floor in front of the shins, overhand grip",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-105
],
[
-15,
-105
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-45,
-95
],
[
-120,
-130
]
]
}
},
{
"name": "Split Jerk",
"view": "side",
"prompt": "standing, barbell racked on the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
92,
92
],
[
92,
92
]
],
"legs": [
[
-45,
-95
],
[
-120,
-130
]
]
}
},
{
"name": "Split Jump",
"view": "side",
"prompt": "on the floor in a deep lunge stance",
"start": {
"torso": 90,
"arms": [
[
-60,
-30
],
[
-120,
-90
]
],
"legs": [
[
0,
-90
],
[
-100,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
80,
90
],
[
80,
90
]
],
"legs": [
[
-70,
-90
],
[
-110,
-100
]
]
}
},
{
"name": "Split Squat with Dumbbells",
"view": "side",
"prompt": "rear foot on a bench behind, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-75,
-95
],
[
-120,
-175
]
]
},
"end": {
"torso": 82,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-10,
-88
],
[
-105,
130
]
],
"head": 88
}
},
{
"name": "Squat Jerk",
"view": "side",
"prompt": "standing, barbell racked on the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
92,
92
],
[
92,
92
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 88
}
},
{
"name": "Squat with Chains",
"view": "side",
"prompt": "standing, barbell with chains across the upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Squat with Plate Movers",
"view": "side",
"prompt": "standing in a rack, barbell across upper back, plate on floor",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 62,
"arms": [
[
-138,
47
],
[
-138,
47
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 80
}
},
{
"name": "Squats on Multipress",
"view": "side",
"prompt": "standing in a Smith machine, bar across upper back",
"start": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 68,
"arms": [
[
-132,
53
],
[
-132,
53
]
],
"legs": [
[
0,
-105
],
[
0,
-105
]
],
"head": 82
}
},
{
"name": "Stairmaster",
"view": "side",
"prompt": "stepping on a stair climber machine, hands on the side rails",
"start": {
"torso": 85,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
-45,
-100
],
[
-95,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-60,
-10
],
[
-60,
-10
]
],
"legs": [
[
-95,
-90
],
[
-45,
-100
]
]
}
},
{
"name": "Standing Elevated Quad Stretch",
"view": "side",
"prompt": "standing with back to a low bench, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-115,
-50
],
[
-115,
-50
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-115,
-50
],
[
-115,
-50
]
],
"legs": [
[
-110,
175
],
[
-85,
-95
]
]
}
},
{
"name": "Standing Hip Flexors",
"view": "side",
"prompt": "standing in a staggered stance, hands on hips",
"start": {
"torso": 90,
"arms": [
[
-115,
-50
],
[
-115,
-50
]
],
"legs": [
[
-80,
-90
],
[
-100,
-100
]
]
},
"end": {
"torso": 95,
"arms": [
[
-110,
-45
],
[
-110,
-45
]
],
"legs": [
[
-60,
-100
],
[
-100,
-140
]
]
}
},
{
"name": "Standing IT Band Stretch",
"view": "front",
"prompt": "standing with legs crossed, arms overhead",
"start": {
"torso": 90,
"arms": [
[
-95,
-95
],
[
-85,
-85
]
],
"legs": [
[
-80,
-75
],
[
-100,
-105
]
]
},
"end": {
"torso": 110,
"arms": [
[
130,
140
],
[
115,
125
]
],
"legs": [
[
-80,
-75
],
[
-100,
-105
]
],
"head": 120
}
},
{
"name": "Standing Long Jump",
"view": "side",
"prompt": "standing in a partial squat on sand",
"start": {
"torso": 60,
"arms": [
[
-130,
-120
],
[
-130,
-120
]
],
"legs": [
[
-45,
-110
],
[
-45,
-110
]
],
"head": 75
},
"end": {
"torso": 70,
"arms": [
[
0,
10
],
[
0,
10
]
],
"legs": [
[
-10,
-80
],
[
-10,
-80
]
],
"head": 80
}
},
{
"name": "Standing Quad Stretch",
"view": "side",
"prompt": "standing on one leg on the floor",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-100
],
[
-115,
-50
]
],
"legs": [
[
-90,
100
],
[
-90,
-90
]
]
}
},
{
"name": "Star Jump",
"view": "front",
"prompt": "standing on the floor, arms close to the body",
"start": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-85,
-90
]
],
"legs": [
[
-100,
-80
],
[
-80,
-100
]
]
},
"end": {
"torso": 90,
"arms": [
[
135,
135
],
[
45,
45
]
],
"legs": [
[
-120,
-120
],
[
-60,
-60
]
]
}
},
{
"name": "Step Jack",
"view": "front",
"prompt": "standing upright, arms at sides",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
100,
95
],
[
80,
85
]
],
"legs": [
[
-112,
-110
],
[
-88,
-90
]
]
}
},
{
"name": "Step Mill",
"view": "side",
"prompt": "standing on a stair-climber machine, hands on the side rails",
"start": {
"torso": 85,
"arms": [
[
-70,
0
],
[
-70,
0
]
],
"legs": [
[
-30,
-100
],
[
-92,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
-70,
0
],
[
-70,
0
]
],
"legs": [
[
-75,
-95
],
[
-60,
-112
]
]
}
},
{
"name": "Stride Jump Crossover",
"view": "side",
"prompt": "one foot on top of a low box, other foot on the floor",
"start": {
"torso": 90,
"arms": [
[
-120,
-100
],
[
-60,
-70
]
],
"legs": [
[
-20,
-95
],
[
-90,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
80,
85
],
[
80,
85
]
],
"legs": [
[
-90,
-90
],
[
-20,
-95
]
]
}
},
{
"name": "Suspended Split Squat",
"view": "side",
"prompt": "standing, rear foot in a suspension strap handle behind",
"start": {
"torso": 90,
"arms": [
[
-70,
10
],
[
-70,
10
]
],
"legs": [
[
-85,
-90
],
[
-120,
-170
]
]
},
"end": {
"torso": 90,
"arms": [
[
-70,
10
],
[
-70,
10
]
],
"legs": [
[
-30,
-95
],
[
-110,
155
]
]
}
},
{
"name": "Tire Flip",
"view": "side",
"prompt": "crouching behind a large tractor tire, hands under its edge",
"start": {
"torso": 20,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-10,
-110
],
[
-10,
-110
]
],
"head": 35
},
"end": {
"torso": 85,
"arms": [
[
20,
40
],
[
20,
40
]
],
"legs": [
[
-80,
-95
],
[
-100,
-90
]
]
}
},
{
"name": "Trail Running/Walking",
"view": "side",
"prompt": "running outdoors on a dirt trail",
"start": {
"torso": 82,
"arms": [
[
-130,
-60
],
[
-50,
30
]
],
"legs": [
[
-30,
-110
],
[
-105,
-100
]
]
},
"end": {
"torso": 82,
"arms": [
[
-95,
-40
],
[
-85,
-30
]
],
"legs": [
[
-90,
-92
],
[
-20,
-135
]
]
}
},
{
"name": "Trap Bar Deadlift",
"view": "side",
"prompt": "standing inside a hexagonal trap bar on the floor, gripping the handles",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-15,
-110
],
[
-15,
-110
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Walking, Treadmill",
"view": "side",
"prompt": "walking on a treadmill",
"start": {
"torso": 88,
"arms": [
[
-110,
-95
],
[
-70,
-60
]
],
"legs": [
[
-70,
-95
],
[
-110,
-100
]
]
},
"end": {
"torso": 88,
"arms": [
[
-92,
-88
],
[
-88,
-85
]
],
"legs": [
[
-90,
-90
],
[
-80,
-125
]
]
}
},
{
"name": "Weighted Jump Squat",
"view": "side",
"prompt": "barbell across the upper back, hands gripping it",
"start": {
"torso": 65,
"arms": [
[
-120,
70
],
[
-120,
70
]
],
"legs": [
[
-25,
-110
],
[
-25,
-110
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
-110,
75
],
[
-110,
75
]
],
"legs": [
[
-90,
-95
],
[
-90,
-95
]
]
}
},
{
"name": "Weighted Sissy Squat",
"view": "side",
"prompt": "standing on toes, one hand on a rack, weight plate held on chest",
"start": {
"torso": 90,
"arms": [
[
-100,
60
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 130,
"arms": [
[
-130,
50
],
[
25,
15
]
],
"legs": [
[
-50,
-130
],
[
-50,
-130
]
],
"head": 110
}
},
{
"name": "Weighted Squat",
"view": "side",
"prompt": "standing on two benches, weight belt with plate hanging between legs",
"start": {
"torso": 90,
"arms": [
[
-60,
0
],
[
-60,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-30,
0
],
[
-30,
0
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Yoke Walk",
"view": "side",
"prompt": "heavy yoke frame across upper back, gripping its uprights",
"start": {
"torso": 88,
"arms": [
[
-100,
80
],
[
-100,
80
]
],
"legs": [
[
-70,
-95
],
[
-110,
-100
]
]
},
"end": {
"torso": 88,
"arms": [
[
-100,
80
],
[
-100,
80
]
],
"legs": [
[
-90,
-90
],
[
-80,
-125
]
]
}
},
{
"name": "Zercher Squats",
"view": "side",
"prompt": "barbell held in the crook of the elbows in front",
"start": {
"torso": 90,
"arms": [
[
-60,
95
],
[
-60,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 75,
"arms": [
[
-45,
95
],
[
-45,
95
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Alternating Cable Shoulder Press",
"view": "side",
"prompt": "standing between cable towers, handles at shoulder height",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Alternating Deltoid Raise",
"view": "side",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-85,
-80
],
[
-85,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
5
],
[
-85,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Alternating Kettlebell Press",
"view": "side",
"prompt": "standing, a kettlebell held at each shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Anti-Gravity Press",
"view": "side",
"prompt": "lying face down on an incline bench, holding a barbell",
"start": {
"torso": 40,
"arms": [
[
-30,
100
],
[
-30,
100
]
],
"legs": [
[
-150,
-120
],
[
-150,
-120
]
],
"head": 40,
"face": "down"
},
"end": {
"torso": 40,
"arms": [
[
40,
40
],
[
40,
40
]
],
"legs": [
[
-150,
-120
],
[
-150,
-120
]
],
"head": 40,
"face": "down"
}
},
{
"name": "Arm Circles",
"view": "front",
"prompt": "standing, arms straight out to the sides",
"start": {
"torso": 90,
"arms": [
[
195,
195
],
[
-15,
-15
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
165,
165
],
[
15,
15
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Arnold Press",
"view": "side",
"prompt": "seated on a bench, a dumbbell in each hand at chin height",
"start": {
"torso": 90,
"arms": [
[
-60,
90
],
[
-60,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Backward Medicine Ball Throw",
"view": "side",
"prompt": "standing, holding a medicine ball",
"start": {
"torso": 40,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-30,
-105
],
[
-30,
-105
]
],
"head": 55
},
"end": {
"torso": 100,
"arms": [
[
115,
125
],
[
115,
125
]
],
"legs": [
[
-85,
-80
],
[
-85,
-80
]
],
"head": 110
}
},
{
"name": "Band Pull-Apart",
"view": "front",
"prompt": "standing, holding a resistance band in front of the chest",
"start": {
"torso": 90,
"arms": [
[
-75,
-60
],
[
-105,
-120
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
178
],
[
0,
2
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Barbell Clean and press",
"view": "side",
"prompt": "standing over a barbell on the floor",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-25,
-105
],
[
-25,
-105
]
],
"head": 40
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Barbell Incline Shoulder Raise",
"view": "side",
"prompt": "lying on an incline bench, barbell held straight above",
"start": {
"torso": 135,
"arms": [
[
62,
62
],
[
62,
62
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
]
},
"end": {
"torso": 135,
"arms": [
[
48,
48
],
[
48,
48
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
]
}
},
{
"name": "Barbell Overhead Press",
"view": "side",
"prompt": "standing, holding a barbell at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Barbell Rear Delt Row",
"view": "side",
"prompt": "bent forward at the hips, holding a barbell with a wide grip",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
120,
-90
],
[
120,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Barbell Shrug Behind The Back",
"view": "side",
"prompt": "standing, holding a barbell behind the back",
"start": {
"torso": 90,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-85
],
[
-95,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Battling Ropes",
"view": "side",
"prompt": "half squat, gripping the ends of two heavy ropes",
"start": {
"torso": 70,
"arms": [
[
-20,
30
],
[
-20,
30
]
],
"legs": [
[
-45,
-110
],
[
-45,
-110
]
],
"head": 85
},
"end": {
"torso": 70,
"arms": [
[
-70,
-40
],
[
-70,
-40
]
],
"legs": [
[
-45,
-110
],
[
-45,
-110
]
],
"head": 85
}
},
{
"name": "Bent High Pulls",
"view": "side",
"prompt": "bent slightly forward, a dumbbell in each hand",
"start": {
"torso": 50,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 60
},
"end": {
"torso": 50,
"arms": [
[
140,
-60
],
[
140,
-60
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 60
}
},
{
"name": "Bent Over Dumbbell Rear Delt Raise With Head On Bench",
"view": "side",
"prompt": "bent over, forehead resting on an incline bench, dumbbells in hands",
"start": {
"torso": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 5
},
"end": {
"torso": 15,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 5
}
},
{
"name": "Bent Over Low-Pulley Side Lateral",
"view": "side",
"prompt": "bent over at the waist, holding a low cable handle",
"start": {
"torso": 10,
"arms": [
[
-100,
-100
],
[
-80,
0
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 20
},
"end": {
"torso": 10,
"arms": [
[
-5,
-10
],
[
-80,
0
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 20
}
},
{
"name": "Bent-over Lateral Raises",
"view": "side",
"prompt": "seated on the end of a bench, leaning forward, dumbbells in hands",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 35
},
"end": {
"torso": 30,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 35
}
},
{
"name": "Bradford/Rocky Presses",
"view": "side",
"prompt": "seated on a bench, holding a barbell at shoulder level",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-130,
80
],
[
-130,
80
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Bus Drivers",
"view": "front",
"prompt": "seated, holding a weight plate in front like a steering wheel",
"start": {
"torso": 90,
"arms": [
[
-70,
90
],
[
-110,
90
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-55,
98
],
[
-90,
121
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
}
},
{
"name": "Cable Front Raise with a small bar",
"view": "side",
"prompt": "standing facing away from a low cable, holding a short bar",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
5,
5
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Lateral Raise",
"view": "front",
"prompt": "standing beside a low cable pulley, holding the handle",
"start": {
"torso": 90,
"arms": [
[
-75,
-70
],
[
-105,
-110
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
178
],
[
0,
2
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Cable Rear Delt Fly",
"view": "front",
"prompt": "standing between two high cable pulleys, arms crossed in front",
"start": {
"torso": 90,
"arms": [
[
-20,
0
],
[
200,
180
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
175,
175
],
[
5,
5
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Cable Rope Rear-Delt Rows",
"view": "side",
"prompt": "seated at a low cable row station, holding a rope",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
10,
-10
],
[
10,
-10
]
]
},
"end": {
"torso": 90,
"arms": [
[
165,
40
],
[
165,
40
]
],
"legs": [
[
10,
-10
],
[
10,
-10
]
]
}
},
{
"name": "Cable Seated Lateral Raise",
"view": "front",
"prompt": "seated on a bench between two low cable pulleys",
"start": {
"torso": 90,
"arms": [
[
-75,
-70
],
[
-105,
-110
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
178
],
[
0,
2
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
}
},
{
"name": "Cable Shoulder Press",
"view": "side",
"prompt": "standing between cable towers, handles at shoulder height",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Cable Shrugs",
"view": "side",
"prompt": "standing, holding a cable bar from a low pulley",
"start": {
"torso": 90,
"arms": [
[
-88,
-88
],
[
-88,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-82,
-78
],
[
-82,
-78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Calf-Machine Shoulder Shrug",
"view": "side",
"prompt": "standing in a calf machine, shoulder pads on the shoulders",
"start": {
"torso": 90,
"arms": [
[
-60,
95
],
[
-60,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-55,
100
],
[
-55,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Car Drivers",
"view": "front",
"prompt": "standing, holding a weight plate out in front at shoulder height",
"start": {
"torso": 90,
"arms": [
[
-70,
90
],
[
-110,
90
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-55,
98
],
[
-90,
121
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Chair Upper Body Stretch",
"view": "side",
"prompt": "seated on the edge of a chair, hands gripping its back",
"start": {
"torso": 90,
"arms": [
[
-130,
-130
],
[
-130,
-130
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 70,
"arms": [
[
-150,
-150
],
[
-150,
-150
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 80
}
},
{
"name": "Circus Bell",
"view": "side",
"prompt": "standing, holding a thick-handled oversized dumbbell at the shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-80,
60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-80,
60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Claps Over the Head",
"view": "front",
"prompt": "standing, feet shoulder width apart",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
65,
75
],
[
115,
105
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Clean Shrug",
"view": "side",
"prompt": "standing, holding a barbell at mid thigh",
"start": {
"torso": 60,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-100
],
[
-80,
-100
]
],
"head": 70
},
"end": {
"torso": 100,
"arms": [
[
-95,
-95
],
[
-95,
-95
]
],
"legs": [
[
-88,
-78
],
[
-88,
-78
]
],
"head": 95
}
},
{
"name": "Clean and Jerk",
"view": "side",
"prompt": "standing over a barbell on the floor",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-55,
-90
],
[
-115,
-130
]
]
}
},
{
"name": "Clean and Press",
"view": "side",
"prompt": "standing over a barbell on the floor",
"start": {
"torso": 25,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-105
],
[
-20,
-105
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Crucifix",
"view": "front",
"prompt": "standing, holding a weight in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
178
],
[
0,
2
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Cuban Press",
"view": "front",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
180,
-90
],
[
0,
-90
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
92
],
[
85,
88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Devil’s Press",
"view": "side",
"prompt": "two dumbbells on the floor, hands on them",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Double Kettlebell Jerk",
"view": "side",
"prompt": "standing, a kettlebell held at each shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 88,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-55,
-115
],
[
-55,
-115
]
]
}
},
{
"name": "Double Kettlebell Push Press",
"view": "side",
"prompt": "standing, a kettlebell held at each shoulder",
"start": {
"torso": 88,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Double Kettlebell Snatch",
"view": "side",
"prompt": "a kettlebell in each hand",
"start": {
"torso": 30,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Dumbbell Bent Over Face Pull",
"view": "side",
"prompt": "bent forward at the hips, a dumbbell in each hand",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 40
},
"end": {
"torso": 30,
"arms": [
[
150,
20
],
[
150,
20
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 40
}
},
{
"name": "Dumbbell Incline Shoulder Raise",
"view": "side",
"prompt": "lying on an incline bench, dumbbells held straight above",
"start": {
"torso": 135,
"arms": [
[
62,
62
],
[
62,
62
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
]
},
"end": {
"torso": 135,
"arms": [
[
48,
48
],
[
48,
48
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
]
}
},
{
"name": "Dumbbell Raise",
"view": "front",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
178,
172
],
[
2,
8
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Dumbbell Rear Delt Row",
"view": "side",
"prompt": "bent forward at the hips, a dumbbell in each hand",
"start": {
"torso": 15,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 25
},
"end": {
"torso": 15,
"arms": [
[
110,
-90
],
[
110,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 25
}
},
{
"name": "Dumbbell Scaption",
"view": "side",
"prompt": "standing, a light dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
5,
5
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Elbow Circles",
"view": "front",
"prompt": "standing, fingertips on the shoulders, elbows out",
"start": {
"torso": 90,
"arms": [
[
200,
23
],
[
-20,
157
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-24
],
[
30,
-156
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "External Rotation",
"view": "side",
"prompt": "lying on one side on a flat bench, dumbbell in the top hand",
"start": {
"torso": 180,
"arms": [
[
-5,
-90
],
[
165,
-10
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
-5,
90
],
[
165,
-10
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "External Rotation Stretch",
"view": "side",
"prompt": "seated on the floor, elbow resting on a raised knee",
"start": {
"torso": 90,
"arms": [
[
-36,
90
],
[
-80,
-60
]
],
"legs": [
[
60,
-60
],
[
0,
-10
]
]
},
"end": {
"torso": 80,
"arms": [
[
-50,
25
],
[
-80,
-60
]
],
"legs": [
[
60,
-60
],
[
0,
-10
]
],
"head": 75
}
},
{
"name": "External Rotation with Band",
"view": "front",
"prompt": "standing side-on to a post, elbow at side holding a band",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
180
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "External Rotation with Cable",
"view": "front",
"prompt": "standing side-on to a cable at elbow height, holding the handle",
"start": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
180
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Face Pull",
"view": "side",
"prompt": "standing facing a high cable pulley, holding a rope",
"start": {
"torso": 88,
"arms": [
[
15,
15
],
[
15,
15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
160,
45
],
[
160,
45
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Front Incline Dumbbell Raise",
"view": "side",
"prompt": "seated on an incline bench, a dumbbell in each hand",
"start": {
"torso": 125,
"arms": [
[
-80,
-70
],
[
-80,
-70
]
],
"legs": [
[
5,
-90
],
[
5,
-90
]
]
},
"end": {
"torso": 125,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
5,
-90
],
[
5,
-90
]
]
}
},
{
"name": "Front Plate Raise",
"view": "side",
"prompt": "standing, holding a weight plate in both hands",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
10,
10
],
[
10,
10
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Front Two-Dumbbell Raise",
"view": "side",
"prompt": "standing, dumbbells in front of the thighs",
"start": {
"torso": 90,
"arms": [
[
-82,
-80
],
[
-82,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Handstand Pushup",
"view": "side",
"prompt": "upside down in a handstand, hands on the floor, heels against a wall",
"start": {
"torso": -90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
90,
90
],
[
90,
90
]
],
"head": -90
},
"end": {
"torso": -90,
"arms": [
[
180,
-80
],
[
180,
-80
]
],
"legs": [
[
90,
90
],
[
90,
90
]
],
"head": -90
}
},
{
"name": "Head Turns",
"view": "side",
"prompt": "standing upright, shoulders relaxed",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 92,
"face": "left"
}
},
{
"name": "Hindu Pushups",
"view": "side",
"prompt": "hands and feet on the floor, hips raised high",
"start": {
"torso": -30,
"arms": [
[
-45,
-45
],
[
-45,
-45
]
],
"legs": [
[
-130,
-130
],
[
-130,
-130
]
],
"head": -40,
"face": "down"
},
"end": {
"torso": 55,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
185,
185
],
[
185,
185
]
],
"head": 70
}
},
{
"name": "Incline Bench Reverse Fly",
"view": "side",
"prompt": "lying face down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 40,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-120,
-100
],
[
-120,
-100
]
],
"head": 40,
"face": "down"
},
"end": {
"torso": 40,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
-120,
-100
],
[
-120,
-100
]
],
"head": 40,
"face": "down"
}
},
{
"name": "Incline Row - Rings",
"view": "side",
"prompt": "leaning back, heels on the floor, hands gripping gymnastic rings above",
"start": {
"torso": 135,
"arms": [
[
60,
60
],
[
60,
60
]
],
"legs": [
[
-45,
-45
],
[
-45,
-45
]
]
},
"end": {
"torso": 120,
"arms": [
[
140,
19
],
[
140,
19
]
],
"legs": [
[
-60,
-60
],
[
-60,
-60
]
]
}
},
{
"name": "Internal Rotation with Band",
"view": "front",
"prompt": "standing side-on to a post, elbow at side holding a band",
"start": {
"torso": 90,
"arms": [
[
-90,
180
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-90,
0
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Jerk Balance",
"view": "side",
"prompt": "barbell racked on the shoulders, feet slightly split",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-80,
-90
],
[
-102,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-55,
-90
],
[
-115,
-130
]
]
}
},
{
"name": "Kettlebell Arnold Press",
"view": "side",
"prompt": "standing, a kettlebell held at one shoulder",
"start": {
"torso": 90,
"arms": [
[
-60,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Pirate Ships",
"view": "front",
"prompt": "wide stance, holding one kettlebell with both hands",
"start": {
"torso": 90,
"arms": [
[
-70,
-70
],
[
-110,
-110
]
],
"legs": [
[
-110,
-100
],
[
-70,
-80
]
]
},
"end": {
"torso": 90,
"arms": [
[
5,
2
],
[
-45,
100
]
],
"legs": [
[
-110,
-100
],
[
-70,
-80
]
]
}
},
{
"name": "Kettlebell Seated Press",
"view": "side",
"prompt": "seated on the floor, legs spread, a kettlebell at the shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-90,
-90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
0,
0
],
[
0,
0
]
]
}
},
{
"name": "Kettlebell Seesaw Press",
"view": "side",
"prompt": "standing, a kettlebell in each hand at the shoulders",
"start": {
"torso": 90,
"arms": [
[
88,
90
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-70,
100
],
[
40,
80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Sumo High Pull",
"view": "front",
"prompt": "wide stance, a kettlebell on the floor between the feet",
"start": {
"torso": 90,
"arms": [
[
-75,
-80
],
[
-105,
-100
]
],
"legs": [
[
-160,
-100
],
[
-20,
-80
]
]
},
"end": {
"torso": 90,
"arms": [
[
120,
-15
],
[
60,
-165
]
],
"legs": [
[
-105,
-95
],
[
-75,
-85
]
]
}
},
{
"name": "Kettlebell Thruster",
"view": "side",
"prompt": "a kettlebell held at each shoulder",
"start": {
"torso": 70,
"arms": [
[
-40,
100
],
[
-40,
100
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Turkish Get-Up",
"view": "side",
"prompt": "lying on the back, kettlebell pressed up in one hand",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
-5,
0
]
],
"legs": [
[
50,
-50
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Kettlebell Turkish Get-Up (Lunge style)",
"view": "side",
"prompt": "lying on the back, kettlebell pressed up in one hand",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
-5,
0
]
],
"legs": [
[
50,
-50
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-5,
-90
],
[
-90,
180
]
]
}
},
{
"name": "Kettlebell Turkish Get-Up (Squat style)",
"view": "side",
"prompt": "lying on the back, kettlebell pressed up in one hand",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
-5,
0
]
],
"legs": [
[
50,
-50
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 75,
"arms": [
[
95,
95
],
[
-60,
-30
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 85
}
},
{
"name": "Kneeling Arm Drill",
"view": "side",
"prompt": "half-kneeling, one foot forward, back knee on the floor",
"start": {
"torso": 90,
"arms": [
[
-30,
70
],
[
-150,
-70
]
],
"legs": [
[
-5,
-90
],
[
-90,
180
]
]
},
"end": {
"torso": 90,
"arms": [
[
-130,
-50
],
[
-55,
50
]
],
"legs": [
[
-5,
-90
],
[
-90,
180
]
]
}
},
{
"name": "Landmine Linear Jammer",
"view": "side",
"prompt": "athletic stance, holding the end of a landmine barbell at the shoulders",
"start": {
"torso": 70,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
],
"head": 85
},
"end": {
"torso": 80,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
-85,
-80
],
[
-85,
-80
]
],
"head": 85
}
},
{
"name": "Lateral Raise - With Bands",
"view": "front",
"prompt": "standing on a resistance band, a handle in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
178
],
[
0,
2
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Lateral Rows on Cable, One Armed",
"view": "front",
"prompt": "standing beside a waist-height cable, holding the handle",
"start": {
"torso": 90,
"arms": [
[
-60,
0
],
[
-80,
-95
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-120,
180
],
[
-80,
-95
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Leverage Shoulder Press",
"view": "side",
"prompt": "seated in a shoulder press machine, gripping the handles",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
80,
85
],
[
80,
85
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Leverage Shrug",
"view": "side",
"prompt": "standing between machine handles, gripping them",
"start": {
"torso": 90,
"arms": [
[
-88,
-88
],
[
-88,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-82,
-78
],
[
-82,
-78
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Log Lift",
"view": "side",
"prompt": "holding a heavy log at the chest by its handles",
"start": {
"torso": 85,
"arms": [
[
-30,
120
],
[
-30,
120
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Low Pulley Row To Neck",
"view": "side",
"prompt": "seated at a low cable row station, holding a rope",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
10,
-10
],
[
10,
-10
]
]
},
"end": {
"torso": 90,
"arms": [
[
160,
40
],
[
160,
40
]
],
"legs": [
[
10,
-10
],
[
10,
-10
]
]
}
},
{
"name": "Lying One-Arm Lateral Raise",
"view": "side",
"prompt": "lying face down on a flat bench, a dumbbell in one hand",
"start": {
"torso": 0,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
-5,
-10
],
[
-90,
-90
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Lying Rear Delt Raise",
"view": "side",
"prompt": "lying face down on a flat bench, a dumbbell in each hand",
"start": {
"torso": 0,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
},
"end": {
"torso": 0,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
180,
180
],
[
180,
180
]
],
"head": 0,
"face": "down"
}
},
{
"name": "Machine Lateral Raise",
"view": "front",
"prompt": "seated in a lateral raise machine, arms against the pads",
"start": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
160
],
[
0,
20
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
}
},
{
"name": "Machine Shoulder (Military) Press",
"view": "side",
"prompt": "seated in a shoulder press machine, gripping the handles",
"start": {
"torso": 90,
"arms": [
[
-60,
100
],
[
-60,
100
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
80,
85
],
[
80,
85
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Machine Side Lateral Raises",
"view": "front",
"prompt": "seated in a lateral raise machine, arms against the pads",
"start": {
"torso": 90,
"arms": [
[
-100,
-80
],
[
-80,
-100
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
160
],
[
0,
20
]
],
"legs": [
[
-125,
-90
],
[
-55,
-90
]
]
}
},
{
"name": "Medicine Ball Scoop Throw",
"view": "side",
"prompt": "half squat, holding a medicine ball near the feet",
"start": {
"torso": 45,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-30,
-105
],
[
-30,
-105
]
],
"head": 60
},
"end": {
"torso": 100,
"arms": [
[
110,
120
],
[
110,
120
]
],
"legs": [
[
-88,
-78
],
[
-88,
-78
]
],
"head": 105
}
},
{
"name": "Military Press mit SZ-Bar",
"view": "side",
"prompt": "standing, holding an EZ curl bar at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Incline Lateral Raise",
"view": "front",
"prompt": "lying on one side against an incline bench, dumbbell in top hand",
"start": {
"torso": 120,
"arms": [
[
-150,
-150
],
[
-80,
-80
]
],
"legs": [
[
-80,
-90
],
[
-70,
-85
]
]
},
"end": {
"torso": 120,
"arms": [
[
-150,
-150
],
[
60,
60
]
],
"legs": [
[
-80,
-90
],
[
-70,
-85
]
]
}
},
{
"name": "One-Arm Kettlebell Clean and Jerk",
"view": "side",
"prompt": "a kettlebell in one hand",
"start": {
"torso": 30,
"arms": [
[
-110,
-110
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Jerk",
"view": "side",
"prompt": "standing, a kettlebell held at one shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 88,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
}
},
{
"name": "One-Arm Kettlebell Military Press To The Side",
"view": "front",
"prompt": "standing, a kettlebell held at one shoulder",
"start": {
"torso": 90,
"arms": [
[
-80,
80
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
115,
108
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Para Press",
"view": "front",
"prompt": "standing, a kettlebell beside the shoulder, elbow out to the side",
"start": {
"torso": 90,
"arms": [
[
200,
90
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
100,
95
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Push Press",
"view": "side",
"prompt": "standing, a kettlebell held at one shoulder",
"start": {
"torso": 88,
"arms": [
[
-70,
100
],
[
-90,
-90
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Snatch",
"view": "side",
"prompt": "a kettlebell in one hand",
"start": {
"torso": 30,
"arms": [
[
-110,
-110
],
[
-90,
-90
]
],
"legs": [
[
-70,
-100
],
[
-70,
-100
]
],
"head": 45
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "One-Arm Kettlebell Split Jerk",
"view": "side",
"prompt": "standing, a kettlebell held at one shoulder",
"start": {
"torso": 90,
"arms": [
[
-70,
100
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-60,
-40
]
],
"legs": [
[
-55,
-90
],
[
-115,
-130
]
]
}
},
{
"name": "One-Arm Kettlebell Split Snatch",
"view": "side",
"prompt": "a kettlebell in one hand",
"start": {
"torso": 40,
"arms": [
[
-95,
-95
],
[
-90,
-90
]
],
"legs": [
[
-35,
-105
],
[
-35,
-105
]
],
"head": 55
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-60,
-40
]
],
"legs": [
[
-55,
-90
],
[
-115,
-130
]
]
}
},
{
"name": "One-Arm Side Laterals",
"view": "front",
"prompt": "leaning sideways, one hand holding a bench, dumbbell in the other",
"start": {
"torso": 80,
"arms": [
[
-80,
-80
],
[
-20,
-20
]
],
"legs": [
[
-92,
-90
],
[
-85,
-88
]
]
},
"end": {
"torso": 80,
"arms": [
[
170,
170
],
[
-20,
-20
]
],
"legs": [
[
-92,
-90
],
[
-85,
-88
]
]
}
},
{
"name": "Plank Shoulder Taps",
"view": "side",
"prompt": "straight-arm plank on the floor",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
},
"end": {
"torso": 23,
"arms": [
[
-60,
118
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
],
"face": "down"
}
},
{
"name": "Power Partials",
"view": "front",
"prompt": "standing, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
175,
175
],
[
5,
5
]
],
"legs": [
[
-92,
-90
],
[
-88,
-90
]
]
}
},
{
"name": "Push Press",
"view": "side",
"prompt": "standing, holding a barbell at the shoulders",
"start": {
"torso": 88,
"arms": [
[
-70,
100
],
[
-70,
100
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Push Press - Behind the Neck",
"view": "side",
"prompt": "standing, barbell resting on the back of the shoulders",
"start": {
"torso": 88,
"arms": [
[
-130,
80
],
[
-130,
80
]
],
"legs": [
[
-60,
-115
],
[
-60,
-115
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
92
],
[
95,
92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Quadriped Arm and Leg Raise",
"view": "side",
"prompt": "on hands and knees on the floor",
"start": {
"torso": 12,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
-90,
180
]
],
"head": 12,
"face": "down"
},
"end": {
"torso": 12,
"arms": [
[
12,
12
],
[
-90,
-90
]
],
"legs": [
[
-90,
180
],
[
192,
192
]
],
"head": 12,
"face": "down"
}
},
{
"name": "Rack Delivery",
"view": "side",
"prompt": "standing, holding a barbell with elbows out to the sides",
"start": {
"torso": 90,
"arms": [
[
-170,
-40
],
[
-170,
-40
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
150
],
[
0,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Rear Delt Raises",
"view": "side",
"prompt": "seated on a bench leaning forward, dumbbells in hands",
"start": {
"torso": 45,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 50
},
"end": {
"torso": 45,
"arms": [
[
-5,
-10
],
[
-5,
-10
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 50
}
},
{
"name": "Return Push from Stance",
"view": "side",
"prompt": "crouched in a three-point stance, one hand on the floor",
"start": {
"torso": 8,
"arms": [
[
-90,
-90
],
[
-60,
-40
]
],
"legs": [
[
-30,
-120
],
[
-30,
-120
]
],
"head": 25
},
"end": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
-75,
-100
],
[
-105,
-95
]
]
}
},
{
"name": "Reverse Flyes",
"view": "side",
"prompt": "lying face down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
},
"end": {
"torso": 30,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
}
},
{
"name": "Reverse Flyes With External Rotation",
"view": "side",
"prompt": "lying face down on an incline bench, a dumbbell in each hand",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
},
"end": {
"torso": 30,
"arms": [
[
-20,
60
],
[
-20,
60
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
}
},
{
"name": "Reverse Machine Flyes",
"view": "side",
"prompt": "seated facing a rear delt machine, chest on the pad, gripping handles",
"start": {
"torso": 90,
"arms": [
[
0,
0
],
[
0,
0
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
165,
170
],
[
165,
170
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Round The World Shoulder Stretch",
"view": "side",
"prompt": "standing, holding a broomstick with a wide grip",
"start": {
"torso": 90,
"arms": [
[
-110,
-110
],
[
-110,
-110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Scapular Pull-Up",
"view": "side",
"prompt": "hanging from a pull-up bar with an overhand grip, arms straight",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 80,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-80,
-90
],
[
-80,
-90
]
],
"head": 95
}
},
{
"name": "Seated Bent-Over Rear Delt Raise",
"view": "side",
"prompt": "seated on the end of a flat bench, leaning forward, dumbbells in hands",
"start": {
"torso": 20,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 25
},
"end": {
"torso": 20,
"arms": [
[
-35,
-30
],
[
-35,
-30
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 25
}
},
{
"name": "Seated Cable Shoulder Press",
"view": "side",
"prompt": "seated upright in a cable press station, gripping handles at shoulders",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Seated Dumbbell Press",
"view": "side",
"prompt": "seated on an upright-back bench, a dumbbell in each hand",
"start": {
"torso": 92,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Seated Front Deltoid",
"view": "side",
"prompt": "sitting upright on the floor, knees bent, partner standing behind",
"start": {
"torso": 90,
"arms": [
[
-30,
-30
],
[
-30,
-30
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 90,
"arms": [
[
165,
170
],
[
165,
170
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
}
},
{
"name": "Seated Side Lateral Raise",
"view": "front",
"prompt": "seated on the end of a flat bench, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
178,
172
],
[
2,
8
]
],
"legs": [
[
-95,
-90
],
[
-85,
-90
]
]
}
},
{
"name": "See-Saw Press (Alternating Side Press)",
"view": "front",
"prompt": "standing, a dumbbell in each hand at shoulder height",
"start": {
"torso": 88,
"arms": [
[
100,
95
],
[
-20,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
-160,
90
],
[
80,
85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Shoulder Circles",
"view": "side",
"prompt": "standing upright, arms relaxed at the sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-110,
-100
],
[
-110,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Shoulder Press - With Bands",
"view": "side",
"prompt": "standing on a resistance band, handles at shoulder height",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Shoulder Raise",
"view": "side",
"prompt": "standing upright, arms relaxed at the sides",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-85,
-88
],
[
-85,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 96
}
},
{
"name": "Shoulder Stretch",
"view": "front",
"prompt": "standing upright, one arm pulled across the chest",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-80,
20
],
[
180,
180
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Shrugs on Multipress",
"view": "side",
"prompt": "standing in a Smith machine, holding the bar at arm's length",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-82,
-82
],
[
-82,
-82
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 96
}
},
{
"name": "Side Laterals to Front Raise",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
0,
5
],
[
0,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Side Wrist Pull",
"view": "front",
"prompt": "standing, one hand holding the other wrist across the body",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 102,
"arms": [
[
-100,
-60
],
[
-125,
-135
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 105
}
},
{
"name": "Side-laying Interior Rotation",
"view": "side",
"prompt": "lying on his side on a flat bench, light weight in bottom hand",
"start": {
"torso": 180,
"arms": [
[
-60,
-90
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
-60,
30
],
[
0,
0
]
],
"legs": [
[
0,
0
],
[
0,
0
]
],
"head": 180
}
},
{
"name": "Single Arm Plank to Row",
"view": "side",
"prompt": "high plank on the floor, hands on kettlebells",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 23,
"arms": [
[
195,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
}
},
{
"name": "Single Dumbbell Raise",
"view": "side",
"prompt": "standing in a wide stance, holding one dumbbell with both hands",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
20,
25
],
[
20,
25
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Single-Arm Linear Jammer",
"view": "side",
"prompt": "staggered wide stance, holding the end of a landmine barbell at the shoulder",
"start": {
"torso": 80,
"arms": [
[
-70,
80
],
[
-90,
-90
]
],
"legs": [
[
-70,
-110
],
[
-110,
-80
]
]
},
"end": {
"torso": 85,
"arms": [
[
50,
50
],
[
-90,
-90
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
}
},
{
"name": "Single-arm Dumbbell Shoulder Press",
"view": "side",
"prompt": "standing upright, one dumbbell held at the shoulder",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Sled Overhead Backward Walk",
"view": "side",
"prompt": "standing facing a weighted sled, holding rope handles overhead",
"start": {
"torso": 90,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
95
],
[
95,
95
]
],
"legs": [
[
-75,
-95
],
[
-110,
-100
]
]
}
},
{
"name": "Sled Reverse Flye",
"view": "front",
"prompt": "standing facing a sled, knees slightly bent, holding rope handles",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-100,
-100
]
],
"legs": [
[
-95,
-85
],
[
-85,
-95
]
]
},
"end": {
"torso": 90,
"arms": [
[
175,
175
],
[
5,
5
]
],
"legs": [
[
-95,
-85
],
[
-85,
-95
]
]
}
},
{
"name": "Smith Incline Shoulder Raise",
"view": "side",
"prompt": "lying on an incline bench under a Smith machine bar, arms straight",
"start": {
"torso": 135,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
-5,
-90
],
[
-5,
-90
]
],
"head": 135
},
"end": {
"torso": 128,
"arms": [
[
45,
45
],
[
45,
45
]
],
"legs": [
[
-5,
-90
],
[
-5,
-90
]
],
"head": 125
}
},
{
"name": "Smith Machine Behind the Back Shrug",
"view": "side",
"prompt": "standing in a Smith machine, bar held behind the thighs",
"start": {
"torso": 90,
"arms": [
[
-100,
-100
],
[
-100,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-97,
-97
],
[
-97,
-97
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 96
}
},
{
"name": "Smith Machine One-Arm Upright Row",
"view": "side",
"prompt": "standing in a Smith machine, one hand on the bar",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Smith Machine Overhead Shoulder Press",
"view": "side",
"prompt": "seated on a bench under a Smith machine bar",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Smith Press",
"view": "side",
"prompt": "seated on an upright bench under a Smith machine bar",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Snatch Shrug",
"view": "side",
"prompt": "standing with a barbell at mid-thigh, wide grip, torso leaning forward",
"start": {
"torso": 60,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-70,
-105
],
[
-70,
-105
]
],
"head": 70
},
"end": {
"torso": 95,
"arms": [
[
-95,
-95
],
[
-95,
-95
]
],
"legs": [
[
-85,
-75
],
[
-85,
-75
]
],
"head": 100
}
},
{
"name": "Standing Alternating Dumbbell Press",
"view": "side",
"prompt": "standing upright, dumbbells held at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Barbell Press Behind Neck",
"view": "side",
"prompt": "standing upright, barbell resting behind the neck",
"start": {
"torso": 90,
"arms": [
[
-110,
100
],
[
-110,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
},
"end": {
"torso": 90,
"arms": [
[
95,
92
],
[
95,
92
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Standing Biceps Stretch Left",
"view": "side",
"prompt": "standing beside a wall, hand on the wall behind",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
180
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Biceps Stretch Right",
"view": "side",
"prompt": "standing beside a wall, hand on the wall behind",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
180
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Bradford Press",
"view": "side",
"prompt": "standing upright, barbell across the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
105,
100
],
[
105,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 92
}
},
{
"name": "Standing Dumbbell Press",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand at head height",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Dumbbell Straight-Arm Front Delt Raise Above Head",
"view": "side",
"prompt": "standing upright, dumbbells in front of the thighs",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Dumbbell Upright Row",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand on the thighs",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
150,
-20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Front Barbell Raise Over Head",
"view": "side",
"prompt": "standing upright, holding a barbell at the thighs",
"start": {
"torso": 90,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Low-Pulley Deltoid Raise",
"view": "front",
"prompt": "standing beside a low cable pulley, one hand holding the handle",
"start": {
"torso": 90,
"arms": [
[
-160,
-160
],
[
-120,
-120
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-160,
-160
],
[
5,
5
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Military Press",
"view": "side",
"prompt": "standing upright, barbell held across the front of the shoulders",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Palm-In One-Arm Dumbbell Press",
"view": "side",
"prompt": "standing, one dumbbell in hand, other hand on an incline bench",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-60,
-60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
-60,
-60
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Palms-In Dumbbell Press",
"view": "side",
"prompt": "standing upright, dumbbells at the shoulders, palms facing in",
"start": {
"torso": 90,
"arms": [
[
-80,
95
],
[
-80,
95
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
88,
90
],
[
88,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Two-Arm Overhead Throw",
"view": "side",
"prompt": "standing, holding a medicine ball with both hands",
"start": {
"torso": 100,
"arms": [
[
110,
-110
],
[
110,
-110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 105
},
"end": {
"torso": 80,
"arms": [
[
35,
35
],
[
35,
35
]
],
"legs": [
[
-80,
-95
],
[
-95,
-90
]
],
"head": 80
}
},
{
"name": "Straight Raises on Incline Bench",
"view": "side",
"prompt": "lying face down on an incline bench, holding a barbell",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
},
"end": {
"torso": 30,
"arms": [
[
30,
30
],
[
30,
30
]
],
"legs": [
[
-120,
-90
],
[
-120,
-90
]
],
"head": 40,
"face": "down"
}
},
{
"name": "TRX Rows",
"view": "side",
"prompt": "leaning back holding suspension straps, heels on the floor",
"start": {
"torso": 120,
"arms": [
[
40,
40
],
[
40,
40
]
],
"legs": [
[
-60,
-60
],
[
-60,
-60
]
],
"head": 120
},
"end": {
"torso": 105,
"arms": [
[
-140,
40
],
[
-140,
40
]
],
"legs": [
[
-75,
-75
],
[
-75,
-75
]
],
"head": 105
}
},
{
"name": "Trap Press",
"view": "side",
"prompt": "lying face up on a flat bench, dumbbells held straight above",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 172,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 172
}
},
{
"name": "Tuck Planche",
"view": "side",
"prompt": "hands flat on the floor, knees tucked to chest",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-20,
-160
],
[
-20,
-160
]
],
"head": 35
},
"end": {
"torso": 10,
"arms": [
[
-80,
-80
],
[
-80,
-80
]
],
"legs": [
[
-10,
-175
],
[
-10,
-175
]
],
"head": 15
}
},
{
"name": "Two-Arm Kettlebell Clean",
"view": "side",
"prompt": "standing over two kettlebells, hips pushed back",
"start": {
"torso": 30,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-60,
-110
],
[
-60,
-110
]
],
"head": 40
},
"end": {
"torso": 90,
"arms": [
[
-70,
110
],
[
-70,
110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Two-Arm Kettlebell Jerk",
"view": "side",
"prompt": "standing, two kettlebells racked at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-70,
110
],
[
-70,
110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-75,
-105
],
[
-75,
-105
]
]
}
},
{
"name": "Two-Arm Kettlebell Military Press",
"view": "side",
"prompt": "standing, two kettlebells racked at the shoulders",
"start": {
"torso": 90,
"arms": [
[
-70,
110
],
[
-70,
110
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Upright Row - With Bands",
"view": "side",
"prompt": "standing on a resistance band, holding the handles",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
150,
-20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Upright Row w/ Dumbbells",
"view": "side",
"prompt": "standing upright, a dumbbell in each hand",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
150,
-20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Upright Row, SZ-bar",
"view": "side",
"prompt": "standing upright, holding an EZ curl bar on the thighs",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
150,
-20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Upright Row, on Multi Press",
"view": "side",
"prompt": "standing in a Smith machine, holding the bar on the thighs",
"start": {
"torso": 90,
"arms": [
[
-85,
-85
],
[
-85,
-85
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
150,
-20
],
[
150,
-20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Upward Stretch",
"view": "side",
"prompt": "standing upright, palms together",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 92,
"arms": [
[
100,
100
],
[
100,
100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 95
}
},
{
"name": "Wall Balls",
"view": "side",
"prompt": "facing a wall, holding a medicine ball at the chest",
"start": {
"torso": 65,
"arms": [
[
-60,
110
],
[
-60,
110
]
],
"legs": [
[
5,
-110
],
[
5,
-110
]
],
"head": 80
},
"end": {
"torso": 90,
"arms": [
[
75,
75
],
[
75,
75
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Band Skull Crusher",
"view": "side",
"prompt": "lying face up on a flat bench, holding a resistance band anchored behind",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Barbell Triceps Extension",
"view": "side",
"prompt": "standing upright, holding a barbell overhead with a narrow grip",
"start": {
"torso": 90,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Bench Dips",
"view": "side",
"prompt": "hands on a bench behind him, legs straight, heels on the floor",
"start": {
"torso": 90,
"arms": [
[
-100,
-90
],
[
-100,
-90
]
],
"legs": [
[
-34,
-34
],
[
-34,
-34
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
-90
],
[
180,
-90
]
],
"legs": [
[
-13,
-13
],
[
-13,
-13
]
]
}
},
{
"name": "Bench Press - Powerlifting",
"view": "side",
"prompt": "lying face up on a flat bench, back arched, holding a barbell",
"start": {
"torso": 180,
"arms": [
[
-75,
90
],
[
-75,
90
]
],
"legs": [
[
-30,
-120
],
[
-30,
-120
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-30,
-120
],
[
-30,
-120
]
],
"head": 180
}
},
{
"name": "Bench Press with Chains",
"view": "side",
"prompt": "lying face up on a flat bench, barbell with hanging chains",
"start": {
"torso": 180,
"arms": [
[
-75,
90
],
[
-75,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Board Press",
"view": "side",
"prompt": "lying face up on a flat bench, barbell above, boards on chest",
"start": {
"torso": 180,
"arms": [
[
-40,
90
],
[
-40,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Body Tricep Press",
"view": "side",
"prompt": "leaning forward on a barbell racked at chest height, feet back",
"start": {
"torso": 60,
"arms": [
[
-15,
-15
],
[
-15,
-15
]
],
"legs": [
[
-120,
-120
],
[
-120,
-120
]
]
},
"end": {
"torso": 45,
"arms": [
[
-60,
70
],
[
-60,
70
]
],
"legs": [
[
-135,
-135
],
[
-135,
-135
]
]
}
},
{
"name": "Body-Up",
"view": "side",
"prompt": "forearm plank on the floor, body straight",
"start": {
"torso": 12,
"arms": [
[
-90,
0
],
[
-90,
0
]
],
"legs": [
[
192,
192
],
[
192,
192
]
]
},
"end": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
}
},
{
"name": "Cable Rope Overhead Triceps Extension",
"view": "side",
"prompt": "standing, facing away from a low cable, rope held overhead",
"start": {
"torso": 85,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
},
"end": {
"torso": 85,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-80,
-90
],
[
-100,
-90
]
]
}
},
{
"name": "Chain Handle Extension",
"view": "side",
"prompt": "lying face up on a flat bench, holding handles with chains attached",
"start": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Close-Grip Dumbbell Press",
"view": "side",
"prompt": "lying face up on a flat bench, holding one dumbbell with both hands",
"start": {
"torso": 180,
"arms": [
[
-85,
85
],
[
-85,
85
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Close-Grip EZ-Bar Press",
"view": "side",
"prompt": "lying face up on a flat bench, narrow grip on an EZ bar",
"start": {
"torso": 180,
"arms": [
[
-85,
85
],
[
-85,
85
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Close-Grip Push-Up",
"view": "side",
"prompt": "push-up position on the floor, hands close together",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 8,
"arms": [
[
165,
-90
],
[
165,
-90
]
],
"legs": [
[
188,
188
],
[
188,
188
]
]
}
},
{
"name": "Close-Grip Push-Up off of a Dumbbell",
"view": "side",
"prompt": "push-up position, both hands on an upright dumbbell",
"start": {
"torso": 32,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
212,
212
],
[
212,
212
]
]
},
"end": {
"torso": 15,
"arms": [
[
165,
-90
],
[
165,
-90
]
],
"legs": [
[
195,
195
],
[
195,
195
]
]
}
},
{
"name": "Dip Machine",
"view": "side",
"prompt": "seated in a dip machine, gripping the handles at his sides",
"start": {
"torso": 90,
"arms": [
[
-140,
-45
],
[
-140,
-45
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
-95,
-90
],
[
-95,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
]
}
},
{
"name": "Dips - Triceps Version",
"view": "side",
"prompt": "supported on parallel dip bars, arms straight, body upright",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
},
"end": {
"torso": 85,
"arms": [
[
-165,
-90
],
[
-165,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
}
},
{
"name": "Dumbbell Floor Press",
"view": "side",
"prompt": "lying face up on the floor, knees bent, a dumbbell in each hand",
"start": {
"torso": 180,
"arms": [
[
-5,
95
],
[
-5,
95
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
}
},
{
"name": "Dumbbell Tricep Extension -Pronated Grip",
"view": "side",
"prompt": "lying face up on a flat bench, a dumbbell in each hand",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Dumbbell Triceps Extension",
"view": "side",
"prompt": "standing upright, one dumbbell held overhead with both hands",
"start": {
"torso": 90,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Floor Dips",
"view": "side",
"prompt": "sitting on the floor, hands behind, knees bent, feet flat",
"start": {
"torso": 110,
"arms": [
[
-117,
-117
],
[
-117,
-117
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
],
"head": 105
},
"end": {
"torso": 130,
"arms": [
[
-150,
-100
],
[
-150,
-100
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
],
"head": 115
}
},
{
"name": "Floor Press",
"view": "side",
"prompt": "lying face up on the floor in a power rack, holding a barbell",
"start": {
"torso": 180,
"arms": [
[
-5,
95
],
[
-5,
95
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
}
},
{
"name": "Floor Press with Chains",
"view": "side",
"prompt": "lying face up on the floor, holding a barbell with chains",
"start": {
"torso": 180,
"arms": [
[
-5,
95
],
[
-5,
95
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
55,
-55
],
[
55,
-55
]
],
"head": 180
}
},
{
"name": "High-Cable Cross Tricep Extention - NB",
"view": "front",
"prompt": "standing between two high cable pulleys, handles crossed in front",
"start": {
"torso": 90,
"arms": [
[
175,
10
],
[
5,
170
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
170,
165
],
[
10,
15
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Incline Push-Up Close-Grip",
"view": "side",
"prompt": "hands close together on a Smith machine bar, body straight",
"start": {
"torso": 50,
"arms": [
[
-40,
-40
],
[
-40,
-40
]
],
"legs": [
[
230,
230
],
[
230,
230
]
]
},
"end": {
"torso": 40,
"arms": [
[
-140,
5
],
[
-140,
5
]
],
"legs": [
[
220,
220
],
[
220,
220
]
]
}
},
{
"name": "Lying Triceps Press",
"view": "side",
"prompt": "lying face up on a flat bench, holding an EZ curl bar",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "One Arm Pronated Dumbbell Triceps Extension",
"view": "side",
"prompt": "lying face up on a flat bench, one dumbbell held above",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
0,
0
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
95,
190
],
[
0,
0
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "One Arm Supinated Dumbbell Triceps Extension",
"view": "side",
"prompt": "lying face up on a flat bench, one dumbbell held above",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
0,
0
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
95,
190
],
[
0,
0
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Overhead Triceps",
"view": "side",
"prompt": "sitting upright on the floor, knees bent, partner behind",
"start": {
"torso": 90,
"arms": [
[
95,
-100
],
[
-60,
-30
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
},
"end": {
"torso": 90,
"arms": [
[
92,
90
],
[
-60,
-30
]
],
"legs": [
[
45,
-45
],
[
45,
-45
]
]
}
},
{
"name": "Overhead Triceps Extension",
"view": "side",
"prompt": "standing upright, a dumbbell held overhead",
"start": {
"torso": 90,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Parallel Bar Dip",
"view": "side",
"prompt": "supported on parallel bars, arms straight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
},
"end": {
"torso": 75,
"arms": [
[
-160,
-90
],
[
-160,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
],
"head": 80
}
},
{
"name": "Pin Presses",
"view": "side",
"prompt": "lying face up on a bench in a rack, barbell on safety pins",
"start": {
"torso": 180,
"arms": [
[
-50,
90
],
[
-50,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Push-Ups - Close Triceps Position",
"view": "side",
"prompt": "push-up position on the floor, hands close together",
"start": {
"torso": 23,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
203,
203
],
[
203,
203
]
]
},
"end": {
"torso": 8,
"arms": [
[
165,
-90
],
[
165,
-90
]
],
"legs": [
[
188,
188
],
[
188,
188
]
]
}
},
{
"name": "Reverse Band Bench Press",
"view": "side",
"prompt": "lying face up on a flat bench, barbell hung from rack bands",
"start": {
"torso": 180,
"arms": [
[
-75,
90
],
[
-75,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Ring Dips",
"view": "side",
"prompt": "supported on gymnastic rings, arms straight",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
]
},
"end": {
"torso": 80,
"arms": [
[
-165,
-90
],
[
-165,
-90
]
],
"legs": [
[
-100,
-150
],
[
-100,
-150
]
],
"head": 85
}
},
{
"name": "Seated Bent-Over One-Arm Dumbbell Triceps Extension",
"view": "side",
"prompt": "seated on the end of a flat bench, leaning forward, one dumbbell",
"start": {
"torso": 25,
"arms": [
[
205,
-90
],
[
-80,
-60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 30
},
"end": {
"torso": 25,
"arms": [
[
205,
205
],
[
-80,
-60
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 30
}
},
{
"name": "Seated Bent-Over Two-Arm Dumbbell Triceps Extension",
"view": "side",
"prompt": "seated on the end of a flat bench, leaning forward, two dumbbells",
"start": {
"torso": 25,
"arms": [
[
205,
-90
],
[
205,
-90
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 30
},
"end": {
"torso": 25,
"arms": [
[
205,
205
],
[
205,
205
]
],
"legs": [
[
0,
-90
],
[
0,
-90
]
],
"head": 30
}
},
{
"name": "Skull Crusher",
"view": "side",
"prompt": "lying face up on a flat bench, holding a barbell",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Skullcrusher Dumbbells",
"view": "side",
"prompt": "lying face up on a flat bench, a dumbbell in each hand",
"start": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
100,
195
],
[
100,
195
]
],
"legs": [
[
-10,
-90
],
[
-10,
-90
]
],
"head": 180
}
},
{
"name": "Sled Overhead Triceps Extension",
"view": "side",
"prompt": "standing facing away from a sled, rope handles held overhead",
"start": {
"torso": 80,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-75,
-95
],
[
-105,
-95
]
],
"head": 85
},
"end": {
"torso": 80,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-75,
-95
],
[
-105,
-95
]
],
"head": 85
}
},
{
"name": "Speed Band Overhead Triceps",
"view": "side",
"prompt": "standing on a resistance band, band held behind the head",
"start": {
"torso": 90,
"arms": [
[
95,
-100
],
[
95,
-100
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
92,
90
],
[
92,
90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing Bent-Over One-Arm Dumbbell Triceps Extension",
"view": "side",
"prompt": "bent forward at the hips, knees soft, one dumbbell in hand",
"start": {
"torso": 20,
"arms": [
[
200,
-90
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
198
],
[
-90,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Standing Bent-Over Two-Arm Dumbbell Triceps Extension",
"view": "side",
"prompt": "bent forward at the hips, knees soft, a dumbbell in each hand",
"start": {
"torso": 20,
"arms": [
[
200,
-90
],
[
200,
-90
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
},
"end": {
"torso": 20,
"arms": [
[
200,
198
],
[
200,
198
]
],
"legs": [
[
-80,
-95
],
[
-80,
-95
]
],
"head": 30
}
},
{
"name": "Standing Low-Pulley One-Arm Triceps Extension",
"view": "side",
"prompt": "standing, facing away from a low cable, handle held overhead",
"start": {
"torso": 90,
"arms": [
[
95,
-100
],
[
70,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
92,
90
],
[
70,
150
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Standing One-Arm Dumbbell Triceps Extension",
"view": "side",
"prompt": "standing upright, one dumbbell held overhead",
"start": {
"torso": 90,
"arms": [
[
90,
90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Supine Chest Throw",
"view": "side",
"prompt": "lying face up on the floor, knees bent, medicine ball on chest",
"start": {
"torso": 180,
"arms": [
[
-10,
120
],
[
-10,
120
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
},
"end": {
"torso": 180,
"arms": [
[
90,
90
],
[
90,
90
]
],
"legs": [
[
60,
-60
],
[
60,
-60
]
],
"head": 180
}
},
{
"name": "Tricep Side Stretch",
"view": "front",
"prompt": "standing upright, one arm across the chest",
"start": {
"torso": 90,
"arms": [
[
-95,
-92
],
[
-85,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
10,
80
],
[
-80,
170
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Triceps Extensions on Cable",
"view": "side",
"prompt": "standing at a high cable, leaning slightly forward, holding a bar",
"start": {
"torso": 82,
"arms": [
[
-95,
20
],
[
-95,
20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 85
},
"end": {
"torso": 82,
"arms": [
[
-92,
-88
],
[
-92,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 85
}
},
{
"name": "Triceps Pushdown - Rope Attachment",
"view": "side",
"prompt": "standing at a high cable, holding a rope attachment",
"start": {
"torso": 85,
"arms": [
[
-95,
20
],
[
-95,
20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 88
},
"end": {
"torso": 85,
"arms": [
[
-92,
-88
],
[
-92,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 88
}
},
{
"name": "Triceps Pushdown - V-Bar Attachment",
"view": "side",
"prompt": "standing at a high cable, holding a V-bar",
"start": {
"torso": 85,
"arms": [
[
-95,
20
],
[
-95,
20
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 88
},
"end": {
"torso": 85,
"arms": [
[
-92,
-88
],
[
-92,
-88
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
],
"head": 88
}
},
{
"name": "Triceps Stretch",
"view": "side",
"prompt": "standing upright, one hand behind the head",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
110,
30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Triceps Stretch Left",
"view": "side",
"prompt": "standing upright, one hand on the upper back",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
110,
30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Triceps Stretch Right",
"view": "side",
"prompt": "standing upright, one hand on the upper back",
"start": {
"torso": 90,
"arms": [
[
-90,
-90
],
[
-90,
-90
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
},
"end": {
"torso": 90,
"arms": [
[
95,
-100
],
[
110,
30
]
],
"legs": [
[
-90,
-90
],
[
-90,
-90
]
]
}
},
{
"name": "Weighted Bench Dip",
"view": "side",
"prompt": "hands on a bench behind, heels on a bench in front, weight on lap",
"start": {
"torso": 90,
"arms": [
[
-100,
-90
],
[
-100,
-90
]
],
"legs": [
[
-2,
-2
],
[
-2,
-2
]
]
},
"end": {
"torso": 90,
"arms": [
[
180,
-90
],
[
180,
-90
]
],
"legs": [
[
18,
18
],
[
18,
18
]
]
}
}
]

In [ ]:
import subprocess
from pathlib import Path

import torch


def run(*args):
    """Run one pipeline step and stop the notebook if it fails."""
    subprocess.run(["python", "pipeline.py", *map(str, args)], check=True)


print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - turn on GPU T4 in Settings")
inputs = Path("/kaggle/input")
DATA = next((p.parent for p in inputs.rglob("metadata.jsonl")), None)
LORA = next((p.parent for p in inputs.rglob("pytorch_lora_weights.safetensors")), None)
MEDIA = {".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm", ".jpg", ".jpeg", ".png", ".webp"}
clip_files = [p for p in inputs.rglob("*") if p.suffix.lower() in MEDIA
              and (DATA is None or DATA not in p.parents) and (LORA is None or LORA not in p.parents)]
CLIPS = Path("/kaggle/working/clips")
CLIPS.mkdir(exist_ok=True)
for p in clip_files:  # optional clips, gathered in one folder
    target = CLIPS / p.name
    if not target.exists():
        target.symlink_to(p)
print("style data:", DATA)
print("saved style (LoRA):", LORA or "none yet, it will be trained")
print("optional clips/photos:", len(clip_files))

## 1. Learn the drawing style (skipped when a saved LoRA was added as input)

In [ ]:
if LORA is None:
    run("train", "--data", DATA, "--lora", "/kaggle/working/lora", "--steps", 2000)
    LORA = "/kaggle/working/lora"
    !cd /kaggle/working && zip -qr gymai-style-lora.zip lora

## 2. Start and end pose of every exercise

In [ ]:
run("specposes", "--specs", "exercise-poses.json", "--poses", "/kaggle/working/poses")
if clip_files:  # clips you added win over the written poses
    run("poses", "--clips", "/kaggle/working/clips", "--poses", "/kaggle/working/clip-poses")
    import json, shutil
    written = json.load(open("/kaggle/working/poses/manifest.json"))
    from_clips = json.load(open("/kaggle/working/clip-poses/manifest.json"))
    for entry in from_clips:
        shutil.copytree(f"/kaggle/working/clip-poses/{entry['slug']}", f"/kaggle/working/poses/{entry['slug']}",
                        dirs_exist_ok=True)
    slugs = {e["slug"] for e in from_clips}
    merged = from_clips + [e for e in written if e["slug"] not in slugs]
    json.dump(merged, open("/kaggle/working/poses/manifest.json", "w"), indent=1)

## 3. Draw each exercise (LIMIT = how many; 0 = all)

In [ ]:
LIMIT = 20
run("generate", "--lora", LORA, "--poses", "/kaggle/working/poses", "--out", "/kaggle/working/drawings",
    "--candidates", 3, "--limit", LIMIT)

## 4. Review and download

In [ ]:
from IPython.display import Image as Show, display
display(Show("/kaggle/working/drawings/review.png"))
!cd /kaggle/working && zip -qr gymai-drawings.zip drawings && ls -la /kaggle/working/*.zip